# OpenVLA-OFT D3: joint proprioception and continuous action chunks

This notebook preserves the existing Panda simulation and collection cells, then fine-tunes the official OpenVLA-OFT implementation on **D3**:

- 700 nominal + 100 boundary demonstrations; all recovery demonstrations are excluded.
- Input: fixed third-person image, language instruction, and `[q1, q2, q3, q4, q5, q6, q7, gripper_open_fraction]`.
- Output: an 8-step chunk of continuous 7D Cartesian actions.
- Objective: continuous L1 regression with parallel decoding, LoRA rank 32, and train-only q01/q99 normalization.

For OFT fine-tuning, start at the server section after Cell E. Use `TRAINING_MODE = "smoke"` first, then change it to `"full"`.

In [ ]:
# Cell 1: Laptop-safe dependency, driver, and thermal preflight.
# Run this before every fresh VS Code/Jupyter kernel.
import importlib.metadata as importlib_metadata
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
from pathlib import Path

# GitHub sync: save locally, then rerun Cell 1 to publish the latest source snapshot.
GITHUB_WEB_URL = "https://github.com/Fariborz-Eshraghi/force-openvla"
GITHUB_HTTPS_REMOTE = f"{GITHUB_WEB_URL}.git"
GITHUB_SSH_REMOTE = "git@github-force-openvla:Fariborz-Eshraghi/force-openvla.git"
GITHUB_SSH_KEY = Path.home() / ".ssh" / "force_openvla_deploy"
GITHUB_REMOTE = GITHUB_SSH_REMOTE if GITHUB_SSH_KEY.is_file() else GITHUB_HTTPS_REMOTE
GITHUB_BRANCH = "main"
GITHUB_USER = "Fariborz-Eshraghi"
GITHUB_CHECKOUT = Path.home() / ".cache" / "force-openvla-github"
GITHUB_SOURCE_DIR = Path.cwd().resolve()
GITHUB_NOTEBOOK = Path(os.environ.get("OPENVLA_NOTEBOOK_PATH", globals().get("__vsc_ipynb_file__", GITHUB_SOURCE_DIR / "OpenVLA26_OFT.ipynb"))).expanduser().resolve()
GITHUB_ROOT_FILES = ("README.md", "assets/force-openvla-architecture.png")
GITHUB_EXTRA_FILES = (
    "prepare_panda_d3.py",
    "openvla_oft_d3_setup.py",
    "openvla_oft_d3_verify.py",
    "openvla_oft_heldout_visual_eval.py",
    "rlds_dataset_builder/panda_pickplace_d3/__init__.py",
    "rlds_dataset_builder/panda_pickplace_d3/panda_pickplace_d3_dataset_builder.py",
)


def _optional_secret(name):
    value = os.environ.get(name, "")
    if value:
        return value
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""


def _git(*args, cwd=GITHUB_CHECKOUT, check=True):
    env = os.environ.copy()
    env.update({"GIT_TERMINAL_PROMPT": "0", "GCM_INTERACTIVE": "never"})
    return subprocess.run(["git", *map(str, args)], cwd=cwd, env=env, text=True, check=check)


def _write_source_only_notebook(destination_dir):
    notebook = None
    try:
        from google.colab import _message
        reply = _message.blocking_request("get_ipynb", timeout_sec=30)
        notebook = reply.get("ipynb") if isinstance(reply, dict) else None
    except Exception:
        pass
    if not isinstance(notebook, dict):
        if not GITHUB_NOTEBOOK.is_file():
            raise FileNotFoundError(f"Save the notebook first or set OPENVLA_NOTEBOOK_PATH: {GITHUB_NOTEBOOK}")
        notebook = json.loads(GITHUB_NOTEBOOK.read_text(encoding="utf-8"))
    raw_name = notebook.get("metadata", {}).get("colab", {}).get("name", GITHUB_NOTEBOOK.name)
    notebook_name = Path(raw_name).name
    if not notebook_name.endswith(".ipynb"):
        notebook_name += ".ipynb"
    for cell in notebook.get("cells", []):
        if cell.get("cell_type") == "code":
            cell["outputs"] = []
            cell["execution_count"] = None
        cell.get("metadata", {}).pop("execution", None)
    destination = destination_dir / notebook_name
    destination.write_text(json.dumps(notebook, ensure_ascii=False, indent=1) + "\n", encoding="utf-8")
    return destination


def sync_to_github():
    use_ssh = GITHUB_SSH_KEY.is_file()
    token = "" if use_ssh else _optional_secret("GITHUB_TOKEN")
    if not use_ssh and not token:
        print("GitHub sync skipped: local SSH key not found; add GITHUB_TOKEN to Colab Secrets and rerun Cell 1.")
        return
    print("GitHub authentication:", "permanent SSH deploy key" if use_ssh else "Colab token")
    GITHUB_CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
    if not (GITHUB_CHECKOUT / ".git").is_dir():
        if GITHUB_CHECKOUT.exists():
            raise RuntimeError(f"GitHub checkout path exists but is not a Git repository: {GITHUB_CHECKOUT}")
        _git("clone", "--branch", GITHUB_BRANCH, "--single-branch", GITHUB_REMOTE, GITHUB_CHECKOUT, cwd=GITHUB_CHECKOUT.parent)
    _git("remote", "set-url", "origin", GITHUB_REMOTE)
    _git("config", "user.name", GITHUB_USER)
    _git("config", "user.email", f"{GITHUB_USER}@users.noreply.github.com")
    if token:
        _git("config", "credential.helper", "cache --timeout=21600")
        subprocess.run(
            ["git", "credential", "approve"],
            cwd=GITHUB_CHECKOUT,
            input=f"protocol=https\nhost=github.com\nusername={GITHUB_USER}\npassword={token}\n\n",
            text=True,
            check=True,
        )
        token = None
    _git("pull", "--rebase", "origin", GITHUB_BRANCH)
    simulation_dir = GITHUB_CHECKOUT / "Simulation"
    simulation_dir.mkdir(parents=True, exist_ok=True)
    staged = [_write_source_only_notebook(simulation_dir).relative_to(GITHUB_CHECKOUT)]
    for relative_name in GITHUB_ROOT_FILES:
        source = GITHUB_SOURCE_DIR / relative_name
        if source.is_file():
            destination = GITHUB_CHECKOUT / relative_name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            staged.append(destination.relative_to(GITHUB_CHECKOUT))
    for relative_name in GITHUB_EXTRA_FILES:
        source = GITHUB_SOURCE_DIR / relative_name
        if source.is_file():
            destination = simulation_dir / relative_name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            staged.append(destination.relative_to(GITHUB_CHECKOUT))
    _git("add", "--", *staged)
    diff = _git("diff", "--cached", "--quiet", check=False)
    if diff.returncode == 1:
        _git("commit", "-m", f"Sync {staged[0].name}, README, and simulation helpers")
    elif diff.returncode != 0:
        raise RuntimeError("Git could not inspect the staged changes.")
    _git("push", "origin", GITHUB_BRANCH)
    print(f"GitHub sync complete: {GITHUB_WEB_URL}/tree/{GITHUB_BRANCH}/Simulation")


sync_to_github()

SAFE_LAPTOP_MODE = True
INSTALL_LIGHTWEIGHT_SIM_PACKAGES = True
INSTALL_HEAVY_VLA_STACK = False  # Flip to True only after updating the NVIDIA driver and closing other apps.

# Keep the i7-7700HQ and cooling system out of the danger zone.
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MKL_NUM_THREADS", "8")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "8")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:64")

LOCAL_HF_CACHE_DIR = Path("/home/fariborz/projects/force-vla-colab/.hf_cache")
LOCAL_HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(LOCAL_HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_HF_CACHE_DIR / "transformers")
for cache_subdir in [os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"]]:
    Path(cache_subdir).mkdir(parents=True, exist_ok=True)

print("Python:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("CPU threads limited to:", os.environ.get("OMP_NUM_THREADS"))
print("HF cache:", LOCAL_HF_CACHE_DIR)
try:
    cache_drive = shutil.disk_usage(LOCAL_HF_CACHE_DIR.anchor)
    print("HF cache drive free GiB:", round(cache_drive.free / 1024**3, 2))
except Exception as exc:
    print("HF cache drive space check unavailable:", exc)

try:
    import psutil
    print("System RAM GiB:", round(psutil.virtual_memory().total / 1024**3, 2))
except Exception as exc:
    print("psutil RAM check unavailable:", exc)


def module_status(module_name, package_name=None):
    package_name = package_name or module_name
    installed = importlib.util.find_spec(module_name) is not None
    version = None
    if installed:
        try:
            version = importlib_metadata.version(package_name)
        except Exception:
            version = "installed"
    return installed, version


def pip_install(args):
    cmd = [sys.executable, "-m", "pip", "install", "--no-cache-dir", *args]
    print("Running:", " ".join(cmd))
    subprocess.check_call(cmd)


LIGHTWEIGHT_PACKAGES = {
    "mujoco": "mujoco",
    "mediapy": "mediapy",
    "imageio_ffmpeg": "imageio-ffmpeg",
    "PIL": "pillow",
    "ipywidgets": "ipywidgets",
    "psutil": "psutil",
}

missing_light = []
print("\nLightweight simulation/display packages")
for module_name, pip_name in LIGHTWEIGHT_PACKAGES.items():
    installed, version = module_status(module_name, pip_name)
    print(f"  {module_name:16s}: {version if installed else 'missing'}")
    if not installed:
        missing_light.append(pip_name)

if missing_light and INSTALL_LIGHTWEIGHT_SIM_PACKAGES:
    pip_install(missing_light)
    print("Lightweight packages installed. If imports still fail, restart the kernel and rerun Cell 1.")
elif missing_light:
    print("Missing lightweight packages:", missing_light)

# OpenVLA's official minimal requirements pin transformers/tokenizers/timm; PyTorch is installed from the PyTorch CUDA 11.8 wheel index.
TORCH_INSTALL_COMMAND = [
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "torch==2.5.1", "torchvision==0.20.1", "torchaudio==2.5.1",
    "--index-url", "https://download.pytorch.org/whl/cu118",
]
VLA_PACKAGE_COMMAND = [
    sys.executable, "-m", "pip", "install", "--no-cache-dir",
    "transformers==4.40.1",
    "tokenizers==0.19.1",
    "timm==0.9.10",
    "accelerate>=0.30.1",
    "bitsandbytes>=0.46.0",
    "safetensors",
    "huggingface_hub",
    "einops",
    "sentencepiece",
]

HEAVY_MODULES = {
    "torch": "torch",
    "torchvision": "torchvision",
    "transformers": "transformers",
    "tokenizers": "tokenizers",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "timm": "timm",
    "huggingface_hub": "huggingface_hub",
    "einops": "einops",
}

print("\nHeavy VLA packages")
missing_heavy = []
for module_name, package_name in HEAVY_MODULES.items():
    installed, version = module_status(module_name, package_name)
    print(f"  {module_name:16s}: {version if installed else 'missing'}")
    if not installed:
        missing_heavy.append(module_name)


def public_nvidia_driver_version(wmi_driver_version):
    # WMI often reports NVIDIA 451.67 as 27.21.14.5167, 536.40 as 31.0.15.3640, etc.
    parts = str(wmi_driver_version).split(".")
    if len(parts) >= 4 and parts[2].isdigit() and parts[3].isdigit():
        branch = (int(parts[2]) - 10) * 100
        return branch + int(parts[3]) / 100.0
    return None


def windows_gpu_report():
    if platform.system() != "Windows":
        return []
    powershell_exe = (
        shutil.which("powershell.exe")
        or shutil.which("powershell")
        or shutil.which("pwsh")
        or r"C:\Windows\System32\WindowsPowerShell\v1.0\powershell.exe"
    )
    if not Path(powershell_exe).exists() and shutil.which(powershell_exe) is None:
        print("PowerShell was not found, so Windows GPU details could not be queried.")
        return []
    ps = "Get-CimInstance Win32_VideoController | Select-Object Name,AdapterRAM,DriverVersion | ConvertTo-Json"
    result = subprocess.run(
        [powershell_exe, "-NoProfile", "-ExecutionPolicy", "Bypass", "-Command", ps],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        check=False,
        timeout=10,
    )
    if result.returncode != 0 or not result.stdout.strip():
        print("Windows GPU report unavailable:", result.stderr.strip())
        return []
    data = json.loads(result.stdout)
    if isinstance(data, dict):
        data = [data]
    return data

print("\nWindows GPU / driver report")
NVIDIA_DRIVER_PUBLIC_VERSION = None
for gpu in windows_gpu_report():
    name = gpu.get("Name", "unknown")
    ram_gib = (int(gpu.get("AdapterRAM") or 0) / 1024**3) if gpu.get("AdapterRAM") else None
    driver = gpu.get("DriverVersion", "unknown")
    public_driver = public_nvidia_driver_version(driver) if "NVIDIA" in name.upper() else None
    if "NVIDIA" in name.upper():
        NVIDIA_DRIVER_PUBLIC_VERSION = public_driver
    ram_text = f", VRAM~{ram_gib:.2f} GiB" if ram_gib else ""
    public_text = f", public driver~{public_driver:.2f}" if public_driver else ""
    print(f"  {name}{ram_text}, WMI driver={driver}{public_text}")

if NVIDIA_DRIVER_PUBLIC_VERSION is not None and NVIDIA_DRIVER_PUBLIC_VERSION < 452.39:
    print("\nWARNING: Your NVIDIA driver appears older than the CUDA 11.x Windows compatibility floor (452.39).")
    print("Update the NVIDIA driver before installing/running CUDA PyTorch. The current driver may make torch.cuda unavailable.")

print("\nPyTorch CUDA check")
torch_cuda_ok = False
if importlib.util.find_spec("torch") is None:
    print("  torch is not installed in this kernel.")
else:
    try:
        import torch
        print("  torch:", torch.__version__)
        print("  torch CUDA runtime:", torch.version.cuda)
        print("  cuda available:", torch.cuda.is_available())
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            print("  device:", torch.cuda.get_device_name(0))
            print("  VRAM GiB:", round(props.total_memory / 1024**3, 2))
            print("  compute capability:", f"{props.major}.{props.minor}")
            x = torch.randn(256, 256, device="cuda", dtype=torch.float16)
            y = x @ x.T
            torch.cuda.synchronize()
            del x, y
            torch.cuda.empty_cache()
            torch_cuda_ok = True
            print("  tiny CUDA tensor test: ok")
    except Exception as exc:
        print("  torch import/CUDA check failed:", repr(exc))

print("\nBitsAndBytes 4-bit CUDA check")
if importlib.util.find_spec("bitsandbytes") is None:
    print("  bitsandbytes is not installed in this kernel.")
elif not torch_cuda_ok:
    print("  skipped because torch CUDA is not ready.")
else:
    try:
        import bitsandbytes as bnb
        layer = bnb.nn.Linear4bit(16, 8, bias=False, quant_type="nf4", compute_dtype=torch.float16).cuda()
        inp = torch.randn(2, 16, device="cuda", dtype=torch.float16)
        with torch.no_grad():
            out = layer(inp)
        torch.cuda.synchronize()
        del layer, inp, out
        torch.cuda.empty_cache()
        print("  tiny NF4 Linear4bit test: ok")
    except Exception as exc:
        print("  bitsandbytes 4-bit CUDA check failed:", repr(exc))

if INSTALL_HEAVY_VLA_STACK:
    print("\nInstalling heavy PyTorch CUDA/OpenVLA packages. This can take a while and several GB of disk/network.")
    subprocess.check_call(TORCH_INSTALL_COMMAND)
    subprocess.check_call(VLA_PACKAGE_COMMAND)
    print("Heavy stack installed. Restart the kernel before loading OpenVLA.")
elif missing_heavy:
    print("\nHeavy VLA stack install is disabled for safety.")
    print("After updating the NVIDIA driver, you can either set INSTALL_HEAVY_VLA_STACK=True and rerun this cell, or run these commands in VS Code's terminal:")
    print(" ".join(TORCH_INSTALL_COMMAND))
    print(" ".join(VLA_PACKAGE_COMMAND))
else:
    print("\nHeavy VLA stack is installed. Keep INSTALL_HEAVY_VLA_STACK=False unless you intentionally want to reinstall it.")

print("\nHF token setup")
print("  Store HF_TOKEN in the environment or Colab Secrets if the model download asks for one.")
print("  Never paste access tokens into a notebook that may be published.")
if os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN"):
    print("  An environment token is also present and can be used as fallback.")


In [ ]:
# Cell 2: Imports and rendering setup for local Windows/VS Code or Linux.
import contextlib
import gc
import json
import os
import platform
import shutil
import subprocess
import time
import xml.etree.ElementTree as ET
from base64 import b64encode
from html import escape
from pathlib import Path

# Windows local notebooks should use GLFW. Linux/headless GPU workstations usually use EGL.
if "MUJOCO_GL" not in os.environ:
    os.environ["MUJOCO_GL"] = "glfw" if platform.system() == "Windows" else "egl"

import imageio_ffmpeg
import mediapy as media
import ipywidgets as widgets
import mujoco
import numpy as np
from IPython.display import HTML, clear_output, display
from PIL import Image

PROJECT_DIR = Path(os.environ.get("OPENVLA_PROJECT_DIR", Path.cwd())).expanduser().resolve()
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

ffmpeg_path = shutil.which("ffmpeg") or imageio_ffmpeg.get_ffmpeg_exe()
media.set_ffmpeg(ffmpeg_path)

print("Project:", PROJECT_DIR)
print("MuJoCo version:", mujoco.__version__)
print("Rendering backend:", os.environ.get("MUJOCO_GL"))
print("ffmpeg:", ffmpeg_path)


In [ ]:
# Cell 3: Build/load a slim Panda scene with only a wrist camera, table, and red block.
MENAGERIE_DIR = PROJECT_DIR / "mujoco_menagerie"
PANDA_DIR = MENAGERIE_DIR / "franka_emika_panda"
SOURCE_PANDA_XML = PANDA_DIR / "panda.xml"
PANDA_WITH_CAMERA_XML = PANDA_DIR / "panda_openvla04_wrist_camera.xml"
PANDA_XML = PANDA_DIR / "openvla04_pickplace_scene.xml"


def download_mujoco_menagerie_without_git():
    import urllib.request

    api_url = "https://api.github.com/repos/google-deepmind/mujoco_menagerie/git/trees/main?recursive=1"
    prefix = "franka_emika_panda/"
    print("git is not available; downloading only mujoco_menagerie/franka_emika_panda from GitHub.")
    MENAGERIE_DIR.mkdir(parents=True, exist_ok=True)

    with urllib.request.urlopen(api_url, timeout=120) as response:
        tree_payload = json.loads(response.read().decode("utf-8"))

    panda_files = [
        item["path"]
        for item in tree_payload.get("tree", [])
        if item.get("type") == "blob" and item.get("path", "").startswith(prefix)
    ]
    if not panda_files:
        raise RuntimeError("Could not find franka_emika_panda files in the GitHub tree response.")

    for rel_path in panda_files:
        dest = MENAGERIE_DIR / rel_path
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() and dest.stat().st_size > 0:
            continue
        raw_url = f"https://raw.githubusercontent.com/google-deepmind/mujoco_menagerie/main/{rel_path}"
        print("  downloading", rel_path)
        with urllib.request.urlopen(raw_url, timeout=120) as response:
            dest.write_bytes(response.read())


if not SOURCE_PANDA_XML.exists():
    git_exe = shutil.which("git")
    if MENAGERIE_DIR.exists() and (MENAGERIE_DIR / ".git").exists() and git_exe:
        subprocess.check_call([git_exe, "-C", str(MENAGERIE_DIR), "pull", "--ff-only"])
    elif MENAGERIE_DIR.exists() and PANDA_DIR.exists():
        pass
    elif MENAGERIE_DIR.exists():
        raise RuntimeError(
            f"{MENAGERIE_DIR} exists but does not look like a complete mujoco_menagerie install. "
            "Rename or remove that folder, then rerun this cell."
        )
    elif git_exe:
        subprocess.check_call([
            git_exe,
            "clone",
            "--depth",
            "1",
            "https://github.com/google-deepmind/mujoco_menagerie.git",
            str(MENAGERIE_DIR),
        ])
    else:
        download_mujoco_menagerie_without_git()

if not SOURCE_PANDA_XML.exists():
    raise FileNotFoundError(f"Could not find Panda model at {SOURCE_PANDA_XML}")


def find_body(root, body_name):
    for body in root.iter("body"):
        if body.get("name") == body_name:
            return body
    raise ValueError(f"Could not find body named {body_name!r}")


panda_tree = ET.parse(SOURCE_PANDA_XML)
panda_root = panda_tree.getroot()
hand_body = find_body(panda_root, "hand")

for child in list(hand_body):
    if child.tag == "body" and child.get("name") == "agent_camera_mount":
        hand_body.remove(child)

mount_body = ET.Element("body", name="agent_camera_mount", pos="0.08 0 -0.12")
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_boom_geom",
    type="capsule",
    fromto="0 0 0 -0.08 0 0.12",
    size="0.008",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_body_geom",
    type="box",
    size="0.020 0.016 0.012",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "camera",
    name="agent_wrist_camera",
    mode="fixed",
    fovy="82",
    xyaxes="-0.000007 -1.000000 0.000034 -0.976592 -0.000000 -0.215100",
)
hand_body.insert(0, mount_body)
ET.indent(panda_tree, space="  ")
panda_tree.write(PANDA_WITH_CAMERA_XML, encoding="unicode")

PANDA_XML.write_text(
    f"""<mujoco model="openvla04 slim panda pick place scene">
  <include file="{PANDA_WITH_CAMERA_XML.name}"/>

  <statistic center="0.50 0 0.35" extent="1.1"/>

  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0 0 0"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global azimuth="120" elevation="-20"/>
  </visual>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="0.2 0.3 0.4" rgb2="0.1 0.2 0.3" markrgb="0.8 0.8 0.8" width="300" height="300"/>
    <material name="groundplane" texture="groundplane" texuniform="true" texrepeat="5 5" reflectance="0.2"/>
    <material name="table_mat" rgba="0.48 0.38 0.26 1"/>
    <material name="red_block_mat" rgba="1 0.03 0.02 1"/>
  </asset>

  <worldbody>
    <light pos="0 0 1.5" dir="0 0 -1" directional="true"/>
    <geom name="floor" size="0 0 0.05" type="plane" material="groundplane"/>
    <body name="table" pos="0.55 0 0.025">
      <geom name="table_top" type="box" size="0.35 0.45 0.025" material="table_mat" friction="1 0.005 0.0001"/>
    </body>
    <body name="red_block" pos="0.50 0 0.085">
      <freejoint name="red_block_freejoint"/>
      <geom name="red_block_geom" type="box" size="0.035 0.035 0.035" material="red_block_mat" mass="0.05" friction="1 0.005 0.0001"/>
    </body>
  </worldbody>
</mujoco>
""",
    encoding="utf-8",
)

model = mujoco.MjModel.from_xml_path(str(PANDA_XML))
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

print("Loaded:", PANDA_XML)
print("Wrist camera:", "agent_wrist_camera")
print("Task block:", "red_block")
print("Tactile/GelSight path: disabled in this slim VLA test notebook")
print("nq:", model.nq, "nv:", model.nv, "nu:", model.nu)
print("timestep:", model.opt.timestep)

In [ ]:
# Cell 4: Task instruction and important scene IDs.
TASK_INSTRUCTION = "pick up the red block from the table and place it on the far side of the table"
OPENVLA_PROMPT = f"In: What action should the robot take to {TASK_INSTRUCTION}?\nOut: "

AGENT_CAMERA_NAME = "agent_wrist_camera"
START_BLOCK_POS = np.array([0.50, 0.00, 0.085], dtype=np.float64)
PLACE_BLOCK_POS = np.array([0.78, 0.00, 0.085], dtype=np.float64)

body_ids = {
    "hand": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "hand"),
    "left_finger": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "left_finger"),
    "right_finger": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "right_finger"),
    "red_block": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "red_block"),
    "agent_camera_mount": mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "agent_camera_mount"),
}
camera_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_CAMERA, AGENT_CAMERA_NAME)

print("Instruction:", TASK_INSTRUCTION)
print("OpenVLA prompt:", repr(OPENVLA_PROMPT))
print("Body IDs:", body_ids)
print("Camera ID:", camera_id)

print("\nActuators:")
for actuator_id in range(model.nu):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, actuator_id)
    trntype = mujoco.mjtTrn(model.actuator_trntype[actuator_id])
    target_id = model.actuator_trnid[actuator_id, 0]
    if trntype == mujoco.mjtTrn.mjTRN_JOINT:
        target = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, target_id)
    elif trntype == mujoco.mjtTrn.mjTRN_TENDON:
        target = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_TENDON, target_id)
    else:
        target = f"id={target_id}"
    print(f" - {actuator_id}: {name} -> {trntype.name}:{target}, ctrlrange={model.actuator_ctrlrange[actuator_id]}")

In [ ]:
# Cell 5: OpenVLA execution setup.
# Default local mode is chunked worker isolation: OpenVLA loads only inside an external child process.
# The worker answers a few policy queries, exits to release VRAM, then restarts if needed.
RUN_OPENVLA = True
HF_TOKEN_IN_NOTEBOOK = _optional_secret("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN", "")

OPENVLA_EXECUTION_MODE = "chunked_worker"  # Recommended for GTX 1050 Ti. "kernel_hybrid" is riskier.
OPENVLA_LOAD_MODE = "hybrid_4bit_vision_cpu"
RAISE_ON_OPENVLA_LOAD_ERROR = False

LOCAL_HF_CACHE_DIR = Path(r"D:\Origins\Polimi\Thesis\Codes\Cache")
LOCAL_HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(LOCAL_HF_CACHE_DIR)
os.environ["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE_DIR / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(LOCAL_HF_CACHE_DIR / "transformers")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:64")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
for cache_subdir in [os.environ["HF_HUB_CACHE"], os.environ["TRANSFORMERS_CACHE"]]:
    Path(cache_subdir).mkdir(parents=True, exist_ok=True)

processor = None
vla = None
COMPUTE_DTYPE = None
OPENVLA_INPUT_PLACEMENT = "chunked_worker"
OPENVLA_LOAD_STATUS = {"loaded": False, "reason": "chunked external worker configured"}
UNNORM_KEY = "bridge_orig"
MODEL_ID = "openvla/openvla-7b"
OFFLOAD_DIR = LOCAL_HF_CACHE_DIR / "openvla_offload"
OFFLOAD_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_OPENVLA:", RUN_OPENVLA)
print("OpenVLA execution mode:", OPENVLA_EXECUTION_MODE)
print("HF/cache directory:", LOCAL_HF_CACHE_DIR)
print("Note: in chunked-worker mode, Cell 5 does not keep OpenVLA loaded in the kernel.")
print("The model loads inside an external worker for a few queries, then exits to release VRAM.")

In [ ]:
# Cell 6: Slim controller, observation, and OpenVLA subprocess helper functions.
HAND_BODY_ID = body_ids["hand"]
LEFT_FINGER_BODY_ID = body_ids["left_finger"]
RIGHT_FINGER_BODY_ID = body_ids["right_finger"]
BLOCK_BODY_ID = body_ids["red_block"]
BLOCK_JOINT_ID = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, "red_block_freejoint")
BLOCK_QPOS_ADR = model.jnt_qposadr[BLOCK_JOINT_ID]

ARM_JOINT_NAMES = [f"joint{i}" for i in range(1, 8)]
ARM_JOINT_IDS = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name) for name in ARM_JOINT_NAMES]
ARM_QPOS_ADR = [model.jnt_qposadr[joint_id] for joint_id in ARM_JOINT_IDS]
ARM_DOF_ADR = [model.jnt_dofadr[joint_id] for joint_id in ARM_JOINT_IDS]

FINGER_JOINT_NAMES = ["finger_joint1", "finger_joint2"]
FINGER_JOINT_IDS = [mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, name) for name in FINGER_JOINT_NAMES]
FINGER_QPOS_ADR = [model.jnt_qposadr[joint_id] for joint_id in FINGER_JOINT_IDS]

HOME_QPOS = {
    "joint1": 0.0,
    "joint2": -0.785,
    "joint3": 0.0,
    "joint4": -2.356,
    "joint5": 0.0,
    "joint6": 1.571,
    "joint7": 0.785,
    "finger_joint1": 0.04,
    "finger_joint2": 0.04,
}


def reset_task_state():
    mujoco.mj_resetData(model, data)
    for joint_name, value in HOME_QPOS.items():
        joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
        data.qpos[model.jnt_qposadr[joint_id]] = value
    data.qpos[BLOCK_QPOS_ADR:BLOCK_QPOS_ADR + 7] = np.array([*START_BLOCK_POS, 1.0, 0.0, 0.0, 0.0])
    mujoco.mj_forward(model, data)


def current_arm_qpos_dict():
    return {
        joint_name: float(data.qpos[model.jnt_qposadr[joint_id]])
        for joint_name, joint_id in zip(ARM_JOINT_NAMES, ARM_JOINT_IDS)
    }


def set_arm_position_targets(joint_targets):
    for actuator_id in range(model.nu):
        if model.actuator_trntype[actuator_id] != mujoco.mjtTrn.mjTRN_JOINT:
            continue
        joint_id = model.actuator_trnid[actuator_id, 0]
        joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, joint_id)
        if joint_name not in joint_targets:
            continue
        value = joint_targets[joint_name]
        if model.actuator_ctrllimited[actuator_id]:
            low, high = model.actuator_ctrlrange[actuator_id]
            value = np.clip(value, low, high)
        data.ctrl[actuator_id] = value


def set_gripper_opening(opening_m):
    ctrl = float(np.clip(opening_m / 0.04 * 255.0, 0.0, 255.0))
    for actuator_id in range(model.nu):
        if model.actuator_trntype[actuator_id] == mujoco.mjtTrn.mjTRN_TENDON:
            data.ctrl[actuator_id] = ctrl


def solve_hand_position_ik(target_pos, seed_qpos=None, max_iter=250, tolerance=0.002):
    if seed_qpos is not None:
        for joint_name, value in seed_qpos.items():
            joint_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_JOINT, joint_name)
            data.qpos[model.jnt_qposadr[joint_id]] = value
    mujoco.mj_forward(model, data)

    jacp = np.zeros((3, model.nv), dtype=np.float64)
    jacr = np.zeros((3, model.nv), dtype=np.float64)

    for _ in range(max_iter):
        error = target_pos - data.xpos[HAND_BODY_ID]
        if np.linalg.norm(error) < tolerance:
            break

        mujoco.mj_jacBody(model, data, jacp, jacr, HAND_BODY_ID)
        jacobian = jacp[:, ARM_DOF_ADR]
        damping = 0.08
        delta_q = jacobian.T @ np.linalg.solve(
            jacobian @ jacobian.T + damping * damping * np.eye(3),
            error,
        )
        delta_q = np.clip(delta_q, -0.035, 0.035)

        for idx, qpos_adr in enumerate(ARM_QPOS_ADR):
            joint_id = ARM_JOINT_IDS[idx]
            low, high = model.jnt_range[joint_id]
            data.qpos[qpos_adr] = np.clip(data.qpos[qpos_adr] + delta_q[idx], low, high)

        mujoco.mj_forward(model, data)

    return current_arm_qpos_dict()


def capture_agent_observation(renderer):
    renderer.update_scene(data, camera=AGENT_CAMERA_NAME)
    rgb = renderer.render()
    return rgb, Image.fromarray(rgb).convert("RGB")


def _openvla_subprocess_script():
    return r"""
import gc, json, os, sys, time, traceback
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

image_path = Path(sys.argv[1])
prompt = sys.argv[2]
unnorm_key = sys.argv[3]
model_id = sys.argv[4]
cache_dir = Path(sys.argv[5])
offload_dir = Path(sys.argv[6])

os.environ["HF_HOME"] = str(cache_dir)
os.environ["HF_HUB_CACHE"] = str(cache_dir / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(cache_dir / "transformers")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:64")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

result = {"ok": False}
try:
    if not torch.cuda.is_available():
        raise RuntimeError("torch.cuda is not available in OpenVLA subprocess")
    dtype = torch.float16
    gc.collect()
    torch.cuda.empty_cache()
    load_start = time.perf_counter()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
        llm_int8_enable_fp32_cpu_offload=True,
    )
    vla = AutoModelForVision2Seq.from_pretrained(
        model_id,
        attn_implementation="eager",
        torch_dtype=dtype,
        quantization_config=quantization_config,
        device_map={"vision_backbone": "cpu", "projector": 0, "language_model": 0},
        offload_folder=str(offload_dir),
        offload_state_dict=True,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    vla.eval()
    load_s = time.perf_counter() - load_start

    image = Image.open(image_path).convert("RGB")
    preprocess_start = time.perf_counter()
    processed = processor(prompt, image)
    inputs = {}
    for key, value in processed.items():
        if not torch.is_tensor(value):
            inputs[key] = value
        elif key == "pixel_values":
            inputs[key] = value.to("cpu", dtype=dtype)
        elif torch.is_floating_point(value):
            inputs[key] = value.to("cuda:0", dtype=dtype)
        else:
            inputs[key] = value.to("cuda:0")
    torch.cuda.synchronize()
    preprocess_s = time.perf_counter() - preprocess_start

    model_start = time.perf_counter()
    with torch.inference_mode():
        action = vla.predict_action(**inputs, unnorm_key=unnorm_key, do_sample=False, use_cache=False)
    torch.cuda.synchronize()
    model_s = time.perf_counter() - model_start
    result = {
        "ok": True,
        "action": np.asarray(action, dtype=np.float32).reshape(-1).tolist(),
        "timings": {
            "vla_load_s": float(load_s),
            "vla_preprocess_s": float(preprocess_s),
            "vla_model_inference_s": float(model_s),
            "openvla_total_s": float(preprocess_s + model_s),
            "subprocess_total_s": float(load_s + preprocess_s + model_s),
        },
        "device_map": {"vision_backbone": "cpu", "projector": 0, "language_model": 0},
    }
except Exception as exc:
    result = {"ok": False, "error": repr(exc), "traceback": traceback.format_exc()[-4000:]}
print("OPENVLA_RESULT_JSON=" + json.dumps(result), flush=True)
sys.exit(0 if result.get("ok") else 2)
"""


def openvla_predict_from_image(image):
    if not globals().get("RUN_OPENVLA", False):
        return None
    import tempfile
    with tempfile.TemporaryDirectory(prefix="openvla04_query_") as tmp:
        image_path = Path(tmp) / "query.png"
        image.convert("RGB").save(image_path)
        env = os.environ.copy()
        env["HF_HOME"] = str(LOCAL_HF_CACHE_DIR)
        env["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE_DIR / "hub")
        env["TRANSFORMERS_CACHE"] = str(LOCAL_HF_CACHE_DIR / "transformers")
        token = globals().get("HF_TOKEN_IN_NOTEBOOK", "").strip()
        if token:
            env["HF_TOKEN"] = token
            env["HUGGINGFACE_HUB_TOKEN"] = token
        result = subprocess.run(
            [
                sys.executable,
                "-c",
                _openvla_subprocess_script(),
                str(image_path),
                OPENVLA_PROMPT,
                UNNORM_KEY,
                MODEL_ID,
                str(LOCAL_HF_CACHE_DIR),
                str(OFFLOAD_DIR),
            ],
            env=env,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=900,
            check=False,
        )
        marker = "OPENVLA_RESULT_JSON="
        payload = None
        for line in result.stdout.splitlines():
            if line.startswith(marker):
                payload = json.loads(line[len(marker):])
        if payload is None:
            raise RuntimeError("OpenVLA subprocess produced no result. stderr tail:\n" + result.stderr[-1600:])
        if not payload.get("ok"):
            raise RuntimeError("OpenVLA subprocess failed: " + payload.get("error", "unknown") + "\n" + payload.get("traceback", ""))
        timings = payload["timings"]
        timings["pipeline_total_s"] = timings.get("subprocess_total_s", timings.get("openvla_total_s", 0.0))
        return np.asarray(payload["action"], dtype=np.float32).reshape(-1), timings


def _openvla_chunked_worker_script():
    return r"""
import gc, json, os, sys, time, traceback
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

prompt_default = sys.argv[1]
unnorm_key_default = sys.argv[2]
model_id = sys.argv[3]
cache_dir = Path(sys.argv[4])
offload_dir = Path(sys.argv[5])

os.environ["HF_HOME"] = str(cache_dir)
os.environ["HF_HUB_CACHE"] = str(cache_dir / "hub")
os.environ["TRANSFORMERS_CACHE"] = str(cache_dir / "transformers")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "max_split_size_mb:64")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

ready_marker = "OPENVLA_WORKER_READY_JSON="
result_marker = "OPENVLA_WORKER_RESULT_JSON="

try:
    if not torch.cuda.is_available():
        raise RuntimeError("torch.cuda is not available in OpenVLA worker")

    dtype = torch.float16
    gc.collect()
    torch.cuda.empty_cache()

    load_start = time.perf_counter()
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=dtype,
        bnb_4bit_use_double_quant=True,
        llm_int8_enable_fp32_cpu_offload=True,
    )
    vla = AutoModelForVision2Seq.from_pretrained(
        model_id,
        attn_implementation="eager",
        torch_dtype=dtype,
        quantization_config=quantization_config,
        device_map={"vision_backbone": "cpu", "projector": 0, "language_model": 0},
        offload_folder=str(offload_dir),
        offload_state_dict=True,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    vla.eval()
    torch.cuda.synchronize()
    load_s = time.perf_counter() - load_start

    print(ready_marker + json.dumps({
        "ok": True,
        "vla_load_s": float(load_s),
        "device_map": {"vision_backbone": "cpu", "projector": 0, "language_model": 0},
    }), flush=True)

    query_count = 0
    for raw_line in sys.stdin:
        raw_line = raw_line.strip()
        if not raw_line:
            continue
        request = json.loads(raw_line)
        if request.get("cmd") == "quit":
            break

        request_id = request.get("request_id")
        response = {"ok": False, "request_id": request_id}
        try:
            image_path = Path(request["image_path"])
            prompt = request.get("prompt") or prompt_default
            unnorm_key = request.get("unnorm_key") or unnorm_key_default

            image = Image.open(image_path).convert("RGB")
            preprocess_start = time.perf_counter()
            processed = processor(prompt, image)
            inputs = {}
            for key, value in processed.items():
                if not torch.is_tensor(value):
                    inputs[key] = value
                elif key == "pixel_values":
                    inputs[key] = value.to("cpu", dtype=dtype)
                elif torch.is_floating_point(value):
                    inputs[key] = value.to("cuda:0", dtype=dtype)
                else:
                    inputs[key] = value.to("cuda:0")
            torch.cuda.synchronize()
            preprocess_s = time.perf_counter() - preprocess_start

            model_start = time.perf_counter()
            with torch.inference_mode():
                action = vla.predict_action(**inputs, unnorm_key=unnorm_key, do_sample=False, use_cache=False)
            torch.cuda.synchronize()
            model_s = time.perf_counter() - model_start

            response = {
                "ok": True,
                "request_id": request_id,
                "action": np.asarray(action, dtype=np.float32).reshape(-1).tolist(),
                "timings": {
                    "vla_load_s": float(load_s if query_count == 0 else 0.0),
                    "worker_load_s": float(load_s),
                    "vla_preprocess_s": float(preprocess_s),
                    "vla_model_inference_s": float(model_s),
                    "openvla_total_s": float(preprocess_s + model_s),
                    "worker_query_index": int(query_count),
                },
                "device_map": {"vision_backbone": "cpu", "projector": 0, "language_model": 0},
            }
            query_count += 1
            del image, processed, inputs
            gc.collect()
        except Exception as exc:
            response = {
                "ok": False,
                "request_id": request_id,
                "error": repr(exc),
                "traceback": traceback.format_exc()[-4000:],
            }
        print(result_marker + json.dumps(response), flush=True)

except Exception as exc:
    print(ready_marker + json.dumps({
        "ok": False,
        "error": repr(exc),
        "traceback": traceback.format_exc()[-4000:],
    }), flush=True)
    sys.exit(2)

try:
    del vla, processor
except Exception:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
sys.exit(0)
"""


class OpenVLAChunkedWorker:
    READY_MARKER = "OPENVLA_WORKER_READY_JSON="
    RESULT_MARKER = "OPENVLA_WORKER_RESULT_JSON="

    def __init__(self, chunk_index, start_timeout_s=420, query_timeout_s=420):
        import queue
        import threading

        self.chunk_index = int(chunk_index)
        self.query_count = 0
        self.start_timeout_s = float(start_timeout_s)
        self.query_timeout_s = float(query_timeout_s)
        self.stdout_queue = queue.Queue()
        self.stdout_tail = []
        self.stderr_path = PROJECT_DIR / f"openvla05_worker_chunk_{self.chunk_index:03d}.stderr.log"
        self.stderr_file = self.stderr_path.open("w", encoding="utf-8")

        env = os.environ.copy()
        env["HF_HOME"] = str(LOCAL_HF_CACHE_DIR)
        env["HF_HUB_CACHE"] = str(LOCAL_HF_CACHE_DIR / "hub")
        env["TRANSFORMERS_CACHE"] = str(LOCAL_HF_CACHE_DIR / "transformers")
        token = globals().get("HF_TOKEN_IN_NOTEBOOK", "").strip()
        if token:
            env["HF_TOKEN"] = token
            env["HUGGINGFACE_HUB_TOKEN"] = token

        self.process = subprocess.Popen(
            [
                sys.executable,
                "-u",
                "-c",
                _openvla_chunked_worker_script(),
                OPENVLA_PROMPT,
                UNNORM_KEY,
                MODEL_ID,
                str(LOCAL_HF_CACHE_DIR),
                str(OFFLOAD_DIR),
            ],
            env=env,
            text=True,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=self.stderr_file,
            bufsize=1,
        )

        def reader():
            try:
                for line in self.process.stdout:
                    self.stdout_queue.put(line)
            finally:
                self.stdout_queue.put(None)

        self.reader_thread = threading.Thread(target=reader, daemon=True)
        self.reader_thread.start()

        ready = self._read_marker(self.READY_MARKER, self.start_timeout_s)
        if not ready.get("ok"):
            self.close(kill=True)
            raise RuntimeError(
                "OpenVLA worker failed to load: "
                + ready.get("error", "unknown")
                + "\n"
                + ready.get("traceback", "")
                + "\nStderr tail:\n"
                + self.stderr_tail()
            )
        self.load_timing = ready
        print(
            f"[openvla_worker] chunk={self.chunk_index} ready "
            f"load={ready.get('vla_load_s', float('nan')):.1f}s stderr={self.stderr_path.name}"
        )

    def _read_marker(self, marker, timeout_s):
        import queue
        import time as _time

        deadline = _time.perf_counter() + float(timeout_s)
        while True:
            remaining = deadline - _time.perf_counter()
            if remaining <= 0:
                self.close(kill=True)
                raise TimeoutError(
                    f"Timed out waiting for OpenVLA worker marker {marker!r}. "
                    f"Recent stdout: {self.stdout_tail[-8:]}\nStderr tail:\n{self.stderr_tail()}"
                )
            try:
                line = self.stdout_queue.get(timeout=remaining)
            except queue.Empty:
                continue
            if line is None:
                raise RuntimeError(
                    "OpenVLA worker exited before returning a result. "
                    f"Recent stdout: {self.stdout_tail[-8:]}\nStderr tail:\n{self.stderr_tail()}"
                )
            line = line.rstrip("\r\n")
            if line:
                self.stdout_tail.append(line)
                self.stdout_tail = self.stdout_tail[-30:]
            if line.startswith(marker):
                return json.loads(line[len(marker):])

    def stderr_tail(self, chars=2400):
        try:
            self.stderr_file.flush()
            if self.stderr_path.exists():
                text = self.stderr_path.read_text(encoding="utf-8", errors="replace")
                return text[-chars:]
        except Exception:
            pass
        return ""

    def predict_from_image_path(self, image_path, request_id):
        if self.process.poll() is not None:
            raise RuntimeError("OpenVLA worker is not running before prediction request.")
        payload = {
            "cmd": "predict",
            "request_id": int(request_id),
            "image_path": str(image_path),
            "prompt": OPENVLA_PROMPT,
            "unnorm_key": UNNORM_KEY,
        }
        send_start = time.perf_counter()
        self.process.stdin.write(json.dumps(payload) + "\n")
        self.process.stdin.flush()
        response = self._read_marker(self.RESULT_MARKER, self.query_timeout_s)
        if response.get("request_id") != int(request_id):
            self.close(kill=True)
            raise RuntimeError(f"OpenVLA worker returned mismatched request id: {response}")
        if not response.get("ok"):
            raise RuntimeError(
                "OpenVLA worker prediction failed: "
                + response.get("error", "unknown")
                + "\n"
                + response.get("traceback", "")
                + "\nStderr tail:\n"
                + self.stderr_tail()
            )
        timings = response["timings"]
        timings["worker_chunk_index"] = float(self.chunk_index)
        timings["worker_query_index"] = float(self.query_count)
        timings["worker_roundtrip_s"] = float(time.perf_counter() - send_start)
        timings["pipeline_total_s"] = timings["worker_roundtrip_s"]
        self.query_count += 1
        return np.asarray(response["action"], dtype=np.float32).reshape(-1), timings

    def close(self, kill=False):
        proc = getattr(self, "process", None)
        try:
            if proc is not None and proc.poll() is None:
                if kill:
                    proc.kill()
                else:
                    try:
                        proc.stdin.write(json.dumps({"cmd": "quit"}) + "\n")
                        proc.stdin.flush()
                        proc.wait(timeout=30)
                    except Exception:
                        proc.kill()
        finally:
            try:
                if getattr(self, "stderr_file", None):
                    self.stderr_file.close()
            except Exception:
                pass


class OpenVLAChunkedWorkerManager:
    def __init__(self, queries_per_load=3, cooldown_s=20, start_timeout_s=420, query_timeout_s=420):
        self.queries_per_load = int(queries_per_load)
        self.cooldown_s = float(cooldown_s)
        self.start_timeout_s = float(start_timeout_s)
        self.query_timeout_s = float(query_timeout_s)
        self.chunk_index = 0
        self.worker = None

    def _ensure_worker(self):
        if self.worker is not None and self.worker.query_count < self.queries_per_load:
            return
        self.close_current()
        if self.chunk_index > 0 and self.cooldown_s > 0:
            print(f"[openvla_worker] cooling down {self.cooldown_s:.0f}s before next chunk")
            time.sleep(self.cooldown_s)
        self.worker = OpenVLAChunkedWorker(
            self.chunk_index,
            start_timeout_s=self.start_timeout_s,
            query_timeout_s=self.query_timeout_s,
        )
        self.chunk_index += 1

    def predict_from_image_path(self, image_path, request_id):
        self._ensure_worker()
        return self.worker.predict_from_image_path(image_path, request_id)

    def close_current(self):
        if self.worker is not None:
            self.worker.close()
            self.worker = None

    def close(self):
        self.close_current()



OPENVLA_ACTION_LABELS = ["delta_x", "delta_y", "delta_z", "delta_roll", "delta_pitch", "delta_yaw", "gripper"]


def safe_hz(seconds):
    seconds = float(seconds)
    return float(1.0 / seconds) if seconds > 0 else float("nan")


def summarize_samples(samples):
    samples = np.asarray(samples, dtype=np.float64)
    samples = samples[np.isfinite(samples)]
    if samples.size == 0:
        return {"count": 0, "mean_s": None, "median_s": None, "min_s": None, "max_s": None, "std_s": None, "mean_hz": None}
    mean_s = float(samples.mean())
    return {
        "count": int(samples.size),
        "mean_s": mean_s,
        "median_s": float(np.median(samples)),
        "min_s": float(samples.min()),
        "max_s": float(samples.max()),
        "std_s": float(samples.std()),
        "mean_hz": safe_hz(mean_s),
    }


def build_timing_summary(vla_log, low_level_step_times=None, low_level_render_times=None):
    low_level_step_times = low_level_step_times or []
    low_level_render_times = low_level_render_times or []
    return {
        "vla_model_inference": summarize_samples([entry.get("vla_model_inference_s", entry.get("inference_time_s")) for entry in vla_log]),
        "vla_load": summarize_samples([entry.get("vla_load_s") for entry in vla_log]),
        "vla_preprocess": summarize_samples([entry.get("vla_preprocess_s") for entry in vla_log]),
        "openvla_total": summarize_samples([entry.get("openvla_total_s") for entry in vla_log]),
        "whole_vla_pipeline_step": summarize_samples([entry.get("pipeline_total_s") for entry in vla_log]),
        "low_level_mujoco_step": summarize_samples(low_level_step_times),
        "video_frame_render_step": summarize_samples(low_level_render_times),
    }


def print_timing_summary(summary):
    print("\nTiming summary")
    labels = [
        ("VLA worker/chunk load", "vla_load"),
        ("VLA model inference only", "vla_model_inference"),
        ("VLA preprocessing", "vla_preprocess"),
        ("OpenVLA total", "openvla_total"),
        ("Whole VLA pipeline step", "whole_vla_pipeline_step"),
        ("Low-level MuJoCo step", "low_level_mujoco_step"),
        ("Video frame render step", "video_frame_render_step"),
    ]
    for label, key in labels:
        stats = summary.get(key, {})
        if not stats or stats.get("count", 0) == 0:
            print(f"  {label:28s}: no samples")
            continue
        print(
            f"  {label:28s}: mean={stats['mean_s']:.4f} s "
            f"({stats['mean_hz']:.2f} Hz), median={stats['median_s']:.4f} s, "
            f"min={stats['min_s']:.4f} s, max={stats['max_s']:.4f} s, n={stats['count']}"
        )


def format_action_lines(action):
    action = np.asarray(action, dtype=np.float32).reshape(-1)
    return [f"{label:>11s}: {value: .5f}" for label, value in zip(OPENVLA_ACTION_LABELS, action)]


def make_action_record(time_s, waypoint, action, timings):
    action = np.asarray(action, dtype=np.float32).reshape(-1)
    record = {
        "time_s": float(time_s),
        "waypoint": str(waypoint),
        "action": action.tolist(),
        "labels": OPENVLA_ACTION_LABELS[: len(action)],
    }
    record.update({key: float(value) for key, value in timings.items()})
    record["inference_time_s"] = record.get("vla_model_inference_s", record.get("openvla_total_s", 0.0))
    record["pipeline_hz"] = safe_hz(record.get("pipeline_total_s", 0.0))
    record["vla_model_hz"] = safe_hz(record.get("vla_model_inference_s", 0.0))
    return record


# === OpenVLA closed-loop controller adapter ===
OPENVLA_ROTATION_CONTROL_ENABLED = False
OPENVLA_ROTATION_NOTE = "rotation deltas are logged but ignored; current end-effector orientation is held implicitly by position-only IK"
OPENVLA_GRIPPER_THRESHOLD = 0.5  # OpenVLA/Bridge convention observed here: near 1=open, near 0=close.
OPENVLA_GRIPPER_OPENING_M = 0.04
OPENVLA_GRIPPER_CLOSED_M = 0.0
OPENVLA_TRANSLATION_SCALE = 1.0
OPENVLA_MAX_TRANSLATION_M = 0.02
OPENVLA_MAX_ROTATION_RAD = np.deg2rad(3.0)
OPENVLA_WORKSPACE_LOW = np.array([0.24, -0.25, 0.12], dtype=np.float64)
OPENVLA_WORKSPACE_HIGH = np.array([0.86, 0.25, 0.62], dtype=np.float64)


def get_ee_pose():
    pos = data.xpos[HAND_BODY_ID].copy()
    rotmat = data.xmat[HAND_BODY_ID].reshape(3, 3).copy()
    return {
        "position": pos,
        "rotation_matrix": rotmat,
        "position_list": pos.tolist(),
        "rotation_matrix_list": rotmat.tolist(),
    }


def clip_openvla_action(action):
    raw = np.asarray(action, dtype=np.float64).reshape(-1)
    if raw.size != 7:
        raise ValueError(f"Expected 7D OpenVLA action, got shape {raw.shape}: {raw}")
    clipped = raw.copy()
    clipped[:3] = np.clip(
        clipped[:3] * OPENVLA_TRANSLATION_SCALE,
        -OPENVLA_MAX_TRANSLATION_M,
        OPENVLA_MAX_TRANSLATION_M,
    )
    clipped[3:6] = np.clip(clipped[3:6], -OPENVLA_MAX_ROTATION_RAD, OPENVLA_MAX_ROTATION_RAD)
    clipped[6] = float(np.clip(clipped[6], 0.0, 1.0))
    return clipped.astype(np.float32)


def openvla_gripper_to_opening(gripper_value):
    gripper_value = float(np.clip(gripper_value, 0.0, 1.0))
    return OPENVLA_GRIPPER_OPENING_M if gripper_value >= OPENVLA_GRIPPER_THRESHOLD else OPENVLA_GRIPPER_CLOSED_M


def cartesian_delta_to_joint_target(delta_xyz, seed_qpos=None):
    ee_pose = get_ee_pose()
    unclipped_target = ee_pose["position"] + np.asarray(delta_xyz, dtype=np.float64)
    target_pos = np.clip(unclipped_target, OPENVLA_WORKSPACE_LOW, OPENVLA_WORKSPACE_HIGH)
    seed = current_arm_qpos_dict() if seed_qpos is None else seed_qpos

    # The existing IK helper uses the live MuJoCo data object internally.
    # Save and restore state so target planning does not secretly move the robot.
    saved_qpos = data.qpos.copy()
    saved_qvel = data.qvel.copy()
    saved_ctrl = data.ctrl.copy()
    try:
        qpos_target = solve_hand_position_ik(target_pos, seed_qpos=seed)
    finally:
        data.qpos[:] = saved_qpos
        data.qvel[:] = saved_qvel
        data.ctrl[:] = saved_ctrl
        mujoco.mj_forward(model, data)
    return qpos_target, target_pos, unclipped_target


def apply_openvla_action(action):
    ee_before = get_ee_pose()
    clipped = clip_openvla_action(action)
    qpos_target, target_pos, unclipped_target = cartesian_delta_to_joint_target(clipped[:3])
    gripper_opening = openvla_gripper_to_opening(clipped[6])
    return {
        "raw_action": np.asarray(action, dtype=np.float32).reshape(-1),
        "clipped_action": clipped,
        "ee_before": ee_before,
        "target_pos": target_pos,
        "unclipped_target_pos": np.asarray(unclipped_target, dtype=np.float64),
        "qpos_target": qpos_target,
        "gripper_opening": float(gripper_opening),
        "rotation_ignored": not OPENVLA_ROTATION_CONTROL_ENABLED,
        "rotation_note": OPENVLA_ROTATION_NOTE,
    }


def step_low_level_controller(qpos_target, gripper_opening, horizon_s=0.6, frame_dt=0.1, qpos_video_frames=None, observation_log=None, waypoint="openvla_closed_loop"):
    control_dt = model.opt.timestep
    steps = max(1, int(horizon_s / control_dt))
    next_frame_time = 0.0
    local_time = 0.0
    low_level_step_times = []
    start_qpos = current_arm_qpos_dict()
    start_gripper = float(np.mean(data.qpos[FINGER_QPOS_ADR]))

    for step_idx in range(steps):
        step_start = time.perf_counter()
        alpha = (step_idx + 1) / steps
        alpha = alpha * alpha * (3.0 - 2.0 * alpha)
        qpos_cmd = {
            joint_name: (1.0 - alpha) * start_qpos[joint_name] + alpha * qpos_target[joint_name]
            for joint_name in ARM_JOINT_NAMES
        }
        gripper_cmd = (1.0 - alpha) * start_gripper + alpha * gripper_opening
        set_arm_position_targets(qpos_cmd)
        set_gripper_opening(gripper_cmd)
        mujoco.mj_step(model, data)
        low_level_step_times.append(time.perf_counter() - step_start)
        local_time += control_dt

        if qpos_video_frames is not None and local_time >= next_frame_time:
            qpos_video_frames.append(data.qpos.copy())
            next_frame_time += frame_dt
        if observation_log is not None and local_time >= next_frame_time - frame_dt:
            observation_log.append({
                "local_time_s": float(local_time),
                "waypoint": waypoint,
                "hand_pos": data.xpos[HAND_BODY_ID].copy().tolist(),
                "block_pos_eval_only": data.xpos[BLOCK_BODY_ID].copy().tolist(),
                "gripper_qpos": data.qpos[FINGER_QPOS_ADR].copy().tolist(),
            })

    mujoco.mj_forward(model, data)
    return {
        "low_level_step_times": low_level_step_times,
        "ee_after": get_ee_pose(),
        "block_pos_eval_only": data.xpos[BLOCK_BODY_ID].copy().tolist(),
        "gripper_qpos_after": data.qpos[FINGER_QPOS_ADR].copy().tolist(),
    }


def render_agent_image_once(width=224, height=224):
    renderer = mujoco.Renderer(model, height=height, width=width)
    try:
        renderer.update_scene(data, camera=AGENT_CAMERA_NAME)
        rgb = renderer.render()
        return rgb, Image.fromarray(rgb).convert("RGB")
    finally:
        renderer.close()


def render_videos_from_qpos_frames(qpos_frames, world_video_path, agent_video_path, fps=10, world_size=(320, 240), agent_size=(224, 224)):
    if not qpos_frames:
        raise RuntimeError("No qpos frames were recorded; cannot render videos.")
    world_w, world_h = world_size
    agent_w, agent_h = agent_size
    saved_qpos = data.qpos.copy()
    world_renderer = mujoco.Renderer(model, height=world_h, width=world_w)
    agent_renderer = mujoco.Renderer(model, height=agent_h, width=agent_w)
    world_camera = mujoco.MjvCamera()
    world_camera.distance = 2.2
    world_camera.azimuth = 135
    world_camera.elevation = -25
    world_camera.lookat[:] = np.array([0.50, 0.00, 0.35])
    world_frames = []
    agent_frames = []
    try:
        for qpos in qpos_frames:
            data.qpos[:] = qpos
            mujoco.mj_forward(model, data)
            world_renderer.update_scene(data, camera=world_camera)
            world_frames.append(world_renderer.render())
            agent_renderer.update_scene(data, camera=AGENT_CAMERA_NAME)
            agent_frames.append(agent_renderer.render())
    finally:
        world_renderer.close()
        agent_renderer.close()
        data.qpos[:] = saved_qpos
        mujoco.mj_forward(model, data)
    media.write_video(world_video_path, world_frames, fps=fps)
    media.write_video(agent_video_path, agent_frames, fps=fps)
    return len(world_frames)


reset_task_state()
print("Initial hand position:", data.xpos[HAND_BODY_ID])
print("Initial block position:", data.xpos[BLOCK_BODY_ID])
print("OpenVLA execution mode:", globals().get("OPENVLA_EXECUTION_MODE", "subprocess"))
print("Tactile path: disabled")

In [ ]:
# Cell 7: scripted_baseline waypoint plan only.
# This is retained for comparison. It is not used by the OpenVLA closed-loop experiment.
SCRIPTED_BASELINE_CONTROL_MODE = "scripted_baseline"
RUN_SCRIPTED_BASELINE = False

WAYPOINT_SPECS = [
    ("home", np.array([0.307, 0.00, 0.590]), 0.04, 0.4),
    ("pregrasp", np.array([0.50, 0.00, 0.280]), 0.04, 1.5),
    ("grasp", np.array([0.50, 0.00, 0.180]), 0.04, 1.0),
    ("close_gripper", np.array([0.50, 0.00, 0.180]), 0.00, 0.8),
    ("lift", np.array([0.50, 0.00, 0.360]), 0.00, 1.0),
    ("transfer_far_side", np.array([0.72, 0.00, 0.360]), 0.00, 1.5),
    ("place", np.array([0.72, 0.00, 0.180]), 0.00, 1.0),
    ("open_gripper", np.array([0.72, 0.00, 0.180]), 0.04, 0.7),
    ("retreat", np.array([0.72, 0.00, 0.320]), 0.04, 1.0),
]

reset_task_state()
seed = current_arm_qpos_dict()
scripted_baseline_waypoints = []

for name, target_pos, gripper_opening, duration_s in WAYPOINT_SPECS:
    qpos_target = solve_hand_position_ik(target_pos, seed_qpos=seed)
    position_error = float(np.linalg.norm(target_pos - data.xpos[HAND_BODY_ID]))
    scripted_baseline_waypoints.append({
        "name": name,
        "target_pos": target_pos,
        "qpos": qpos_target,
        "gripper_opening": gripper_opening,
        "duration_s": duration_s,
        "ik_error_m": position_error,
    })
    seed = qpos_target
    print(f"[scripted_baseline] {name:18s} target={target_pos} hand={data.xpos[HAND_BODY_ID]} ik_error={position_error:.4f} m")

print("\nScripted baseline waypoints planned:", len(scripted_baseline_waypoints))
print("RUN_SCRIPTED_BASELINE:", RUN_SCRIPTED_BASELINE)
print("The OpenVLA closed-loop experiment does not call scripted_baseline_waypoints.")

In [ ]:
# Cell 8: Optional scripted_baseline runner.
# Disabled by default. The main experiment is Cell 9: openvla_closed_loop.
if not RUN_SCRIPTED_BASELINE:
    print("scripted_baseline is disabled. Set RUN_SCRIPTED_BASELINE=True only for baseline comparison.")
else:
    video_fps = 10
    qpos_video_frames = []
    low_level_step_times = []
    reset_task_state()
    set_arm_position_targets(scripted_baseline_waypoints[0]["qpos"])
    set_gripper_opening(scripted_baseline_waypoints[0]["gripper_opening"])
    mujoco.mj_forward(model, data)
    qpos_video_frames.append(data.qpos.copy())
    previous_qpos = scripted_baseline_waypoints[0]["qpos"].copy()
    previous_gripper = scripted_baseline_waypoints[0]["gripper_opening"]

    for waypoint in scripted_baseline_waypoints[1:]:
        result = step_low_level_controller(
            waypoint["qpos"],
            waypoint["gripper_opening"],
            horizon_s=waypoint["duration_s"],
            frame_dt=1.0 / video_fps,
            qpos_video_frames=qpos_video_frames,
            observation_log=None,
            waypoint="scripted_baseline:" + waypoint["name"],
        )
        low_level_step_times.extend(result["low_level_step_times"])
        print(f"[scripted_baseline] after {waypoint['name']:18s} hand={data.xpos[HAND_BODY_ID]} block={data.xpos[BLOCK_BODY_ID]}")

    final_block_pos = data.xpos[BLOCK_BODY_ID].copy()
    success = bool(final_block_pos[0] > 0.70 and abs(final_block_pos[1]) < 0.08 and 0.07 < final_block_pos[2] < 0.12)
    world_video_path = PROJECT_DIR / "openvla05_scripted_baseline_world.mp4"
    agent_video_path = PROJECT_DIR / "openvla05_scripted_baseline_agent_view.mp4"
    log_path = PROJECT_DIR / "openvla05_scripted_baseline_log.json"
    render_videos_from_qpos_frames(qpos_video_frames, world_video_path, agent_video_path, fps=video_fps)
    log_payload = {
        "control_mode": "scripted_baseline",
        "success": success,
        "final_block_pos": final_block_pos.tolist(),
        "scripted_waypoint_commands_used": len(scripted_baseline_waypoints),
        "openvla_queries_used_for_control": 0,
    }
    log_path.write_text(json.dumps(log_payload, indent=2), encoding="utf-8")
    print("scripted_baseline complete")
    print("Success:", success)
    print("Saved:", world_video_path, agent_video_path, log_path)

In [ ]:
# Cell 9: openvla_closed_loop main experiment.
# No scripted pick/place waypoints, no privileged object coordinates for control, no phase labels.
CONTROL_MODE = "openvla_closed_loop"
RUN_OPENVLA_CLOSED_LOOP = True
USE_ZERO_ACTIONS = False
USE_RANDOM_ACTIONS = False

# Laptop-safe chunked-worker defaults.
# 8 queries is still a short task attempt, but it is enough to see whether motion trends toward the block.
OPENVLA_CLOSED_LOOP_MAX_QUERIES = 16
CONTROL_HORIZON_S = 0.6
VIDEO_FPS = 10
STOP_ON_WORKSPACE_VIOLATION = True

OPENVLA_WORKER_ENABLED = True
OPENVLA_WORKER_QUERIES_PER_LOAD = 8
OPENVLA_COOLDOWN_BETWEEN_WORKERS_S = 20
OPENVLA_WORKER_START_TIMEOUT_S = 600
OPENVLA_WORKER_QUERY_TIMEOUT_S = 600

if USE_ZERO_ACTIONS and USE_RANDOM_ACTIONS:
    raise ValueError("Choose at most one sanity ablation: USE_ZERO_ACTIONS or USE_RANDOM_ACTIONS.")
if not RUN_OPENVLA_CLOSED_LOOP:
    raise RuntimeError("RUN_OPENVLA_CLOSED_LOOP is False. Enable it to run the OpenVLA closed-loop experiment.")
if not RUN_OPENVLA and not (USE_ZERO_ACTIONS or USE_RANDOM_ACTIONS):
    raise RuntimeError("OpenVLA is unavailable and no ablation is enabled. This cell will not fall back to scripted control.")

reset_task_state()
qpos_video_frames = [data.qpos.copy()]
closed_loop_log = []
observation_log = []
all_low_level_step_times = []
scripted_waypoint_commands_used = 0
openvla_actions_applied_to_controller = False
episode_stop_reason = "max_policy_queries_reached"
openvla_worker_manager = None

print("Control mode:", CONTROL_MODE)
print("Policy queries planned:", OPENVLA_CLOSED_LOOP_MAX_QUERIES)
print("Control horizon per action:", CONTROL_HORIZON_S, "s")
print("Sanity ablations: USE_ZERO_ACTIONS=", USE_ZERO_ACTIONS, "USE_RANDOM_ACTIONS=", USE_RANDOM_ACTIONS)
print("OpenVLA worker enabled:", OPENVLA_WORKER_ENABLED)
print("Worker queries per load:", OPENVLA_WORKER_QUERIES_PER_LOAD)
print("Worker cooldown between chunks:", OPENVLA_COOLDOWN_BETWEEN_WORKERS_S, "s")
print("Rotation handling:", OPENVLA_ROTATION_NOTE)
print("Gripper convention: action[6] >= ", OPENVLA_GRIPPER_THRESHOLD, "=> open, otherwise close")

try:
    if RUN_OPENVLA and OPENVLA_WORKER_ENABLED and not (USE_ZERO_ACTIONS or USE_RANDOM_ACTIONS):
        openvla_worker_manager = OpenVLAChunkedWorkerManager(
            queries_per_load=OPENVLA_WORKER_QUERIES_PER_LOAD,
            cooldown_s=OPENVLA_COOLDOWN_BETWEEN_WORKERS_S,
            start_timeout_s=OPENVLA_WORKER_START_TIMEOUT_S,
            query_timeout_s=OPENVLA_WORKER_QUERY_TIMEOUT_S,
        )

    for query_idx in range(OPENVLA_CLOSED_LOOP_MAX_QUERIES):
        sim_time = float(query_idx * CONTROL_HORIZON_S)
        ee_before = get_ee_pose()
        camera_rgb, camera_image = render_agent_image_once(width=224, height=224)
        camera_image_path = PROJECT_DIR / f"openvla05_closed_loop_query_{query_idx:03d}.png"
        camera_image.save(camera_image_path)

        if USE_ZERO_ACTIONS:
            raw_action = np.zeros(7, dtype=np.float32)
            raw_action[6] = 1.0
            timings = {"vla_model_inference_s": 0.0, "openvla_total_s": 0.0, "pipeline_total_s": 0.0}
            action_source = "zero_ablation"
        elif USE_RANDOM_ACTIONS:
            raw_action = np.random.uniform(-1.0, 1.0, size=7).astype(np.float32)
            raw_action[6] = np.random.uniform(0.0, 1.0)
            timings = {"vla_model_inference_s": 0.0, "openvla_total_s": 0.0, "pipeline_total_s": 0.0}
            action_source = "random_ablation"
        else:
            pipeline_start = time.perf_counter()
            if OPENVLA_WORKER_ENABLED:
                prediction = openvla_worker_manager.predict_from_image_path(camera_image_path, request_id=query_idx)
                action_source = "openvla_chunked_worker"
            else:
                prediction = openvla_predict_from_image(camera_image)
                action_source = "openvla_one_shot_subprocess"
            if prediction is None:
                raise RuntimeError("OpenVLA prediction returned None. Refusing to fall back to scripted control.")
            raw_action, timings = prediction
            timings["pipeline_total_s"] = float(time.perf_counter() - pipeline_start)

        adapter = apply_openvla_action(raw_action)
        low_level = step_low_level_controller(
            adapter["qpos_target"],
            adapter["gripper_opening"],
            horizon_s=CONTROL_HORIZON_S,
            frame_dt=1.0 / VIDEO_FPS,
            qpos_video_frames=qpos_video_frames,
            observation_log=observation_log,
            waypoint=CONTROL_MODE,
        )
        all_low_level_step_times.extend(low_level["low_level_step_times"])
        openvla_actions_applied_to_controller = openvla_actions_applied_to_controller or action_source.startswith("openvla")

        record = {
            "query_index": int(query_idx),
            "sim_time_s": sim_time,
            "camera_image_source": str(camera_image_path),
            "action_source": action_source,
            "raw_openvla_action": np.asarray(raw_action, dtype=np.float32).reshape(-1).tolist(),
            "clipped_action": adapter["clipped_action"].tolist(),
            "ee_pose_before": adapter["ee_before"]["position_list"],
            "ee_target_after_action": adapter["target_pos"].tolist(),
            "ee_unclipped_target_after_action": adapter["unclipped_target_pos"].tolist(),
            "ee_pose_after_low_level": low_level["ee_after"]["position_list"],
            "rotation_ignored": adapter["rotation_ignored"],
            "rotation_note": adapter["rotation_note"],
            "gripper_command_opening_m": adapter["gripper_opening"],
            "gripper_qpos_after": low_level["gripper_qpos_after"],
            "block_pose_eval_only": low_level["block_pos_eval_only"],
            "timings": {key: float(value) for key, value in timings.items()},
            "effective_policy_hz_including_load": safe_hz(timings.get("pipeline_total_s", timings.get("openvla_total_s", 0.0))),
        }
        closed_loop_log.append(record)

        print(
            f"[{CONTROL_MODE}] query={query_idx} source={action_source} "
            f"raw={record['raw_openvla_action']} clipped={record['clipped_action']} "
            f"target={record['ee_target_after_action']} after={record['ee_pose_after_low_level']} "
            f"gripper={record['gripper_command_opening_m']:.3f}"
        )

        ee_after = np.asarray(low_level["ee_after"]["position"], dtype=np.float64)
        if STOP_ON_WORKSPACE_VIOLATION and (np.any(ee_after < OPENVLA_WORKSPACE_LOW - 0.03) or np.any(ee_after > OPENVLA_WORKSPACE_HIGH + 0.03)):
            episode_stop_reason = "workspace_violation"
            print("Stopping: end-effector left the safety workspace.")
            break
finally:
    if openvla_worker_manager is not None:
        openvla_worker_manager.close()

final_block_pos = data.xpos[BLOCK_BODY_ID].copy()
success = bool(final_block_pos[0] > 0.70 and abs(final_block_pos[1]) < 0.08 and 0.07 < final_block_pos[2] < 0.12)

world_video_path = PROJECT_DIR / "openvla05_closed_loop_world.mp4"
agent_video_path = PROJECT_DIR / "openvla05_closed_loop_agent_view.mp4"
log_path = PROJECT_DIR / "openvla05_closed_loop_log.json"
frames_rendered = render_videos_from_qpos_frames(qpos_video_frames, world_video_path, agent_video_path, fps=VIDEO_FPS)

timing_summary = build_timing_summary(
    [
        {
            **entry["timings"],
            "pipeline_total_s": entry["timings"].get("pipeline_total_s", 0.0),
        }
        for entry in closed_loop_log
    ],
    all_low_level_step_times,
    [],
)

control_provenance = {
    "control_mode": CONTROL_MODE,
    "number_of_openvla_queries_used_for_control": int(sum(1 for entry in closed_loop_log if entry["action_source"].startswith("openvla"))),
    "number_of_scripted_waypoint_commands_used": scripted_waypoint_commands_used,
    "openvla_actions_applied_to_controller": bool(openvla_actions_applied_to_controller),
    "zero_action_ablation": bool(USE_ZERO_ACTIONS),
    "random_action_ablation": bool(USE_RANDOM_ACTIONS),
    "openvla_worker_enabled": bool(OPENVLA_WORKER_ENABLED),
    "openvla_worker_queries_per_load": int(OPENVLA_WORKER_QUERIES_PER_LOAD),
    "openvla_worker_cooldown_s": float(OPENVLA_COOLDOWN_BETWEEN_WORKERS_S),
}

log_payload = {
    "instruction": TASK_INSTRUCTION,
    "control_provenance": control_provenance,
    "stop_reason": episode_stop_reason,
    "success_eval_only": success,
    "start_block_pos_eval_only": START_BLOCK_POS.tolist(),
    "place_block_pos_eval_only": PLACE_BLOCK_POS.tolist(),
    "final_block_pos_eval_only": final_block_pos.tolist(),
    "workspace_low": OPENVLA_WORKSPACE_LOW.tolist(),
    "workspace_high": OPENVLA_WORKSPACE_HIGH.tolist(),
    "max_translation_m": OPENVLA_MAX_TRANSLATION_M,
    "rotation_control_enabled": OPENVLA_ROTATION_CONTROL_ENABLED,
    "rotation_note": OPENVLA_ROTATION_NOTE,
    "gripper_threshold": OPENVLA_GRIPPER_THRESHOLD,
    "control_horizon_s": CONTROL_HORIZON_S,
    "policy_queries": closed_loop_log,
    "observations": observation_log,
    "timing_summary": timing_summary,
    "videos": {
        "world": str(world_video_path),
        "agent": str(agent_video_path),
        "frames_rendered": int(frames_rendered),
    },
}
log_path.write_text(json.dumps(log_payload, indent=2), encoding="utf-8")

print("\nControl mode: OpenVLA closed-loop")
print("Number of OpenVLA queries used for control:", control_provenance["number_of_openvla_queries_used_for_control"])
print("Number of scripted waypoint commands used:", control_provenance["number_of_scripted_waypoint_commands_used"])
print("OpenVLA actions applied to controller:", control_provenance["openvla_actions_applied_to_controller"])
print("Task success from OpenVLA-driven motion:", success)
print("Stop reason:", episode_stop_reason)
print("Saved world video:", world_video_path)
print("Saved agent video:", agent_video_path)
print("Saved JSON log:", log_path)


In [ ]:
# Cell 10: Display the OpenVLA closed-loop videos and action provenance.
videos = [
    ("OpenVLA closed-loop external view", PROJECT_DIR / "openvla05_closed_loop_world.mp4"),
    ("OpenVLA closed-loop wrist camera", PROJECT_DIR / "openvla05_closed_loop_agent_view.mp4"),
]
log_path = PROJECT_DIR / "openvla05_closed_loop_log.json"

missing = [str(path) for _, path in videos if not path.exists()]
if missing:
    raise FileNotFoundError("Run Cell 9 first. Missing: " + ", ".join(missing))
if not log_path.exists():
    raise FileNotFoundError("Run Cell 9 first. Missing: " + str(log_path))

saved_log = json.loads(log_path.read_text(encoding="utf-8"))
provenance = saved_log.get("control_provenance", {})
policy_queries = saved_log.get("policy_queries", [])

video_cards = []
for title, path in videos:
    encoded_video = b64encode(path.read_bytes()).decode("ascii")
    video_cards.append(
        f"""
        <div style="min-width: 280px; max-width: 560px; flex: 1;">
          <div style="font-weight: 600; margin: 0 0 6px 0;">{escape(title)}</div>
          <video controls loop muted playsinline
                 style="width: 100%; border: 1px solid #ccc; background: #111;"
                 src="data:video/mp4;base64,{encoded_video}"></video>
          <div style="font-family: monospace; font-size: 12px; color: #666; margin-top: 4px;">
            {escape(str(path))}
          </div>
        </div>
        """
    )


def query_table_html(entries):
    if not entries:
        return "<div style='font-family: monospace; color: #666;'>No policy queries logged.</div>"
    blocks = []
    for entry in entries:
        raw = entry.get("raw_openvla_action", [])
        clipped = entry.get("clipped_action", [])
        rows = "".join(
            f"<tr><td>{escape(label)}</td><td style='text-align:right'>{float(r): .5f}</td><td style='text-align:right'>{float(c): .5f}</td></tr>"
            for label, r, c in zip(OPENVLA_ACTION_LABELS, raw, clipped)
        )
        timings = entry.get("timings", {})
        blocks.append(
            f"""
            <details open style="margin-bottom: 10px; border-bottom: 1px solid #ddd; padding-bottom: 8px;">
              <summary style="cursor: pointer; font-weight: 600;">
                query={entry.get('query_index')} | source={escape(str(entry.get('action_source')))} | pipeline={float(timings.get('pipeline_total_s', 0.0)):.2f}s
              </summary>
              <div style="font-family: monospace; font-size: 12px; margin-top: 6px;">
                ee before: {escape(str(entry.get('ee_pose_before')))}<br/>
                ee target: {escape(str(entry.get('ee_target_after_action')))}<br/>
                ee after: {escape(str(entry.get('ee_pose_after_low_level')))}<br/>
                gripper opening m: {float(entry.get('gripper_command_opening_m', 0.0)):.3f}
              </div>
              <table style="width:100%; font-family: monospace; font-size: 12px; border-collapse: collapse; margin-top: 6px;">
                <tr><th style='text-align:left'>Action</th><th>Raw</th><th>Clipped</th></tr>
                {rows}
              </table>
            </details>
            """
        )
    return "\n".join(blocks)


panel_html = f"""
<div style="display: grid; grid-template-columns: minmax(320px, 2fr) minmax(320px, 1fr); gap: 18px; align-items: start;">
  <div style="display: flex; gap: 16px; flex-wrap: wrap;">
    {"".join(video_cards)}
  </div>
  <div style="border: 1px solid #ccc; padding: 10px; max-height: 760px; overflow: auto; background: #fafafa;">
    <div style="font-weight: 700; margin-bottom: 8px;">Control provenance</div>
    <div style="font-family: monospace; font-size: 12px; line-height: 1.45;">
      mode: {escape(str(provenance.get('control_mode')))}<br/>
      OpenVLA queries used: {escape(str(provenance.get('number_of_openvla_queries_used_for_control')))}<br/>
      scripted waypoint commands used: {escape(str(provenance.get('number_of_scripted_waypoint_commands_used')))}<br/>
      OpenVLA actions applied: {escape(str(provenance.get('openvla_actions_applied_to_controller')))}<br/>
      success eval only: {escape(str(saved_log.get('success_eval_only')))}<br/>
      stop reason: {escape(str(saved_log.get('stop_reason')))}
    </div>
    <div style="font-weight: 700; margin: 14px 0 8px 0;">Policy Queries</div>
    {query_table_html(policy_queries)}
  </div>
</div>
"""

display(HTML(panel_html))

In [ ]:
# Cell 11: Native MuJoCo GUI viewer is disabled in this slim laptop notebook.
OPEN_NATIVE_GUI = False
if OPEN_NATIVE_GUI:
    raise RuntimeError("The native GUI path is intentionally disabled for this laptop VLA test. Use the videos from Cell 8 instead.")
print("Native MuJoCo GUI is disabled. Use Cell 8 videos instead.")

In [ ]:
# Cell 12: Native MuJoCo GUI replay is disabled in this slim laptop notebook.
OPEN_NATIVE_GUI_REPLAY = False
if OPEN_NATIVE_GUI_REPLAY:
    raise RuntimeError("The native GUI replay path is intentionally disabled for this laptop VLA test. Use the videos from Cell 8 instead.")
print("Native MuJoCo GUI replay is disabled. Use Cell 8 videos instead.")

## LoRA data collection append cells

Run the original notebook through Cell 6 first. Then run these cells.

These cells do not query OpenVLA. They use the original scripted expert trajectory only to collect demonstrations for later LoRA fine-tuning, with 10 Hz image/action samples and explicit metadata.


In [ ]:
# Cell A: Rebuild the Panda scene with the working right-finger grasp contact geometry.
# This intentionally keeps tactile rendering/data disabled. The GelSight-shaped body is used only
# as the physical right-finger contact pad that made the original scripted grasp work reliably.
import importlib.util

if importlib.util.find_spec("imageio") is None:
    pip_install(["imageio"])
import imageio.v2 as imageio

MENAGERIE_DIR = PROJECT_DIR / "mujoco_menagerie"
PANDA_DIR = MENAGERIE_DIR / "franka_emika_panda"
SOURCE_PANDA_XML = PANDA_DIR / "panda.xml"
PANDA_WITH_DATA_GRASP_XML = PANDA_DIR / "panda_openvla07_lora_data_grasp.xml"
PANDA_XML = PANDA_DIR / "openvla07_lora_data_scene.xml"

GELSIGHT_BODY_NAME = "gelsight_mini_right"
GELSIGHT_SITE_NAME = "gelsight_mini_right_site"
GELSIGHT_MOUNT_BODY_NAME = "right_finger"
GELSIGHT_GEL_PAD_GEOM_NAME = "gelsight_mini_right_gel_pad_geom"
GELSIGHT_GEOM_NAMES = [
    "gelsight_mini_right_case_geom",
    "gelsight_mini_right_gel_pad_geom",
    "gelsight_mini_right_label_geom",
    "gelsight_mini_right_cable_geom",
    "gelsight_mini_right_mount_plate_geom",
    "gelsight_mini_right_standoff_top_geom",
    "gelsight_mini_right_standoff_bottom_geom",
]


def _find_body(root, body_name):
    for body in root.iter("body"):
        if body.get("name") == body_name:
            return body
    raise ValueError(f"Could not find body named {body_name!r}")


def _remove_named_child_body(parent, child_name):
    for child in list(parent):
        if child.tag == "body" and child.get("name") == child_name:
            parent.remove(child)


if not SOURCE_PANDA_XML.exists():
    raise FileNotFoundError(f"Could not find Panda model at {SOURCE_PANDA_XML}. Run original Cell 3 first.")

panda_tree = ET.parse(SOURCE_PANDA_XML)
panda_root = panda_tree.getroot()
hand_body = _find_body(panda_root, "hand")
right_finger_body = _find_body(panda_root, GELSIGHT_MOUNT_BODY_NAME)

_remove_named_child_body(hand_body, "agent_camera_mount")
_remove_named_child_body(right_finger_body, GELSIGHT_BODY_NAME)

# Match the successful OpenVLA05 setup: remove the stock right-finger collision pads and replace
# that contact surface with a small, soft, high-friction gel pad. Left finger remains stock.
removed_right_contact_geoms = []
for child in list(right_finger_body):
    if child.tag != "geom":
        continue
    geom_class = child.get("class", "")
    is_stock_contact = geom_class == "collision" or geom_class.startswith("fingertip_pad_collision")
    if is_stock_contact:
        removed_right_contact_geoms.append(child.get("name", geom_class))
        right_finger_body.remove(child)

mount_body = ET.Element("body", name="agent_camera_mount", pos="0.08 0 -0.12")
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_boom_geom",
    type="capsule",
    fromto="0 0 0 -0.08 0 0.12",
    size="0.008",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "geom",
    name="agent_camera_body_geom",
    type="box",
    size="0.020 0.016 0.012",
    rgba="0.02 0.02 0.02 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    mount_body,
    "camera",
    name="agent_wrist_camera",
    mode="fixed",
    fovy="82",
    xyaxes="-0.000007 -1.000000 0.000034 -0.976592 -0.000000 -0.215100",
)

sensor_body = ET.Element("body", name=GELSIGHT_BODY_NAME, pos="0.0 0.01 0.045")
ET.SubElement(sensor_body, "inertial", mass="0.025", pos="0 0 0", diaginertia="3.6e-6 4.8e-6 1.8e-6")
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_mount_plate_geom",
    type="box",
    size="0.013 0.0012 0.019",
    pos="0 -0.011 0",
    rgba="0.08 0.08 0.08 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_standoff_top_geom",
    type="capsule",
    fromto="0.007 -0.010 0.012 0.007 -0.005 0.012",
    size="0.0016",
    rgba="0.02 0.02 0.025 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_standoff_bottom_geom",
    type="capsule",
    fromto="0.007 -0.010 -0.012 0.007 -0.005 -0.012",
    size="0.0016",
    rgba="0.02 0.02 0.025 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_case_geom",
    type="box",
    size="0.012 0.005 0.018",
    rgba="0.02 0.02 0.025 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name=GELSIGHT_GEL_PAD_GEOM_NAME,
    type="box",
    size="0.009 0.0012 0.013",
    pos="0 -0.0072 0",
    rgba="0.0 0.75 0.95 0.70",
    contype="1",
    conaffinity="1",
    condim="4",
    friction="1.0 0.005 0.0001",
    solimp="0.90 0.95 0.001",
    solref="0.004 1",
    group="3",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_label_geom",
    type="box",
    size="0.008 0.0010 0.0025",
    pos="0 -0.0074 0.015",
    rgba="0.92 0.92 0.88 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "geom",
    name="gelsight_mini_right_cable_geom",
    type="capsule",
    fromto="0 0.006 -0.016 0 0.018 -0.038",
    size="0.0018",
    rgba="0.01 0.01 0.01 1",
    contype="0",
    conaffinity="0",
    group="2",
)
ET.SubElement(
    sensor_body,
    "site",
    name=GELSIGHT_SITE_NAME,
    type="box",
    size="0.009 0.0010 0.013",
    pos="0 -0.0086 0",
    rgba="0.0 0.9 1.0 0.35",
    group="4",
)
right_finger_body.append(sensor_body)
hand_body.insert(0, mount_body)

ET.indent(panda_tree, space="  ")
panda_tree.write(PANDA_WITH_DATA_GRASP_XML, encoding="unicode")

PANDA_XML.write_text(
    f"""<mujoco model="openvla07 lora data panda pick place scene">
  <include file="{PANDA_WITH_DATA_GRASP_XML.name}"/>

  <statistic center="0.50 0 0.35" extent="1.1"/>

  <visual>
    <headlight diffuse="0.6 0.6 0.6" ambient="0.3 0.3 0.3" specular="0 0 0"/>
    <rgba haze="0.15 0.25 0.35 1"/>
    <global azimuth="120" elevation="-20" offwidth="1920" offheight="1080"/>
  </visual>

  <asset>
    <texture type="skybox" builtin="gradient" rgb1="0.3 0.5 0.7" rgb2="0 0 0" width="512" height="3072"/>
    <texture type="2d" name="groundplane" builtin="checker" mark="edge" rgb1="0.2 0.3 0.4" rgb2="0.1 0.2 0.3" markrgb="0.8 0.8 0.8" width="300" height="300"/>
    <material name="groundplane" texture="groundplane" texuniform="true" texrepeat="5 5" reflectance="0.2"/>
    <material name="table_mat" rgba="0.48 0.38 0.26 1"/>
    <material name="red_block_mat" rgba="1 0.03 0.02 1"/>
  </asset>

  <worldbody>
    <light pos="0 0 1.5" dir="0 0 -1" directional="true"/>
    <geom name="floor" size="0 0 0.05" type="plane" material="groundplane"/>
    <body name="table" pos="0.55 0 0.025">
      <geom name="table_top" type="box" size="0.35 0.45 0.025" material="table_mat" friction="1 0.005 0.0001"/>
    </body>
    <body name="red_block" pos="0.50 0 0.085">
      <freejoint name="red_block_freejoint"/>
      <geom name="red_block_geom" type="box" size="0.035 0.035 0.035" material="red_block_mat" mass="0.05" friction="1 0.005 0.0001"/>
    </body>
  </worldbody>
</mujoco>
""",
    encoding="utf-8",
)

model = mujoco.MjModel.from_xml_path(str(PANDA_XML))
data = mujoco.MjData(model)
mujoco.mj_forward(model, data)

print("Loaded LoRA data scene:", PANDA_XML)
print("Right-finger stock contact geoms removed:", removed_right_contact_geoms)
print("Gel pad contact geom:", GELSIGHT_GEL_PAD_GEOM_NAME)
print("Tactile images/data: disabled; this is only grasp contact geometry.")
print("nq:", model.nq, "nv:", model.nv, "nu:", model.nu, "timestep:", model.opt.timestep)


In [ ]:
# Cell B: Refresh scene IDs and controller helpers after replacing model/data in Cell A.
AGENT_CAMERA_NAME = "agent_wrist_camera"
TASK_INSTRUCTION = "pick up the red block from the table and place it on the far side of the table"
OPENVLA_PROMPT = f"In: What action should the robot take to {TASK_INSTRUCTION}?\nOut: "

START_BLOCK_POS = np.array([0.50, 0.00, 0.085], dtype=np.float64)
PLACE_BLOCK_POS = np.array([0.78, 0.00, 0.085], dtype=np.float64)


def require_mj_id(obj_type, name):
    obj_id = mujoco.mj_name2id(model, obj_type, name)
    if obj_id < 0:
        raise RuntimeError(f"Missing MuJoCo object {name!r} of type {obj_type}")
    return obj_id


body_ids = {
    "hand": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "hand"),
    "left_finger": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "left_finger"),
    "right_finger": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "right_finger"),
    "red_block": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "red_block"),
    "agent_camera_mount": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "agent_camera_mount"),
    GELSIGHT_BODY_NAME: require_mj_id(mujoco.mjtObj.mjOBJ_BODY, GELSIGHT_BODY_NAME),
}
site_ids = {GELSIGHT_SITE_NAME: require_mj_id(mujoco.mjtObj.mjOBJ_SITE, GELSIGHT_SITE_NAME)}
camera_id = require_mj_id(mujoco.mjtObj.mjOBJ_CAMERA, AGENT_CAMERA_NAME)

HAND_BODY_ID = body_ids["hand"]
LEFT_FINGER_BODY_ID = body_ids["left_finger"]
RIGHT_FINGER_BODY_ID = body_ids["right_finger"]
GELSIGHT_BODY_ID = body_ids[GELSIGHT_BODY_NAME]
GELSIGHT_SITE_ID = site_ids[GELSIGHT_SITE_NAME]
BLOCK_BODY_ID = body_ids["red_block"]
BLOCK_JOINT_ID = require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, "red_block_freejoint")
BLOCK_QPOS_ADR = model.jnt_qposadr[BLOCK_JOINT_ID]

ARM_JOINT_NAMES = [f"joint{i}" for i in range(1, 8)]
ARM_JOINT_IDS = [require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name) for name in ARM_JOINT_NAMES]
ARM_QPOS_ADR = [model.jnt_qposadr[joint_id] for joint_id in ARM_JOINT_IDS]
ARM_DOF_ADR = [model.jnt_dofadr[joint_id] for joint_id in ARM_JOINT_IDS]

FINGER_JOINT_NAMES = ["finger_joint1", "finger_joint2"]
FINGER_JOINT_IDS = [require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name) for name in FINGER_JOINT_NAMES]
FINGER_QPOS_ADR = [model.jnt_qposadr[joint_id] for joint_id in FINGER_JOINT_IDS]

HOME_QPOS = {
    "joint1": 0.0,
    "joint2": -0.785,
    "joint3": 0.0,
    "joint4": -2.356,
    "joint5": 0.0,
    "joint6": 1.571,
    "joint7": 0.785,
    "finger_joint1": 0.04,
    "finger_joint2": 0.04,
}


def current_arm_qpos_dict():
    return {
        joint_name: float(data.qpos[model.jnt_qposadr[joint_id]])
        for joint_name, joint_id in zip(ARM_JOINT_NAMES, ARM_JOINT_IDS)
    }


def current_finger_opening_m():
    return float(np.mean([data.qpos[qpos_adr] for qpos_adr in FINGER_QPOS_ADR]))


def set_arm_position_targets(joint_targets):
    for actuator_id in range(model.nu):
        if model.actuator_trntype[actuator_id] != mujoco.mjtTrn.mjTRN_JOINT:
            continue
        joint_id = model.actuator_trnid[actuator_id, 0]
        joint_name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_JOINT, joint_id)
        if joint_name not in joint_targets:
            continue
        value = float(joint_targets[joint_name])
        if model.actuator_ctrllimited[actuator_id]:
            low, high = model.actuator_ctrlrange[actuator_id]
            value = float(np.clip(value, low, high))
        data.ctrl[actuator_id] = value


def set_gripper_opening(opening_m):
    # Panda menagerie maps 0.04 m open to actuator ctrl=255 and 0 m closed to ctrl=0.
    ctrl = float(np.clip(opening_m / 0.04 * 255.0, 0.0, 255.0))
    for actuator_id in range(model.nu):
        if model.actuator_trntype[actuator_id] == mujoco.mjtTrn.mjTRN_TENDON:
            data.ctrl[actuator_id] = ctrl


def reset_task_state_randomized(cube_pos=None, robot_joint_noise=None, settle_steps=80):
    cube_pos = START_BLOCK_POS if cube_pos is None else np.asarray(cube_pos, dtype=np.float64)
    robot_joint_noise = {} if robot_joint_noise is None else robot_joint_noise
    mujoco.mj_resetData(model, data)
    data.qvel[:] = 0.0
    for joint_name, value in HOME_QPOS.items():
        joint_id = require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, joint_name)
        qpos_adr = model.jnt_qposadr[joint_id]
        if joint_name in ARM_JOINT_NAMES:
            value = float(value) + float(robot_joint_noise.get(joint_name, 0.0))
            low, high = model.jnt_range[joint_id]
            value = float(np.clip(value, low, high))
        data.qpos[qpos_adr] = value
    data.qpos[BLOCK_QPOS_ADR:BLOCK_QPOS_ADR + 7] = np.array([*cube_pos, 1.0, 0.0, 0.0, 0.0])
    set_arm_position_targets(current_arm_qpos_dict())
    set_gripper_opening(0.04)
    mujoco.mj_forward(model, data)
    for _ in range(settle_steps):
        set_arm_position_targets(current_arm_qpos_dict())
        set_gripper_opening(0.04)
        mujoco.mj_step(model, data)
    return cube_pos.copy()


def reset_task_state():
    return reset_task_state_randomized()


def solve_hand_position_ik(target_pos, seed_qpos=None, max_iter=300, tolerance=0.0015):
    target_pos = np.asarray(target_pos, dtype=np.float64)
    if seed_qpos is not None:
        for joint_name, value in seed_qpos.items():
            joint_id = require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, joint_name)
            data.qpos[model.jnt_qposadr[joint_id]] = float(value)
    mujoco.mj_forward(model, data)

    jacp = np.zeros((3, model.nv), dtype=np.float64)
    jacr = np.zeros((3, model.nv), dtype=np.float64)

    for _ in range(max_iter):
        error = target_pos - data.xpos[HAND_BODY_ID]
        if np.linalg.norm(error) < tolerance:
            break
        mujoco.mj_jacBody(model, data, jacp, jacr, HAND_BODY_ID)
        jacobian = jacp[:, ARM_DOF_ADR]
        damping = 0.08
        delta_q = jacobian.T @ np.linalg.solve(
            jacobian @ jacobian.T + damping * damping * np.eye(3),
            error,
        )
        delta_q = np.clip(delta_q, -0.035, 0.035)
        for idx, qpos_adr in enumerate(ARM_QPOS_ADR):
            joint_id = ARM_JOINT_IDS[idx]
            low, high = model.jnt_range[joint_id]
            data.qpos[qpos_adr] = np.clip(data.qpos[qpos_adr] + delta_q[idx], low, high)
        mujoco.mj_forward(model, data)
    return current_arm_qpos_dict()


def get_ee_pose():
    quat = np.empty(4, dtype=np.float64)
    mujoco.mju_mat2Quat(quat, data.xmat[HAND_BODY_ID].copy())
    return {
        "position": data.xpos[HAND_BODY_ID].copy(),
        "quat_wxyz": quat.copy(),
    }


def get_cube_pose():
    quat = np.empty(4, dtype=np.float64)
    mujoco.mju_mat2Quat(quat, data.xmat[BLOCK_BODY_ID].copy())
    return {
        "position": data.xpos[BLOCK_BODY_ID].copy(),
        "quat_wxyz": quat.copy(),
    }


def make_third_person_camera(lookat=None):
    cam = mujoco.MjvCamera()
    cam.distance = 2.2
    cam.azimuth = 135
    cam.elevation = -25
    cam.lookat[:] = np.array([0.55, 0.00, 0.35], dtype=np.float64) if lookat is None else lookat
    return cam


reset_task_state_randomized()
print("Refreshed IDs for LoRA data scene.")
print("Body IDs:", body_ids)
print("Camera ID:", camera_id)
print("Initial hand:", data.xpos[HAND_BODY_ID])
print("Initial cube:", data.xpos[BLOCK_BODY_ID])


In [ ]:
# Cell C: LoRA raw-data collection knobs.
# For smoke testing, keep this at 2 demos with videos ON. For the real 100-demo collection,
# set COLLECT_SMOKE_TEST=False and usually SAVE_EPISODE_VIDEOS=False to save disk.
from datetime import datetime

DATA_ROOT = Path(r"/home/fariborz/projects/force-vla-colab/loradata")
RUN_NAME = "panda_pickplace_commanded_gripper_contact"
COLLECT_SMOKE_TEST = False
SMOKE_TEST_DEMOS = 2
NUM_DEMONSTRATIONS = 300
TRAIN_FRACTION = 0.90

POLICY_HZ = 10.0
IMAGE_FORMAT = "jpg"
JPEG_QUALITY = 95

AGENTIC_IMAGE_SIZE = (256, 256)       # wrist/agentic image for OpenVLA-style observations
THIRD_PERSON_IMAGE_SIZE = (640, 480)  # useful for debugging and RLDS side observations

SAVE_EPISODE_VIDEOS = True            # requested ON for the first smoke tests
VIDEO_FPS = 10
AGENTIC_VIDEO_SIZE = (1080, 1080)
THIRD_PERSON_VIDEO_SIZE = (1920, 1080)

# Conservative randomization first. Widen only after smoke demos look physically correct.
CUBE_X_RANGE = (0.492, 0.508)
CUBE_Y_RANGE = (-0.010, 0.010)
TARGET_X_RANGE = (0.720, 0.750)
TARGET_Y_RANGE = (-0.012, 0.012)
ROBOT_JOINT_NOISE_RAD = 0.010

SUCCESS_X_MIN_MARGIN_M = -0.030
SUCCESS_Y_TOLERANCE_M = 0.090
SUCCESS_Z_RANGE_M = (0.070, 0.125)
REQUIRE_SUCCESSFUL_DEMOS = True
MAX_ATTEMPTS_MULTIPLIER = 4

GRIPPER_ACTION_CONVENTION = "commanded_open_fraction: 1.0=open, 0.0=closed; measured gripper_qpos is saved separately for debugging"
ROTATION_ACTION_CONVENTION = "droll/dpitch/dyaw are saved as 0.0 because this expert keeps wrist orientation implicit through IK seeds."

RUN_DIR = DATA_ROOT / f"{RUN_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

print("Data run directory:", RUN_DIR)
print("Demos to collect now:", SMOKE_TEST_DEMOS if COLLECT_SMOKE_TEST else NUM_DEMONSTRATIONS)
print("Policy/data sample rate Hz:", POLICY_HZ)
print("Save full-HD videos:", SAVE_EPISODE_VIDEOS)
print("Still image sizes:", {"agentic": AGENTIC_IMAGE_SIZE, "third_person": THIRD_PERSON_IMAGE_SIZE})


In [ ]:
# Cell D: Contact-faithful scripted expert recorder.
# This is for collecting imitation/fine-tuning demonstrations only. OpenVLA is not used here.


def sample_episode_randomization(rng):
    cube_pos = np.array([
        rng.uniform(*CUBE_X_RANGE),
        rng.uniform(*CUBE_Y_RANGE),
        0.085,
    ], dtype=np.float64)
    target_pos = np.array([
        rng.uniform(*TARGET_X_RANGE),
        rng.uniform(*TARGET_Y_RANGE),
        0.085,
    ], dtype=np.float64)
    robot_noise = {
        name: rng.uniform(-ROBOT_JOINT_NOISE_RAD, ROBOT_JOINT_NOISE_RAD)
        for name in ARM_JOINT_NAMES
    }
    return cube_pos, target_pos, robot_noise


def build_pickplace_waypoint_specs(cube_pos, target_pos):
    # The high-level expert is intentionally the original pick-place path, parameterized only
    # by randomized cube and target positions for data collection.
    cube_xy = np.asarray(cube_pos[:2], dtype=np.float64)
    target_xy = np.asarray(target_pos[:2], dtype=np.float64)
    return [
        ("home", np.array([0.307, 0.00, 0.590], dtype=np.float64), 0.04, 0.4),
        ("pregrasp", np.array([cube_xy[0], cube_xy[1], 0.280], dtype=np.float64), 0.04, 1.5),
        ("grasp", np.array([cube_xy[0], cube_xy[1], 0.180], dtype=np.float64), 0.04, 1.0),
        ("close_gripper", np.array([cube_xy[0], cube_xy[1], 0.180], dtype=np.float64), 0.00, 0.8),
        ("lift", np.array([cube_xy[0], cube_xy[1], 0.360], dtype=np.float64), 0.00, 1.0),
        ("transfer_far_side", np.array([target_xy[0], target_xy[1], 0.360], dtype=np.float64), 0.00, 1.5),
        ("place", np.array([target_xy[0], target_xy[1], 0.180], dtype=np.float64), 0.00, 1.0),
        ("open_gripper", np.array([target_xy[0], target_xy[1], 0.180], dtype=np.float64), 0.04, 0.7),
        ("retreat", np.array([target_xy[0], target_xy[1], 0.320], dtype=np.float64), 0.04, 1.0),
    ]


def plan_expert_waypoints(cube_pos, target_pos, robot_noise=None):
    reset_task_state_randomized(cube_pos=cube_pos, robot_joint_noise=robot_noise)
    seed = current_arm_qpos_dict()
    waypoints = []
    for name, target_pos_m, gripper_opening, duration_s in build_pickplace_waypoint_specs(cube_pos, target_pos):
        qpos_target = solve_hand_position_ik(target_pos_m, seed_qpos=seed)
        ik_error_m = float(np.linalg.norm(target_pos_m - data.xpos[HAND_BODY_ID]))
        waypoints.append({
            "name": name,
            "target_pos": target_pos_m.copy(),
            "qpos": qpos_target.copy(),
            "gripper_opening": float(gripper_opening),
            "duration_s": float(duration_s),
            "ik_error_m": ik_error_m,
        })
        seed = qpos_target
    return waypoints


def episode_success(final_cube_pos, target_pos):
    final_cube_pos = np.asarray(final_cube_pos, dtype=np.float64)
    target_pos = np.asarray(target_pos, dtype=np.float64)
    far_enough = final_cube_pos[0] >= target_pos[0] + SUCCESS_X_MIN_MARGIN_M
    y_ok = abs(final_cube_pos[1] - target_pos[1]) <= SUCCESS_Y_TOLERANCE_M
    z_ok = SUCCESS_Z_RANGE_M[0] <= final_cube_pos[2] <= SUCCESS_Z_RANGE_M[1]
    return bool(far_enough and y_ok and z_ok)


def save_jpg(rgb, path, quality=JPEG_QUALITY):
    Image.fromarray(np.asarray(rgb, dtype=np.uint8)).save(path, quality=quality, optimize=True)


def relative_to_episode(path, episode_dir):
    return str(Path(path).resolve().relative_to(Path(episode_dir).resolve())).replace("\\", "/")


def open_video_writer(path, fps):
    path.parent.mkdir(parents=True, exist_ok=True)
    return imageio.get_writer(
        path,
        fps=fps,
        codec="libx264",
        quality=8,
        macro_block_size=1,
    )


def render_with_renderer(renderer, camera):
    renderer.update_scene(data, camera=camera)
    return renderer.render()


def collect_one_expert_episode(episode_id, split, seed, run_dir=RUN_DIR):
    rng = np.random.default_rng(seed)
    cube_pos, target_pos, robot_noise = sample_episode_randomization(rng)
    waypoints = plan_expert_waypoints(cube_pos, target_pos, robot_noise=robot_noise)

    episode_dir = run_dir / split / f"episode_{episode_id:06d}"
    agentic_dir = episode_dir / "images" / "agentic"
    third_dir = episode_dir / "images" / "third_person"
    agentic_dir.mkdir(parents=True, exist_ok=True)
    third_dir.mkdir(parents=True, exist_ok=True)

    agentic_renderer = mujoco.Renderer(model, height=AGENTIC_IMAGE_SIZE[1], width=AGENTIC_IMAGE_SIZE[0])
    third_renderer = mujoco.Renderer(model, height=THIRD_PERSON_IMAGE_SIZE[1], width=THIRD_PERSON_IMAGE_SIZE[0])
    third_camera = make_third_person_camera()

    agentic_video_renderer = None
    third_video_renderer = None
    agentic_video_writer = None
    third_video_writer = None
    video_paths = {}
    if SAVE_EPISODE_VIDEOS:
        video_dir = episode_dir / "videos"
        agentic_video_path = video_dir / "agentic_full_hd.mp4"
        third_video_path = video_dir / "third_person_full_hd.mp4"
        agentic_video_renderer = mujoco.Renderer(model, height=AGENTIC_VIDEO_SIZE[1], width=AGENTIC_VIDEO_SIZE[0])
        third_video_renderer = mujoco.Renderer(model, height=THIRD_PERSON_VIDEO_SIZE[1], width=THIRD_PERSON_VIDEO_SIZE[0])
        agentic_video_writer = open_video_writer(agentic_video_path, VIDEO_FPS)
        third_video_writer = open_video_writer(third_video_path, VIDEO_FPS)
        video_paths = {
            "agentic_full_hd": relative_to_episode(agentic_video_path, episode_dir),
            "third_person_full_hd": relative_to_episode(third_video_path, episode_dir),
        }

    sample_dt = 1.0 / float(POLICY_HZ)
    snapshots = []
    qpos_samples = []
    step_id = 0
    next_sample_time = 0.0
    elapsed_s = 0.0

    def capture_snapshot(phase_name, commanded_gripper_opening):
        nonlocal step_id
        commanded_gripper_open_fraction = float(np.clip(commanded_gripper_opening / 0.04, 0.0, 1.0))
        measured_gripper_open_fraction = float(np.clip(current_finger_opening_m() / 0.04, 0.0, 1.0))
        agentic_path = agentic_dir / f"step_{step_id:06d}.{IMAGE_FORMAT}"
        third_path = third_dir / f"step_{step_id:06d}.{IMAGE_FORMAT}"

        agentic_rgb = render_with_renderer(agentic_renderer, AGENT_CAMERA_NAME)
        third_rgb = render_with_renderer(third_renderer, third_camera)
        save_jpg(agentic_rgb, agentic_path)
        save_jpg(third_rgb, third_path)

        if SAVE_EPISODE_VIDEOS:
            agentic_video_writer.append_data(render_with_renderer(agentic_video_renderer, AGENT_CAMERA_NAME))
            third_video_writer.append_data(render_with_renderer(third_video_renderer, third_camera))

        ee_pose = get_ee_pose()
        cube_pose = get_cube_pose()
        snapshot = {
            "episode_id": int(episode_id),
            "step_id": int(step_id),
            "time_s": float(elapsed_s),
            "phase": phase_name,
            "image_before_action": {
                "agentic": relative_to_episode(agentic_path, episode_dir),
                "third_person": relative_to_episode(third_path, episode_dir),
            },
            "language_instruction": TASK_INSTRUCTION,
            "end_effector_pose": {
                "position": ee_pose["position"].astype(float).tolist(),
                "quat_wxyz": ee_pose["quat_wxyz"].astype(float).tolist(),
            },
            "joint_qpos": {name: float(data.qpos[qpos_adr]) for name, qpos_adr in zip(ARM_JOINT_NAMES, ARM_QPOS_ADR)},
            "gripper_qpos": {name: float(data.qpos[qpos_adr]) for name, qpos_adr in zip(FINGER_JOINT_NAMES, FINGER_QPOS_ADR)},
            "cube_pose": {
                "position": cube_pose["position"].astype(float).tolist(),
                "quat_wxyz": cube_pose["quat_wxyz"].astype(float).tolist(),
            },
            "target_corner_pose": {
                "position": target_pos.astype(float).tolist(),
                "quat_wxyz": [1.0, 0.0, 0.0, 0.0],
            },
            "camera_name": {
                "agentic": AGENT_CAMERA_NAME,
                "third_person": "free_camera_az135_el-25",
            },
            "camera_parameters": {
                "agentic": {
                    "width": AGENTIC_IMAGE_SIZE[0],
                    "height": AGENTIC_IMAGE_SIZE[1],
                    "fovy": float(model.cam_fovy[camera_id]),
                },
                "third_person": {
                    "width": THIRD_PERSON_IMAGE_SIZE[0],
                    "height": THIRD_PERSON_IMAGE_SIZE[1],
                    "distance": float(third_camera.distance),
                    "azimuth": float(third_camera.azimuth),
                    "elevation": float(third_camera.elevation),
                    "lookat": third_camera.lookat.astype(float).tolist(),
                },
            },
            "random_seed": int(seed),
            "randomization": {
                "cube_start_position": cube_pos.astype(float).tolist(),
                "target_corner_position": target_pos.astype(float).tolist(),
                "robot_joint_noise_rad": {k: float(v) for k, v in robot_noise.items()},
            },
            "commanded_gripper_open_fraction": commanded_gripper_open_fraction,
            "measured_gripper_open_fraction": measured_gripper_open_fraction,
            "qpos": data.qpos.copy(),
        }
        snapshots.append(snapshot)
        qpos_samples.append(data.qpos.copy())
        step_id += 1

    try:
        reset_task_state_randomized(cube_pos=cube_pos, robot_joint_noise=robot_noise)
        set_arm_position_targets(waypoints[0]["qpos"])
        set_gripper_opening(waypoints[0]["gripper_opening"])
        mujoco.mj_forward(model, data)
        capture_snapshot("start", waypoints[0]["gripper_opening"])
        next_sample_time = sample_dt

        previous_qpos = waypoints[0]["qpos"].copy()
        previous_gripper = waypoints[0]["gripper_opening"]
        for waypoint in waypoints[1:]:
            steps = max(1, int(round(waypoint["duration_s"] / model.opt.timestep)))
            for low_step in range(steps):
                alpha = (low_step + 1) / steps
                joint_targets = {
                    name: (1.0 - alpha) * previous_qpos[name] + alpha * waypoint["qpos"][name]
                    for name in ARM_JOINT_NAMES
                }
                gripper_opening = (1.0 - alpha) * previous_gripper + alpha * waypoint["gripper_opening"]
                set_arm_position_targets(joint_targets)
                set_gripper_opening(gripper_opening)
                mujoco.mj_step(model, data)
                elapsed_s += float(model.opt.timestep)
                if elapsed_s + 1e-9 >= next_sample_time:
                    capture_snapshot(waypoint["name"], gripper_opening)
                    next_sample_time += sample_dt
            previous_qpos = waypoint["qpos"].copy()
            previous_gripper = waypoint["gripper_opening"]
            print(
                f"[episode {episode_id:06d}] after {waypoint['name']:18s} "
                f"hand={data.xpos[HAND_BODY_ID]} cube={data.xpos[BLOCK_BODY_ID]}"
            )

        final_cube_pos = data.xpos[BLOCK_BODY_ID].copy()
        success = episode_success(final_cube_pos, target_pos)

        # Convert snapshots to DROID/RLDS-builder-friendly JSONL records.
        steps_jsonl = episode_dir / "steps.jsonl"
        with steps_jsonl.open("w", encoding="utf-8") as f:
            for idx, snapshot in enumerate(snapshots):
                if idx < len(snapshots) - 1:
                    next_snapshot = snapshots[idx + 1]
                    cur_pos = np.asarray(snapshot["end_effector_pose"]["position"], dtype=np.float64)
                    next_pos = np.asarray(next_snapshot["end_effector_pose"]["position"], dtype=np.float64)
                    dpos = next_pos - cur_pos
                    gripper_action = float(next_snapshot["commanded_gripper_open_fraction"])
                else:
                    dpos = np.zeros(3, dtype=np.float64)
                    gripper_action = float(snapshot["commanded_gripper_open_fraction"])
                action_7d = [
                    float(dpos[0]),
                    float(dpos[1]),
                    float(dpos[2]),
                    0.0,
                    0.0,
                    0.0,
                    gripper_action,
                ]
                row = {k: v for k, v in snapshot.items() if k != "qpos"}
                row["action"] = action_7d
                row["done"] = bool(idx == len(snapshots) - 1)
                row["success"] = bool(success) if row["done"] else None
                row["control_source"] = "scripted_expert_for_lora_data"
                f.write(json.dumps(row) + "\n")

        np.savez_compressed(
            episode_dir / "trajectory_arrays.npz",
            qpos=np.asarray(qpos_samples, dtype=np.float64),
            final_cube_pos=final_cube_pos.astype(np.float64),
            target_pos=target_pos.astype(np.float64),
            cube_start_pos=cube_pos.astype(np.float64),
        )

        metadata = {
            "episode_id": int(episode_id),
            "split": split,
            "seed": int(seed),
            "success": bool(success),
            "num_steps": int(len(snapshots)),
            "policy_hz": float(POLICY_HZ),
            "language_instruction": TASK_INSTRUCTION,
            "action_format": "[dx, dy, dz, droll, dpitch, dyaw, gripper]",
            "gripper_action_convention": GRIPPER_ACTION_CONVENTION,
            "gripper_action_source": "expert commanded gripper opening, not measured gripper qpos",
            "rotation_action_convention": ROTATION_ACTION_CONVENTION,
            "image_modalities": ["agentic", "third_person"],
            "video_paths": video_paths,
            "scene_xml": str(PANDA_XML),
            "contact_note": "Right finger uses OpenVLA05 GelSight-shaped contact pad; tactile rendering is not saved.",
            "cube_start_position": cube_pos.astype(float).tolist(),
            "target_corner_position": target_pos.astype(float).tolist(),
            "final_cube_position": final_cube_pos.astype(float).tolist(),
            "waypoints": [
                {
                    "name": w["name"],
                    "target_pos": w["target_pos"].astype(float).tolist(),
                    "gripper_opening": float(w["gripper_opening"]),
                    "duration_s": float(w["duration_s"]),
                    "ik_error_m": float(w["ik_error_m"]),
                }
                for w in waypoints
            ],
        }
        (episode_dir / "episode_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
        return metadata

    finally:
        for writer in [agentic_video_writer, third_video_writer]:
            if writer is not None:
                writer.close()
        for renderer in [agentic_renderer, third_renderer, agentic_video_renderer, third_video_renderer]:
            if renderer is not None:
                renderer.close()


def collect_lora_dataset():
    total = SMOKE_TEST_DEMOS if COLLECT_SMOKE_TEST else NUM_DEMONSTRATIONS
    train_count = int(round(total * TRAIN_FRACTION))
    if total >= 2:
        train_count = min(max(1, train_count), total - 1)
    max_attempts = max(total, total * MAX_ATTEMPTS_MULTIPLIER)
    successful_metadata = []
    manifest_rows = []
    base_seed = 7707
    attempt = 0
    while len(successful_metadata) < total and attempt < max_attempts:
        episode_id = len(successful_metadata)
        split = "train" if episode_id < train_count else "test"
        seed = base_seed + attempt
        print(f"\nCollecting attempt {attempt + 1}/{max_attempts}: episode={episode_id:06d}, split={split}, seed={seed}")
        try:
            metadata = collect_one_expert_episode(episode_id=episode_id, split=split, seed=seed)
        except Exception as exc:
            print("Episode failed with exception:", repr(exc))
            metadata = None
        attempt += 1
        if metadata is None:
            continue
        print("Episode success:", metadata["success"], "final cube:", metadata["final_cube_position"])
        if REQUIRE_SUCCESSFUL_DEMOS and not metadata["success"]:
            print("Discarding failed demo because REQUIRE_SUCCESSFUL_DEMOS=True")
            continue
        successful_metadata.append(metadata)
        manifest_rows.append({
            "episode_id": metadata["episode_id"],
            "split": metadata["split"],
            "episode_dir": f"{metadata['split']}/episode_{metadata['episode_id']:06d}",
            "success": metadata["success"],
            "num_steps": metadata["num_steps"],
            "seed": metadata["seed"],
        })

    if len(successful_metadata) < total:
        raise RuntimeError(f"Collected {len(successful_metadata)} demos, but requested {total}. Check grasp quality and randomization ranges.")

    with (RUN_DIR / "manifest.jsonl").open("w", encoding="utf-8") as f:
        for row in manifest_rows:
            f.write(json.dumps(row) + "\n")
    dataset_metadata = {
        "run_dir": str(RUN_DIR),
        "num_episodes": len(successful_metadata),
        "policy_hz": float(POLICY_HZ),
        "task_instruction": TASK_INSTRUCTION,
        "action_format": "[dx, dy, dz, droll, dpitch, dyaw, gripper]",
        "gripper_action_convention": GRIPPER_ACTION_CONVENTION,
        "gripper_action_source": "expert commanded gripper opening, not measured gripper qpos",
        "rotation_action_convention": ROTATION_ACTION_CONVENTION,
        "image_before_action_modalities": ["agentic", "third_person"],
        "saved_video": bool(SAVE_EPISODE_VIDEOS),
        "ready_for_rlds_builder": True,
        "note": "Raw JSONL/JPG structure. Convert to RLDS with a dataset builder before OpenVLA LoRA fine-tuning.",
    }
    (RUN_DIR / "dataset_metadata.json").write_text(json.dumps(dataset_metadata, indent=2), encoding="utf-8")
    print("\nCollection complete:", RUN_DIR)
    print("Episodes:", len(successful_metadata))
    return successful_metadata


In [ ]:
# Cell E: Run the smoke collection now.
# Expected for default settings: 2 successful demos, 10 Hz image/action samples, and full-HD videos.
collected_episode_metadata = collect_lora_dataset()

print("\nSmoke run saved at:", RUN_DIR)
for metadata in collected_episode_metadata:
    print(
        f"episode={metadata['episode_id']:06d} split={metadata['split']} "
        f"success={metadata['success']} steps={metadata['num_steps']} "
        f"final_cube={np.round(metadata['final_cube_position'], 4).tolist()} "
        f"target={np.round(metadata['target_corner_position'], 4).tolist()}"
    )

first_ep = collected_episode_metadata[0]
first_episode_dir = RUN_DIR / first_ep["split"] / f"episode_{first_ep['episode_id']:06d}"
print("\nFirst episode files:")
print("  steps:", first_episode_dir / "steps.jsonl")
print("  metadata:", first_episode_dir / "episode_metadata.json")
print("  agentic images:", first_episode_dir / "images" / "agentic")
print("  third-person images:", first_episode_dir / "images" / "third_person")
if SAVE_EPISODE_VIDEOS:
    print("  videos:", first_episode_dir / "videos")

sample_third = sorted((first_episode_dir / "images" / "third_person").glob("*.jpg"))
sample_agentic = sorted((first_episode_dir / "images" / "agentic").glob("*.jpg"))
if sample_third:
    print("Preview third-person:", sample_third[min(10, len(sample_third) - 1)])
if sample_agentic:
    print("Preview agentic:", sample_agentic[min(10, len(sample_agentic) - 1)])


## Server: D3 OpenVLA-OFT fine-tuning

Run Cells F through K in order. Cell K defaults to three optimizer steps and does not save a checkpoint. After that succeeds, set `TRAINING_MODE = "full"` in Cell K.

In [ ]:
# Cell F: Paths and subprocess helpers for the dedicated OpenVLA-OFT workflow.
from pathlib import Path
import importlib.metadata as importlib_metadata
import json
import os
import shlex
import shutil
import subprocess
import sys
import time

PROJECT_DIR = Path("/home/fariborz/projects/force-vla-colab")  # Adjust if this project moves.
OFT_REPO = Path("/home/fariborz/projects/openvla-oft")  # Official moojink/openvla-oft checkout.
OFT_PYTHON = Path("/home/fariborz/miniforge3/envs/openvla/bin/python")  # Python with RTX 5090 PyTorch.
if not OFT_PYTHON.exists():
    OFT_PYTHON = Path(sys.executable)
OFT_BIN_DIR = OFT_PYTHON.parent

D2_RAW_DIR = PROJECT_DIR / "loradata/panda_pickplace_rotation_recovery_v1_20260803_235336"
D3_RAW_DIR = PROJECT_DIR / "loradata/panda_pickplace_d3_nominal_boundary_v1"
D3_BUILDER_DIR = PROJECT_DIR / "rlds_dataset_builder/panda_pickplace_d3"
DATA_ROOT_DIR = PROJECT_DIR / "RLDS_TFDS"
DATASET_NAME = "panda_pickplace_d3"

PREPARE_D3_SCRIPT = PROJECT_DIR / "prepare_panda_d3.py"
OFT_SETUP_SCRIPT = PROJECT_DIR / "openvla_oft_d3_setup.py"
OFT_VERIFY_SCRIPT = PROJECT_DIR / "openvla_oft_d3_verify.py"
RUN_ROOT_DIR = PROJECT_DIR / "openvla_oft_d3_runs"
HF_CACHE_DIR = PROJECT_DIR / "hf_cache"

HF_TOKEN_IN_NOTEBOOK = ""  # Optional: use an environment variable instead when possible.
USE_WANDB_OFFLINE = False  # Set True to log locally without contacting wandb.ai.
RUN_OFT_TRAINING = True  # Set False to print/check configuration without launching training.

for path in (RUN_ROOT_DIR, HF_CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

def oft_env(extra=None):
    env = os.environ.copy()
    env["HF_HOME"] = str(HF_CACHE_DIR)
    env["HF_HUB_CACHE"] = str(HF_CACHE_DIR / "hub")
    env.pop("TRANSFORMERS_CACHE", None)
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    env["TF_CPP_MIN_LOG_LEVEL"] = "3"
    env["WANDB_MODE"] = "offline" if USE_WANDB_OFFLINE else "online"
    if HF_TOKEN_IN_NOTEBOOK.strip():
        env["HF_TOKEN"] = HF_TOKEN_IN_NOTEBOOK.strip()
        env["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN_IN_NOTEBOOK.strip()
    repo_path = str(OFT_REPO)
    old_pythonpath = env.get("PYTHONPATH", "")
    env["PYTHONPATH"] = repo_path + (os.pathsep + old_pythonpath if old_pythonpath else "")
    cuda_libs = sorted((OFT_PYTHON.parent.parent / "lib").glob("python*/site-packages/nvidia/cu13/lib"))
    if cuda_libs:
        old_ld = env.get("LD_LIBRARY_PATH", "")
        env["LD_LIBRARY_PATH"] = str(cuda_libs[0]) + (os.pathsep + old_ld if old_ld else "")
    if extra:
        env.update({str(key): str(value) for key, value in extra.items()})
    return env

def run(cmd, cwd=None, check=True, env=None):
    pretty = " ".join(shlex.quote(str(part)) for part in cmd)
    print("$", pretty)
    result = subprocess.run(
        [str(part) for part in cmd],
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {pretty}")
    return result

print("Notebook Python:", sys.executable)
print("OFT Python:", OFT_PYTHON)
print("OFT repository:", OFT_REPO)
print("D3 raw directory:", D3_RAW_DIR)
print("D3 RLDS directory:", DATA_ROOT_DIR / DATASET_NAME / "1.0.0")
print("Run directory:", RUN_ROOT_DIR)

In [ ]:
# Cell G: Create D3 as an exact filtered view of D2 and audit its mode counts.
# Adjustable only if the source/output dataset locations change.
for required in (PREPARE_D3_SCRIPT, D2_RAW_DIR / "manifest.jsonl"):
    assert required.exists(), f"Missing: {required}"

prepared = run(
    [
        OFT_PYTHON,
        PREPARE_D3_SCRIPT,
        "--source", D2_RAW_DIR,
        "--output", D3_RAW_DIR,
    ],
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert "PANDA_D3_RAW_READY" in prepared.stdout

rows = [json.loads(line) for line in (D3_RAW_DIR / "manifest.jsonl").read_text().splitlines()]
assert len(rows) == 800
assert sum(row["trajectory_mode"] == "nominal" for row in rows) == 700
assert sum(row["trajectory_mode"] == "boundary" for row in rows) == 100
assert not any("recovery" in row["trajectory_mode"] for row in rows)
D3_RAW_VERIFIED = True
print("D3 confirmed: 700 nominal + 100 boundary + 0 recovery episodes.")

In [ ]:
# Cell H: Build the independent D3 RLDS dataset, or reuse its verified build.
REBUILD_D3_RLDS = False  # Set True only after intentionally changing the D3 builder or source selection.
D3_VERSION_DIR = DATA_ROOT_DIR / DATASET_NAME / "1.0.0"
builder_file = D3_BUILDER_DIR / "panda_pickplace_d3_dataset_builder.py"
assert globals().get("D3_RAW_VERIFIED", False), "Run Cell G first."
assert builder_file.is_file(), f"Missing builder: {builder_file}"

if REBUILD_D3_RLDS or not (D3_VERSION_DIR / "dataset_info.json").is_file():
    build_cmd = [
        OFT_PYTHON,
        "-m", "tensorflow_datasets.scripts.cli.main",
        "build",
        "--data_dir", DATA_ROOT_DIR,
    ]
    if REBUILD_D3_RLDS:
        build_cmd.append("--overwrite")
    run(
        build_cmd,
        cwd=D3_BUILDER_DIR,
        env=oft_env({"PANDA_D3_RAW_DIR": D3_RAW_DIR, "CUDA_VISIBLE_DEVICES": ""}),
    )
else:
    print("Reusing:", D3_VERSION_DIR)

split_probe = run(
    [
        OFT_PYTHON,
        "-c",
        (
            "import tensorflow_datasets as tfds; "
            f"b=tfds.builder('{DATASET_NAME}', data_dir=r'{DATA_ROOT_DIR}'); "
            "print(b.info.splits['train'].num_examples, b.info.splits['test'].num_examples)"
        ),
    ],
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert split_probe.stdout.strip().endswith("720 80")
D3_RLDS_VERIFIED = True
print("D3 RLDS confirmed: train=720, test=80.")

In [ ]:
# Cell I: Clone and install the official OpenVLA-OFT code without replacing RTX 5090 PyTorch.
if not OFT_REPO.exists():
    run(["git", "clone", "https://github.com/moojink/openvla-oft.git", OFT_REPO])
else:
    print("Reusing existing OFT checkout; no reset/pull is performed:", OFT_REPO)

dependency_specs = [
    "accelerate>=0.25.0", "draccus==0.8.0", "einops", "huggingface_hub",
    "json-numpy", "jsonlines", "matplotlib", "peft==0.11.1", "protobuf<4",
    "rich", "sentencepiece==0.1.99", "timm==0.9.10", "tokenizers==0.19.1",
    "wandb", "tensorflow==2.15.0", "tensorflow_datasets==4.9.3",
    "tensorflow_graphics==2021.12.3", "diffusers==0.30.3", "imageio",
]
run([OFT_PYTHON, "-m", "pip", "install", "--upgrade-strategy", "only-if-needed", *dependency_specs])

# OFT requires this Transformers fork for bidirectional attention and parallel decoding.
transformers_probe = run(
    [
        OFT_PYTHON,
        "-c",
        (
            "from importlib.metadata import distribution; "
            "u=distribution('transformers').read_text('direct_url.json') or ''; "
            "print(u); raise SystemExit(0 if 'transformers-openvla-oft' in u else 2)"
        ),
    ],
    check=False,
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
if transformers_probe.returncode != 0:
    run([
        OFT_PYTHON, "-m", "pip", "install", "--force-reinstall", "--no-deps",
        "transformers @ git+https://github.com/moojink/transformers-openvla-oft.git",
    ])

run([OFT_PYTHON, "-m", "pip", "install", "--no-deps", "-e", OFT_REPO])
environment_probe = run(
    [
        OFT_PYTHON,
        "-c",
        (
            "import torch, transformers; "
            "from importlib.metadata import distribution; "
            "print('torch', torch.__version__, 'cuda', torch.cuda.is_available()); "
            "print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'); "
            "print('transformers', distribution('transformers').read_text('direct_url.json'))"
        ),
    ],
    env=oft_env(),
)
assert "transformers-openvla-oft" in environment_probe.stdout
assert "cuda True" in environment_probe.stdout
OFT_ENV_VERIFIED = True

In [ ]:
# Cell J: Register D3, 8D joint proprioception, and Panda OFT constants.
assert globals().get("D3_RLDS_VERIFIED", False), "Run Cell H first."
assert globals().get("OFT_ENV_VERIFIED", False), "Run Cell I first."

registration = run(
    [OFT_PYTHON, OFT_SETUP_SCRIPT, "--repo", OFT_REPO],
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert "OPENVLA_OFT_D3_REGISTRATION_INSTALLED" in registration.stdout

constant_probe = run(
    [
        OFT_PYTHON,
        "-c",
        (
            "import sys; sys.argv.append('panda_d3'); "
            "from prismatic.vla.constants import ACTION_DIM, PROPRIO_DIM, NUM_ACTIONS_CHUNK; "
            "print(ACTION_DIM, PROPRIO_DIM, NUM_ACTIONS_CHUNK)"
        ),
    ],
    cwd=OFT_REPO,
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert constant_probe.stdout.strip().endswith("7 8 8")
OFT_D3_REGISTERED = True

In [ ]:
# Cell K: Exercise real D3 statistics, proprio normalization, and OFT chunking before model loading.
assert globals().get("OFT_D3_REGISTERED", False), "Run Cell J first."
verification = run(
    [
        OFT_PYTHON,
        OFT_VERIFY_SCRIPT,
        "--repo", OFT_REPO,
        "--data-root", DATA_ROOT_DIR,
        "--dataset-name", DATASET_NAME,
    ],
    env=oft_env({"CUDA_VISIBLE_DEVICES": ""}),
)
assert "OPENVLA_OFT_D3_VERIFIED" in verification.stdout
OFT_D3_VERIFIED = True
print("Cell K passed: nonzero 8D proprio -> one proprio token; action chunks are 8 x 7.")

## OFT training smoke test and full run

The default below runs three optimizer steps with no checkpoint save or merge. A full run uses the same command and changes only the adjustable training values.

In [ ]:
# Cell L: Launch OpenVLA-OFT with continuous L1 chunks and joint proprioception.
TRAINING_MODE = "full"  # Change to "full" only after the three-step smoke test passes.
SMOKE_MAX_STEPS = 3  # Adjustable smoke-test optimizer steps.
FULL_MAX_STEPS = 50_000  # Adjustable full-run optimizer steps; evaluate saved checkpoints.

BATCH_SIZE = 1  # Keep at 1 for a 32 GB RTX 5090; raise only after measuring VRAM.
GRAD_ACCUMULATION_STEPS = 8  # Adjustable effective batch multiplier.
LORA_RANK = 32  # Official OFT LoRA rank.
LEARNING_RATE = 5e-4  # Official OFT starting learning rate.
IMAGE_AUG = True  # Recommended; inference must use the matching center crop.
SHUFFLE_BUFFER_SIZE = 1000  # Raise if host RAM permits.
FULL_SAVE_FREQ = 10_000  # Adjustable full-run checkpoint interval.
FULL_VAL_FREQ = 10_000  # Adjustable held-out test-split evaluation interval.
MERGE_LORA_DURING_FULL_TRAINING = False # Requires roughly 30+ GB free disk for a merged 7B checkpoint.
MIN_FREE_DISK_GB_FOR_MERGE = 30  # Adjustable safety threshold.

assert globals().get("OFT_D3_VERIFIED", False), "Run Cells F-K before training."
assert TRAINING_MODE in {"smoke", "full"}

is_smoke = TRAINING_MODE == "smoke"
max_steps = SMOKE_MAX_STEPS if is_smoke else FULL_MAX_STEPS
save_freq = max_steps + 1000 if is_smoke else FULL_SAVE_FREQ
use_val_set = not is_smoke
merge_lora = False if is_smoke else MERGE_LORA_DURING_FULL_TRAINING
run_note = f"panda-d3-joint-proprio-oft-{'smoke' if is_smoke else 'full'}"

if merge_lora:
    free_gb = shutil.disk_usage(RUN_ROOT_DIR).free / 1024**3
    print("Free disk space:", round(free_gb, 1), "GB")
    assert free_gb >= MIN_FREE_DISK_GB_FOR_MERGE, (
        f"Free at least {MIN_FREE_DISK_GB_FOR_MERGE} GB before merged checkpoint saves; "
        "or set MERGE_LORA_DURING_FULL_TRAINING=False and merge later."
    )

torchrun = OFT_BIN_DIR / "torchrun"
torchrun = torchrun if torchrun.exists() else Path("torchrun")
cmd = [
    torchrun,
    "--standalone", "--nnodes", "1", "--nproc-per-node", "1",
    OFT_REPO / "vla-scripts/finetune.py",
    "--vla_path", "openvla/openvla-7b",
    "--data_root_dir", DATA_ROOT_DIR,
    "--dataset_name", DATASET_NAME,
    "--run_root_dir", RUN_ROOT_DIR,
    "--use_l1_regression", "True",
    "--use_diffusion", "False",
    "--use_film", "False",
    "--num_images_in_input", "1",
    "--use_proprio", "True",
    "--batch_size", str(BATCH_SIZE),
    "--grad_accumulation_steps", str(GRAD_ACCUMULATION_STEPS),
    "--lora_rank", str(LORA_RANK),
    "--learning_rate", str(LEARNING_RATE),
    "--image_aug", str(IMAGE_AUG),
    "--shuffle_buffer_size", str(SHUFFLE_BUFFER_SIZE),
    "--max_steps", str(max_steps),
    "--save_freq", str(save_freq),
    "--use_val_set", str(use_val_set),
    "--val_freq", str(FULL_VAL_FREQ),
    "--merge_lora_during_training", str(merge_lora),
    "--wandb_project", "openvla_oft_panda_d3",
    "--wandb_entity", "family-of-note-politecnico-di-milano",
    "--run_id_note", run_note,
]

print("Training mode:", TRAINING_MODE)
print("OFT input: third-person RGB + language + 8D joint/gripper proprio")
print("OFT output: 8 future actions x 7 continuous dimensions")
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUMULATION_STEPS)
if is_smoke:
    print("Smoke mode: no checkpoint save and no LoRA merge.")

if not RUN_OFT_TRAINING:
    print("RUN_OFT_TRAINING=False; command is configured but was not launched.")
else:
    train_env = oft_env({"WANDB_MODE": "disabled" if is_smoke else os.environ.get("WANDB_MODE", "online")})
    start = time.time()
    process = subprocess.Popen(
        [str(part) for part in cmd],
        cwd=str(OFT_REPO),
        env=train_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )

    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        print(chunk.decode("utf-8", errors="replace"), end="", flush=True)

    return_code = process.wait()
    assert return_code == 0
    print("Elapsed minutes:", round((time.time() - start) / 60, 2))
    OFT_TRAINING_FINISHED = True

In [ ]:
# Cell L2: Three training curves for only the run that produced the 50,000-step checkpoint.
from pathlib import Path
import json

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import pandas as pd
from wandb.proto import wandb_internal_pb2
from wandb.sdk.internal import datastore

L3_RUN_ID = "thfwkiyw"  # The single full run that ended at checkpoint step 50,000.
L3_FINAL_STEP = 50_000
L3_SMOOTH_ROWS = 101  # Logs are every 10 steps: trailing mean over about 1,000 steps.

assert "OFT_REPO" in globals(), "Run Cell F first."
l3_run_matches = list((Path(OFT_REPO) / "wandb").glob(f"run-*-{L3_RUN_ID}"))
assert len(l3_run_matches) == 1, f"Could not uniquely locate local W&B run {L3_RUN_ID}."
l3_run_dir = l3_run_matches[0]
l3_wandb_files = list(l3_run_dir.glob(f"run-{L3_RUN_ID}.wandb"))
assert len(l3_wandb_files) == 1, f"Missing local history in {l3_run_dir}."

l3_reader = datastore.DataStore()
l3_reader.open_for_scan(str(l3_wandb_files[0]))
l3_rows = []
while True:
    l3_record_bytes = l3_reader.scan_data()
    if l3_record_bytes is None:
        break
    l3_record = wandb_internal_pb2.Record()
    l3_record.ParseFromString(l3_record_bytes)
    if l3_record.WhichOneof("record_type") != "history":
        continue
    l3_row = {}
    for item in l3_record.history.item:
        key = item.key or "/".join(item.nested_key)
        if key:
            l3_row[key] = json.loads(item.value_json)
    l3_rows.append(l3_row)

l3_metric_panels = [
    ("train_loss", "VLA Train/Loss", "overall 8 × 7 continuous-action L1"),
    ("current_action_l1_loss", "VLA Train/Curr Action L1 Loss", "current action: horizon 0"),
    ("next_actions_l1_loss", "VLA Train/Next Actions L1 Loss", "next actions: horizons 1–7"),
]
l3_columns = ["_step", *[metric for _, metric, _ in l3_metric_panels]]
l3_history = pd.DataFrame(l3_rows)
l3_missing = [column for column in l3_columns if column not in l3_history]
assert not l3_missing, "Missing saved metrics: " + ", ".join(l3_missing)
for column in l3_columns:
    l3_history[column] = pd.to_numeric(l3_history[column], errors="coerce")
l3_history = (
    l3_history.dropna(subset=l3_columns)[l3_columns]
    .sort_values("_step")
    .drop_duplicates("_step", keep="last")
)
assert int(l3_history["_step"].min()) == 0
assert int(l3_history["_step"].max()) == L3_FINAL_STEP, (
    f"Run {L3_RUN_ID} ends at {int(l3_history['_step'].max()):,}, not {L3_FINAL_STEP:,}."
)

raw_color = "#F06AD7"
smooth_color = "#C026B8"
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
for ax, (title, metric, description) in zip(axes, l3_metric_panels):
    raw_values = l3_history[metric]
    smooth_values = raw_values.rolling(L3_SMOOTH_ROWS, min_periods=1).mean()
    ax.plot(
        l3_history["_step"], raw_values,
        color=raw_color, alpha=0.28, linewidth=0.7, label="raw logged value",
    )
    ax.plot(
        l3_history["_step"], smooth_values,
        color=smooth_color, linewidth=2.0, label="~1,000-step trailing mean",
    )
    final_mean = float(raw_values.tail(L3_SMOOTH_ROWS).mean())
    ax.scatter([L3_FINAL_STEP], [final_mean], color=smooth_color, s=30, zorder=3)
    ax.annotate(
        f"final ~1k mean: {final_mean:.4f}",
        (L3_FINAL_STEP, final_mean),
        xytext=(-8, 10), textcoords="offset points", ha="right", fontsize=9,
    )
    ax.set(
        title=f"{title}\n{description}",
        xlabel="Step",
        ylabel="L1 mean absolute error\n(normalized training coordinates)",
        xlim=(0, L3_FINAL_STEP),
        ylim=(0, None),
    )
    ax.set_xticks([0, 10_000, 20_000, 30_000, 40_000, 50_000])
    ax.xaxis.set_major_formatter(
        FuncFormatter(lambda value, _: "0" if value == 0 else f"{int(value / 1000)}k")
    )
    ax.grid(axis="y", alpha=0.25)
    ax.legend(loc="upper right", fontsize=8)

fig.suptitle(
    "Single continuous-OFT training run ending at checkpoint 50,000\n"
    f"W&B run {L3_RUN_ID}; no intermediate-checkpoint series",
    fontsize=15,
)
plt.show()

print("Plotted only run:", l3_run_dir.name)
print(f"History rows: {len(l3_history):,}; range: 0–{L3_FINAL_STEP:,} optimizer steps.")
print(
    "No action_accuracy curve exists for this run: it predicts continuous actions with L1 regression, "
    "whereas the older action_accuracy plot measured exact discrete action-token matches."
)

In [ ]:
# Cell M: Locate and validate the newest saved full OFT checkpoint.
# This cell reports no checkpoint after the default smoke run, which intentionally saves nothing.
checkpoint_candidates = sorted(
    [
        path
        for path in RUN_ROOT_DIR.glob("**/*")
        if path.is_dir()
        and (path / "dataset_statistics.json").is_file()
        and any(path.glob("action_head--*checkpoint.pt"))
        and any(path.glob("proprio_projector--*checkpoint.pt"))
    ],
    key=lambda path: path.stat().st_mtime,
)
if not checkpoint_candidates:
    print("No saved checkpoint yet. This is expected after TRAINING_MODE='smoke'.")
    LATEST_OFT_CHECKPOINT = None
else:
    LATEST_OFT_CHECKPOINT = checkpoint_candidates[-1]
    required_patterns = ["action_head--*checkpoint.pt", "proprio_projector--*checkpoint.pt"]
    for pattern in required_patterns:
        matches = list(LATEST_OFT_CHECKPOINT.glob(pattern))
        assert len(matches) == 1, (pattern, matches)
    stats = json.loads((LATEST_OFT_CHECKPOINT / "dataset_statistics.json").read_text())
    proprio_stats = stats[DATASET_NAME]["proprio"]
    assert len(proprio_stats["q01"]) == 8 and len(proprio_stats["q99"]) == 8
    print("Latest OFT checkpoint:", LATEST_OFT_CHECKPOINT)
    print("8D proprio statistics confirmed.")
    if not (LATEST_OFT_CHECKPOINT / "model.safetensors.index.json").exists():
        print("Checkpoint is unmerged; merge LoRA before using the standard OFT inference loader.")

In [ ]:
# Merge the saved LoRA adapter into the OpenVLA-7B base checkpoint.
import json
import shutil
import subprocess
from pathlib import Path

assert LATEST_OFT_CHECKPOINT is not None, "Run Cell M first."
LATEST_OFT_CHECKPOINT = Path(LATEST_OFT_CHECKPOINT)

merge_script = OFT_REPO / "vla-scripts/merge_lora_weights_and_save.py"
merged_index = LATEST_OFT_CHECKPOINT / "model.safetensors.index.json"

assert merge_script.is_file(), f"Missing merge script: {merge_script}"
assert (LATEST_OFT_CHECKPOINT / "lora_adapter/adapter_model.safetensors").is_file()

free_gib = shutil.disk_usage(LATEST_OFT_CHECKPOINT).free / (1024**3)
print(f"Available disk space: {free_gib:.1f} GiB")
assert free_gib >= 18, "Insufficient disk space for the merged OpenVLA-7B checkpoint."

if merged_index.is_file():
    print("Checkpoint is already merged:", LATEST_OFT_CHECKPOINT)
else:
    merge_command = [
        str(OFT_PYTHON),
        str(merge_script),
        "--base_checkpoint",
        "openvla/openvla-7b",
        "--lora_finetuned_checkpoint_dir",
        str(LATEST_OFT_CHECKPOINT),
    ]

    print("Starting LoRA merge. This may take several minutes...")
    merge_process = subprocess.Popen(
        merge_command,
        cwd=str(OFT_REPO),
        env=oft_env(),
    )
    merge_return_code = merge_process.wait()
    assert merge_return_code == 0, f"LoRA merge failed with exit code {merge_return_code}"

# Validate every merged model shard.
assert merged_index.is_file(), "Merge completed without creating the model index."
index_data = json.loads(merged_index.read_text())
model_shards = sorted(set(index_data["weight_map"].values()))

assert model_shards, "No model shards listed in the merged index."
for shard_name in model_shards:
    shard_path = LATEST_OFT_CHECKPOINT / shard_name
    assert shard_path.is_file() and shard_path.stat().st_size > 0, (
        f"Missing or empty merged shard: {shard_path}"
    )

assert (LATEST_OFT_CHECKPOINT / "config.json").is_file()
assert any(LATEST_OFT_CHECKPOINT.glob("action_head--*checkpoint.pt"))
assert any(LATEST_OFT_CHECKPOINT.glob("proprio_projector--*checkpoint.pt"))

print(f"LoRA merge verified: {len(model_shards)} model shard(s)")
print("Merged checkpoint ready for Cell N:", LATEST_OFT_CHECKPOINT)

# Testing the fine-tuned model

These tests use the merged 50,000-step OpenVLA-OFT checkpoint, its continuous action head, and its proprio projector.

## 1. In-distribution test

Teacher-forced evaluation on untouched D3 test demonstrations. Each image is paired with its measured `[q1..q7, gripper_open_fraction]`, and every predicted `8 x 7` chunk is compared with the corresponding eight future expert actions.

In [ ]:
# Cell O: Teacher-forced OFT evaluation on held-out D3 demonstrations.
from datetime import datetime
from pathlib import Path
import json
import subprocess

from IPython.display import Image as NotebookImage
from IPython.display import Video, display

# Evaluate one nominal and one boundary demonstration by default.
TEST_EPISODE_INDICES = [10]
MAX_STEPS_PER_EPISODE = None  # Set a small positive integer for a quick smoke test.
TEST_VIDEO_FPS = 10
DISPLAY_ID_VIDEO_INLINE = True

assert LATEST_OFT_CHECKPOINT is not None, "Run Cell M first."
O_CHECKPOINT_DIR = Path(LATEST_OFT_CHECKPOINT)
O_EVALUATOR = PROJECT_DIR / "openvla_oft_heldout_visual_eval.py"

required_globals = ["OFT_REPO", "DATA_ROOT_DIR", "DATASET_NAME", "OFT_PYTHON", "oft_env"]
missing_globals = [name for name in required_globals if name not in globals()]
assert not missing_globals, "Run Cell F first. Missing: " + ", ".join(missing_globals)

required_checkpoint_files = [
    "config.json",
    "dataset_statistics.json",
    "model.safetensors.index.json",
    "preprocessor_config.json",
    "tokenizer_config.json",
]
missing_files = [
    name for name in required_checkpoint_files if not (O_CHECKPOINT_DIR / name).is_file()
]
assert not missing_files, "The merged checkpoint is incomplete. Missing: " + ", ".join(missing_files)
assert any(O_CHECKPOINT_DIR.glob("action_head--*checkpoint.pt"))
assert any(O_CHECKPOINT_DIR.glob("proprio_projector--*checkpoint.pt"))
assert O_EVALUATOR.is_file(), f"Evaluator script is missing: {O_EVALUATOR}"
assert TEST_EPISODE_INDICES, "Choose at least one D3 test episode."

O_OUTPUT_DIR = (
    PROJECT_DIR
    / "test_eval_visuals"
    / "oft-d3-in-distribution"
    / datetime.now().strftime("%Y%m%d_%H%M%S")
)
O_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

command = [
    str(OFT_PYTHON),
    str(O_EVALUATOR),
    "--checkpoint", str(O_CHECKPOINT_DIR),
    "--data-root", str(DATA_ROOT_DIR),
    "--dataset-name", DATASET_NAME,
    "--openvla-repo", str(OFT_REPO),
    "--output-dir", str(O_OUTPUT_DIR),
    "--episode-indices", *[str(index) for index in TEST_EPISODE_INDICES],
    "--fps", str(TEST_VIDEO_FPS),
]
if MAX_STEPS_PER_EPISODE is not None:
    command.extend(["--max-steps", str(MAX_STEPS_PER_EPISODE)])

evaluation_env = oft_env({
    "TF_CPP_MIN_LOG_LEVEL": "3",
    "MPLBACKEND": "Agg",
    "MPLCONFIGDIR": "/tmp/openvla_oft_id_matplotlib",
    "WANDB_MODE": "disabled",
})

print("Held-out split: D3 test")
print("Input: third-person RGB + language + measured 8D joint/gripper proprio")
print("Output: 8 future actions x 7 continuous dimensions")
print("Checkpoint:", O_CHECKPOINT_DIR)
print("Output directory:", O_OUTPUT_DIR)
print("Launching OFT evaluator...\n")

process = subprocess.Popen(
    command,
    cwd=str(OFT_REPO),
    env=evaluation_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")

return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Held-out OFT visual evaluation failed with exit code {return_code}.")

summary_path = O_OUTPUT_DIR / "summary.json"
summary = json.loads(summary_path.read_text(encoding="utf-8"))
for episode in summary["episodes"]:
    print("\n" + "=" * 78)
    print(
        f"Held-out episode {episode['episode_index']:03d} | "
        f"{episode['evaluated_queries']} valid 8-step chunks | "
        f"phase boundaries {episode['phase_boundaries']}"
    )
    print("Solid lines = test demonstration; dashed lines = OFT horizon-0 prediction.")
    display(NotebookImage(filename=episode["commands_plot"]))
    display(NotebookImage(filename=episode["chunk_plot"]))
    display(NotebookImage(filename=episode["proprio_plot"]))
    display(NotebookImage(filename=episode["contact_sheet"]))
    if DISPLAY_ID_VIDEO_INLINE:
        display(Video(episode["video"], embed=True, html_attributes="controls loop"))
    else:
        print("Video:", episode["video"])

print("\nSaved summary:", summary_path)
print("OPENVLA_OFT_IN_DISTRIBUTION_TEST_COMPLETE")

## 2. Out-of-distribution test

Closed-loop MuJoCo evaluation with cube starts outside the D3 training envelope. The policy is queried only from live third-person RGB, language, and measured 8D proprio; the full eight-action chunk is executed before the next query.

In [ ]:
# Cell P0: Quaternion helpers and 6D pose IK for closed-loop execution.
import math
import numpy as np


def quat_normalize(quat):
    quat = np.asarray(quat, dtype=np.float64)
    norm = float(np.linalg.norm(quat))
    if norm < 1e-12:
        raise ValueError("Cannot normalize a zero quaternion")
    return quat / norm


def quat_conjugate(quat):
    w, x, y, z = quat_normalize(quat)
    return np.array([w, -x, -y, -z], dtype=np.float64)


def quat_multiply(left, right):
    w1, x1, y1, z1 = quat_normalize(left)
    w2, x2, y2, z2 = quat_normalize(right)
    return quat_normalize(np.array([
        w1 * w2 - x1 * x2 - y1 * y2 - z1 * z2,
        w1 * x2 + x1 * w2 + y1 * z2 - z1 * y2,
        w1 * y2 - x1 * z2 + y1 * w2 + z1 * x2,
        w1 * z2 + x1 * y2 - y1 * x2 + z1 * w2,
    ], dtype=np.float64))


def quat_from_rotvec(rotvec):
    rotvec = np.asarray(rotvec, dtype=np.float64)
    angle = float(np.linalg.norm(rotvec))
    if angle < 1e-12:
        return np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float64)
    axis = rotvec / angle
    return np.array([
        math.cos(angle / 2.0),
        *(axis * math.sin(angle / 2.0)),
    ], dtype=np.float64)


def quat_apply_world_delta(quat, rotvec_world):
    return quat_multiply(quat_from_rotvec(rotvec_world), quat)


def quat_error_world(target_quat, current_quat):
    error_quat = quat_multiply(target_quat, quat_conjugate(current_quat))
    if error_quat[0] < 0.0:
        error_quat = -error_quat
    vector_norm = float(np.linalg.norm(error_quat[1:]))
    if vector_norm < 1e-12:
        return np.zeros(3, dtype=np.float64)
    angle = 2.0 * math.atan2(vector_norm, float(error_quat[0]))
    return error_quat[1:] / vector_norm * angle


def solve_hand_pose_ik(
    target_pos,
    target_quat,
    seed_qpos=None,
    max_iter=450,
    position_tolerance=0.00005,
    orientation_tolerance=np.deg2rad(0.01),
):
    """Solve one 6D hand-pose target with damped least squares."""
    target_pos = np.asarray(target_pos, dtype=np.float64)
    target_quat = quat_normalize(target_quat)
    if seed_qpos is not None:
        for name, value in seed_qpos.items():
            joint_id = require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name)
            data.qpos[model.jnt_qposadr[joint_id]] = float(value)
    mujoco.mj_forward(model, data)

    jacp = np.zeros((3, model.nv), dtype=np.float64)
    jacr = np.zeros((3, model.nv), dtype=np.float64)
    iterations = 0
    for iterations in range(1, max_iter + 1):
        current = get_ee_pose()
        position_error = target_pos - current["position"]
        rotation_error = quat_error_world(target_quat, current["quat_wxyz"])
        # Always perform at least one DLS update for a nonzero policy command.
        if (
            iterations > 1
            and np.linalg.norm(position_error) < position_tolerance
            and np.linalg.norm(rotation_error) < orientation_tolerance
        ):
            break

        mujoco.mj_jacBody(model, data, jacp, jacr, HAND_BODY_ID)
        jacobian = np.vstack([
            jacp[:, ARM_DOF_ADR],
            jacr[:, ARM_DOF_ADR],
        ])
        error = np.concatenate([position_error, rotation_error])
        damping = 0.055
        delta_q = jacobian.T @ np.linalg.solve(
            jacobian @ jacobian.T + damping * damping * np.eye(6),
            error,
        )
        delta_q = np.clip(delta_q, -0.035, 0.035)

        for index, address in enumerate(ARM_QPOS_ADR):
            low, high = model.jnt_range[ARM_JOINT_IDS[index]]
            data.qpos[address] = np.clip(
                data.qpos[address] + delta_q[index],
                low,
                high,
            )
        mujoco.mj_forward(model, data)

    final_pose = get_ee_pose()
    diagnostics = {
        "iterations": int(iterations),
        "position_error_m": float(
            np.linalg.norm(target_pos - final_pose["position"])
        ),
        "orientation_error_rad": float(np.linalg.norm(
            quat_error_world(target_quat, final_pose["quat_wxyz"])
        )),
    }
    diagnostics["converged"] = bool(
        diagnostics["position_error_m"] < position_tolerance
        and diagnostics["orientation_error_rad"] < orientation_tolerance
    )
    return current_arm_qpos_dict(), diagnostics


def plan_pose_qpos(target_pos, target_quat):
    """Plan in temporary MuJoCo state, then restore the live rollout state."""
    saved_qpos = data.qpos.copy()
    saved_qvel = data.qvel.copy()
    saved_ctrl = data.ctrl.copy()
    saved_time = float(data.time)
    try:
        return solve_hand_pose_ik(
            target_pos,
            target_quat,
            seed_qpos=current_arm_qpos_dict(),
        )
    finally:
        data.qpos[:] = saved_qpos
        data.qvel[:] = saved_qvel
        data.ctrl[:] = saved_ctrl
        data.time = saved_time
        mujoco.mj_forward(model, data)


In [ ]:
# Cell P: Out-of-distribution closed-loop MuJoCo rollout with OpenVLA-OFT.
# Every policy query uses live third-person RGB, language, and measured 8D proprio.
# Only horizon 0 is executed before replanning at 10 Hz; horizons 1-7 are logged
# for diagnostics and never substituted into the physical controller.

from pathlib import Path
from datetime import datetime
import hashlib
import io
import json
import os
import select
import subprocess
import sys
import time
import xml.etree.ElementTree as ET

import imageio.v2 as imageio
import numpy as np
from PIL import Image, ImageDraw
from scipy.ndimage import gaussian_filter
from IPython.display import Video, display

# ----- User-facing rollout knobs -----
TASK_PROMPT = "pick up the red block from the table and place it on the far side of the table"
OPENVLA_ROLLOUT_PROMPT = f"In: What action should the robot take to {TASK_PROMPT}?\nOut: "   # \n means start a new line.

N_ROLLOUTS = 4
MAX_EXECUTED_ACTIONS = 200  # 20 simulated seconds at the D3 10 Hz control rate.
ACTIONS_PER_CHUNK = 1  # Re-observe and execute only the newly predicted horizon-0 action.
CONTROL_HORIZON_S = 0.1
VIDEO_FPS = round(1.0 / CONTROL_HORIZON_S)
ROLLOUT_SEED = 7
DISPLAY_ROLLOUT_VIDEOS = True

# D3 trained on cube x=[0.47, 0.53], y=[-0.04, 0.04]. These starts are outside it.
OOD_CUBE_STARTS = np.array([
    [0.455, -0.060, 0.085],
    [0.545, 0.060, 0.085],
    [0.455, 0.060, 0.085],
    [0.545, -0.060, 0.085],
], dtype=np.float64)

# Which OOD_CUBE_STARTS corner(s) to actually run, in order. Leave as None to
# run every corner once, indices 0..N_ROLLOUTS-1 (the original behavior).
# Set to an explicit list of indices into OOD_CUBE_STARTS to run only (and
# exactly) those corners, e.g. [2] for just the third corner, or [0, 2, 0] to
# repeat corner 0 with corner 2 in between. Output folders are still named
# rollout_000, rollout_001, ... in the order run, regardless of which corner
# each one used; N_ROLLOUTS is ignored whenever this is set.
SELECTED_OOD_START_INDICES = None  # e.g. [2]
OOD_EE_JITTER_M = 0.0
WRIST_IMAGE_SIZE = 256

# GelSight Mini acquisition is independent of policy inference. MuJoCo's native
# tactile sensor supplies geometry-derived indentation and tangential speeds;
# A matched passive MuJoCo solve supplies the synchronized contact wrench at
# the post-step live state without refreshing or changing the rollout data.
GELSIGHT_TACTILE_FPS = 30
GELSIGHT_TAXELS_X = 48
GELSIGHT_TAXELS_Z = 64
GELSIGHT_RGB_WIDTH = 320
GELSIGHT_RGB_HEIGHT = 240
GELSIGHT_GEL_THICKNESS_M = 0.0045
GELSIGHT_DEPTH_ACTIVE_EPS_M = 1.0e-7
GELSIGHT_DEPTH_DISPLAY_MAX_M = 0.0008
GELSIGHT_DEPTH_DISPLAY_GAMMA = 0.45  # <1 boosts contrast for shallow native depths.
GELSIGHT_FORCE_DISPLAY_MAX_N = 5.0
DISPLAY_GELSIGHT_VIDEOS = True

# Fine, physical verification texture for the 70 mm cube. Each sphere center
# lies on a box face, so exactly one hemisphere protrudes. These are ordinary
# MuJoCo collision geoms in the live rollout model (not pixels added to the
# tactile video). The same 7 x 7 lattice is applied to every one of six faces.
CUBE_TEXTURE_GRID_SIZE = 7
CUBE_TEXTURE_GRID_HALF_SPAN_M = 0.027
CUBE_TEXTURE_BUMP_RADIUS_M = 0.00150
CUBE_TEXTURE_GEOM_PREFIX = "red_block_tactile_bump_"

# The policy predicts D3 Cartesian deltas, a world-frame rotation vector, and a
# continuous gripper open fraction. These gains compensate the physical actuator
# response measured against exact D3 expert-action replay.
OPENVLA_MAX_TRANSLATION_M = 0.040
OPENVLA_MAX_ROTATION_RAD = np.deg2rad(20.0)
OPENVLA_GRIPPER_OPENING_M = 0.04
OPENVLA_GRIPPER_CLOSED_M = 0.00
WORKSPACE_LOW = np.array([0.24, -0.25, 0.180], dtype=np.float64)
WORKSPACE_HIGH = np.array([0.86, 0.25, 0.62], dtype=np.float64)
APPROACH_TRANSLATION_GAIN = np.array([0.9, 1.05, 1.0], dtype=np.float64)
CARRY_TRANSLATION_GAIN = np.array([2.5, 1.05, 1.0], dtype=np.float64)
CARRY_MAX_TRANSLATION_M = 0.030
ROTATION_GAIN = 0.4
ACTUATOR_LEAD = 2.0
FINE_APPROACH_START_Z_M = 0.0
FINE_APPROACH_MAX_XY_STEP_M = 0.0025
FINE_APPROACH_MAX_DESCENT_STEP_M = 0.005
CARRY_FILTER_ALPHA = 0.5
CARRY_ROTATION_FILTER_ALPHA = 0.35
CARRY_AXIS_MAX_STEP_M = np.array([0.02, 0.004, 0.008], dtype=np.float64)
PLACEMENT_AXIS_MAX_STEP_M = np.array([0.01, 0.003, 0.004], dtype=np.float64)
WRIST_VISUAL_SERVO_ENABLED = True
WRIST_VISUAL_SERVO_START_Z_M = 0.3
WRIST_VISUAL_SERVO_STOP_Z_M = 0.0
WRIST_VISUAL_SERVO_GAIN = 0.35
WRIST_VISUAL_SERVO_MAX_STEP_M = 0.004
WRIST_ALIGNMENT_PLANE_Z_M = 0.120
WRIST_ESTIMATE_TO_GRASP_OFFSET_M = np.array([-0.009, 0.001], dtype=np.float64)
WRIST_ALIGNMENT_TOLERANCE_M = 0.003
SAFE_CARRY_Y_BOUNDS_M = np.array([-0.08, 0.08], dtype=np.float64)
MAX_VALID_GRASP_CLOSE_Z_M = 0.205

# Track grasp/release phases for evaluation and termination while applying the
# policy's continuous gripper fraction directly, as in the D3 demonstrations.
GRIPPER_CLOSE_TRIGGER = 0.50
GRIPPER_OPEN_TRIGGER = 0.65
GRIPPER_CLOSE_PROFILE_STEPS = 8
GRIPPER_OPEN_PROFILE_STEPS = 7
LIFT_LOOKAHEAD_MIN_DZ_M = 0.005
LIFT_BOOTSTRAP_HEIGHT_M = 0.12
CARRY_Z_FLOOR_OFFSET_M = 0.12
CARRY_DESCENT_START_X_M = 0.65
RELEASE_CLEARANCE_M = 0.03
RELEASE_SETTLE_S = 0.5

#   ----------------- Internal rollout parameters -----------------
POLICY_CAMERA_VIEW = "third_person"

required_globals = [
    "model", "data", "mujoco", "AGENT_CAMERA_NAME", "START_BLOCK_POS", "PLACE_BLOCK_POS",
    "RUN_ROOT_DIR", "OFT_REPO", "OFT_PYTHON", "DATASET_NAME", "oft_env", "LATEST_OFT_CHECKPOINT",
    "reset_task_state_randomized", "current_arm_qpos_dict", "set_arm_position_targets",
    "set_gripper_opening", "current_finger_opening_m", "solve_hand_position_ik", "get_ee_pose", "get_cube_pose",
    "HAND_BODY_ID", "require_mj_id", "ARM_JOINT_NAMES", "ARM_JOINT_IDS",
    "ARM_QPOS_ADR", "ARM_DOF_ADR", "FINGER_QPOS_ADR", "make_third_person_camera",
    "THIRD_PERSON_IMAGE_SIZE", "quat_apply_world_delta", "quat_error_world",
    "quat_normalize", "solve_hand_pose_ik", "plan_pose_qpos",
    "PANDA_DIR", "PANDA_XML", "GELSIGHT_BODY_NAME", "GELSIGHT_SITE_NAME",
    "GELSIGHT_GEL_PAD_GEOM_NAME", "FINGER_JOINT_NAMES",
]
missing = [name for name in required_globals if name not in globals()]
assert not missing, "Run Cells 2, 3, 4, A, B, C, F, M, and the LoRA merge cell first. Missing: " + ", ".join(missing)

POLICY_IMAGE_WIDTH = THIRD_PERSON_IMAGE_SIZE[0]
POLICY_IMAGE_HEIGHT = THIRD_PERSON_IMAGE_SIZE[1]

# Finding complete model directories and choosing the one with the newest directory modification time.
def latest_complete_checkpoint(run_root):
    required = [
        "config.json",
        "dataset_statistics.json",
        "model.safetensors.index.json",
        "preprocessor_config.json",
        "tokenizer_config.json",
    ]
    candidates = []
    for p in Path(run_root).iterdir():
        if p.is_dir() and all((p / f).exists() for f in required):
            candidates.append(p)
    assert candidates, f"No complete merged OpenVLA checkpoint found in {run_root}"
    return max(candidates, key=lambda p: p.stat().st_mtime)


################### Change this part for each run

# checkpoint_dir = latest_complete_checkpoint(RUN_ROOT_DIR)   # Use the latest merged checkpoint in RUN_ROOT_DIR.
#print("Using checkpoint:", checkpoint_dir)
#print("Prompt:", repr(OPENVLA_ROLLOUT_PROMPT))

assert LATEST_OFT_CHECKPOINT is not None, "Run Cell M first."
checkpoint_dir = Path(LATEST_OFT_CHECKPOINT)
required_checkpoint_files = [
    "config.json", "dataset_statistics.json", "model.safetensors.index.json",
    "preprocessor_config.json", "tokenizer_config.json",
]
missing_checkpoint_files = [
    name for name in required_checkpoint_files if not (checkpoint_dir / name).is_file()
]
assert not missing_checkpoint_files, "Incomplete merged checkpoint: " + ", ".join(missing_checkpoint_files)
assert any(checkpoint_dir.glob("action_head--*checkpoint.pt"))
assert any(checkpoint_dir.glob("proprio_projector--*checkpoint.pt"))
assert ACTIONS_PER_CHUNK == 1, "The validated controller replans after every horizon-0 action."

print("Using merged OpenVLA-OFT checkpoint:")
print(checkpoint_dir)
print("Policy input: third-person RGB + language + live 8D proprio")
print("Policy replanning: execute horizon 0, then acquire a new image and proprio state")
print("Action selection: only horizon 0 is controlled; horizons 1-7 are diagnostic")

#########################

run_dir = Path(
    PROJECT_DIR if "PROJECT_DIR" in globals() else Path.cwd()
) / f"openvla_oft_ood_closed_loop_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_dir.mkdir(parents=True, exist_ok=True)


# -----------------------------------------------------------------------------
# Physical cube surface texture (live rollout model)
# -----------------------------------------------------------------------------
# The core box remains the 50 g dynamic body used by the policy. Massless
# collision spheres are half embedded in it, producing genuine solver contacts
# and native tactile indentation while leaving mass, inertia, joints, cameras,
# actuators, controller code, and policy inputs unchanged.

_CUBE_TEXTURE_FACE_SPECS = (
    ("px", 0, +1, 1, 2),
    ("nx", 0, -1, 1, 2),
    ("py", 1, +1, 0, 2),
    ("ny", 1, -1, 0, 2),
    ("pz", 2, +1, 0, 1),
    ("nz", 2, -1, 0, 1),
)


def _texture_number_string(values):
    return " ".join(f"{float(value):.17g}" for value in np.asarray(values).reshape(-1))


def _texture_file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _texture_joint_layout(model_):
    return [
        (
            mujoco.mj_id2name(model_, mujoco.mjtObj.mjOBJ_JOINT, joint_id),
            int(model_.jnt_type[joint_id]),
            int(model_.jnt_qposadr[joint_id]),
            int(model_.jnt_dofadr[joint_id]),
        )
        for joint_id in range(model_.njnt)
    ]


def _texture_actuator_layout(model_):
    return [
        (
            mujoco.mj_id2name(model_, mujoco.mjtObj.mjOBJ_ACTUATOR, actuator_id),
            int(model_.actuator_trntype[actuator_id]),
            tuple(int(value) for value in model_.actuator_trnid[actuator_id]),
        )
        for actuator_id in range(model_.nu)
    ]


def _copy_rollout_state(source_data, destination_model, destination_data):
    destination_data.time = float(source_data.time)
    destination_data.qpos[:] = source_data.qpos
    destination_data.qvel[:] = source_data.qvel
    if destination_model.na:
        destination_data.act[:] = source_data.act
    if destination_model.nu:
        destination_data.ctrl[:] = source_data.ctrl
    destination_data.qacc_warmstart[:] = source_data.qacc_warmstart
    destination_data.qfrc_applied[:] = source_data.qfrc_applied
    destination_data.xfrc_applied[:] = source_data.xfrc_applied
    if destination_model.nmocap:
        destination_data.mocap_pos[:] = source_data.mocap_pos
        destination_data.mocap_quat[:] = source_data.mocap_quat
    if destination_model.nuserdata:
        destination_data.userdata[:] = source_data.userdata
    if destination_model.neq:
        destination_data.eq_active[:] = source_data.eq_active
    if hasattr(destination_data, "plugin_state") and destination_data.plugin_state.size:
        assert destination_data.plugin_state.shape == source_data.plugin_state.shape
        destination_data.plugin_state[:] = source_data.plugin_state
    mujoco.mj_forward(destination_model, destination_data)


def _refresh_scene_bindings_after_texture():
    global body_ids, site_ids, camera_id
    global HAND_BODY_ID, LEFT_FINGER_BODY_ID, RIGHT_FINGER_BODY_ID
    global GELSIGHT_BODY_ID, GELSIGHT_SITE_ID
    global BLOCK_BODY_ID, BLOCK_JOINT_ID, BLOCK_QPOS_ADR
    global ARM_JOINT_IDS, ARM_QPOS_ADR, ARM_DOF_ADR
    global FINGER_JOINT_IDS, FINGER_QPOS_ADR

    body_ids = {
        "hand": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "hand"),
        "left_finger": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "left_finger"),
        "right_finger": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "right_finger"),
        "red_block": require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "red_block"),
        "agent_camera_mount": require_mj_id(
            mujoco.mjtObj.mjOBJ_BODY, "agent_camera_mount"
        ),
        GELSIGHT_BODY_NAME: require_mj_id(
            mujoco.mjtObj.mjOBJ_BODY, GELSIGHT_BODY_NAME
        ),
    }
    site_ids = {
        GELSIGHT_SITE_NAME: require_mj_id(
            mujoco.mjtObj.mjOBJ_SITE, GELSIGHT_SITE_NAME
        )
    }
    camera_id = require_mj_id(mujoco.mjtObj.mjOBJ_CAMERA, AGENT_CAMERA_NAME)

    HAND_BODY_ID = body_ids["hand"]
    LEFT_FINGER_BODY_ID = body_ids["left_finger"]
    RIGHT_FINGER_BODY_ID = body_ids["right_finger"]
    GELSIGHT_BODY_ID = body_ids[GELSIGHT_BODY_NAME]
    GELSIGHT_SITE_ID = site_ids[GELSIGHT_SITE_NAME]
    BLOCK_BODY_ID = body_ids["red_block"]
    BLOCK_JOINT_ID = require_mj_id(
        mujoco.mjtObj.mjOBJ_JOINT, "red_block_freejoint"
    )
    BLOCK_QPOS_ADR = int(model.jnt_qposadr[BLOCK_JOINT_ID])

    ARM_JOINT_IDS = [
        require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name)
        for name in ARM_JOINT_NAMES
    ]
    ARM_QPOS_ADR = [int(model.jnt_qposadr[joint_id]) for joint_id in ARM_JOINT_IDS]
    ARM_DOF_ADR = [int(model.jnt_dofadr[joint_id]) for joint_id in ARM_JOINT_IDS]
    FINGER_JOINT_IDS = [
        require_mj_id(mujoco.mjtObj.mjOBJ_JOINT, name)
        for name in FINGER_JOINT_NAMES
    ]
    FINGER_QPOS_ADR = [
        int(model.jnt_qposadr[joint_id]) for joint_id in FINGER_JOINT_IDS
    ]

    assert HAND_BODY_ID == require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "hand")
    assert BLOCK_BODY_ID == require_mj_id(mujoco.mjtObj.mjOBJ_BODY, "red_block")
    assert GELSIGHT_BODY_ID == require_mj_id(
        mujoco.mjtObj.mjOBJ_BODY, GELSIGHT_BODY_NAME
    )
    assert camera_id == require_mj_id(
        mujoco.mjtObj.mjOBJ_CAMERA, AGENT_CAMERA_NAME
    )


def _install_physical_cube_texture():
    global model, data

    assert CUBE_TEXTURE_GRID_SIZE >= 3 and CUBE_TEXTURE_GRID_SIZE % 2 == 1
    assert CUBE_TEXTURE_BUMP_RADIUS_M > 0.0
    expected_bump_count = (
        len(_CUBE_TEXTURE_FACE_SPECS) * CUBE_TEXTURE_GRID_SIZE ** 2
    )
    old_model = model
    old_data = data
    old_dimensions = (
        old_model.nq, old_model.nv, old_model.nu, old_model.na,
        old_model.njnt, old_model.nbody, old_model.nsite, old_model.ncam,
    )
    old_joint_layout = _texture_joint_layout(old_model)
    old_actuator_layout = _texture_actuator_layout(old_model)
    old_qpos0 = old_model.qpos0.copy()
    old_joint_ranges = old_model.jnt_range.copy()
    old_ctrl_ranges = old_model.actuator_ctrlrange.copy()
    old_timestep = float(old_model.opt.timestep)

    cube_body_id = mujoco.mj_name2id(
        old_model, mujoco.mjtObj.mjOBJ_BODY, "red_block"
    )
    core_geom_id = mujoco.mj_name2id(
        old_model, mujoco.mjtObj.mjOBJ_GEOM, "red_block_geom"
    )
    assert cube_body_id >= 0 and core_geom_id >= 0
    assert int(old_model.geom_bodyid[core_geom_id]) == cube_body_id
    assert int(old_model.geom_type[core_geom_id]) == int(mujoco.mjtGeom.mjGEOM_BOX)
    core_half_size = np.asarray(
        old_model.geom_size[core_geom_id, :3], dtype=np.float64
    ).copy()
    assert np.allclose(core_half_size, [0.035, 0.035, 0.035], atol=1e-12)
    assert (
        CUBE_TEXTURE_GRID_HALF_SPAN_M + CUBE_TEXTURE_BUMP_RADIUS_M
        < float(np.min(core_half_size))
    )

    old_cube_mass = float(old_model.body_mass[cube_body_id])
    old_cube_inertia = old_model.body_inertia[cube_body_id].copy()
    old_core_properties = {
        "size": old_model.geom_size[core_geom_id].copy(),
        "friction": old_model.geom_friction[core_geom_id].copy(),
        "solref": old_model.geom_solref[core_geom_id].copy(),
        "solimp": old_model.geom_solimp[core_geom_id].copy(),
        "solmix": float(old_model.geom_solmix[core_geom_id]),
        "condim": int(old_model.geom_condim[core_geom_id]),
        "contype": int(old_model.geom_contype[core_geom_id]),
        "conaffinity": int(old_model.geom_conaffinity[core_geom_id]),
        "priority": int(old_model.geom_priority[core_geom_id]),
        "margin": float(old_model.geom_margin[core_geom_id]),
        "gap": float(old_model.geom_gap[core_geom_id]),
        "group": int(old_model.geom_group[core_geom_id]),
        "matid": int(old_model.geom_matid[core_geom_id]),
    }
    material_name = None
    if old_core_properties["matid"] >= 0:
        material_name = mujoco.mj_id2name(
            old_model, mujoco.mjtObj.mjOBJ_MATERIAL,
            old_core_properties["matid"],
        )
        assert material_name is not None

    prior_live_bump_names = [
        mujoco.mj_id2name(old_model, mujoco.mjtObj.mjOBJ_GEOM, geom_id)
        for geom_id in range(old_model.ngeom)
        if (
            mujoco.mj_id2name(old_model, mujoco.mjtObj.mjOBJ_GEOM, geom_id)
            or ""
        ).startswith(CUBE_TEXTURE_GEOM_PREFIX)
    ]

    source_xml = run_dir / "mujoco_rollout_model_before_cube_texture.xml"
    textured_xml = run_dir / "mujoco_rollout_with_physical_cube_texture.xml"

    # mj_saveLastXML serializes MuJoCo's most recently loaded XML structure,
    # not an arbitrary MjModel's structure. A prior Cell-P run most recently
    # loaded the passive sensor mirror, so saving old_model directly can leak
    # that mirror's invisible sampling geom into the main model. Reload the
    # exact Cell-A scene first, verify it matches every non-texture dynamic
    # layout, and then flatten that known canonical model.
    canonical_xml = Path(PANDA_XML).resolve()
    assert canonical_xml.is_file()
    canonical_model = mujoco.MjModel.from_xml_path(str(canonical_xml))
    assert (
        canonical_model.nq, canonical_model.nv,
        canonical_model.nu, canonical_model.na,
        canonical_model.njnt, canonical_model.nbody,
        canonical_model.nsite, canonical_model.ncam,
    ) == old_dimensions
    assert _texture_joint_layout(canonical_model) == old_joint_layout
    assert _texture_actuator_layout(canonical_model) == old_actuator_layout
    assert np.allclose(canonical_model.qpos0, old_qpos0, rtol=0.0, atol=0.0)
    assert np.allclose(
        canonical_model.jnt_range, old_joint_ranges, rtol=0.0, atol=0.0
    )
    assert np.allclose(
        canonical_model.actuator_ctrlrange, old_ctrl_ranges,
        rtol=0.0, atol=0.0,
    )
    assert np.isclose(
        canonical_model.opt.timestep, old_timestep, rtol=0.0, atol=0.0
    )
    canonical_cube_body_id = mujoco.mj_name2id(
        canonical_model, mujoco.mjtObj.mjOBJ_BODY, "red_block"
    )
    canonical_core_geom_id = mujoco.mj_name2id(
        canonical_model, mujoco.mjtObj.mjOBJ_GEOM, "red_block_geom"
    )
    assert canonical_cube_body_id >= 0 and canonical_core_geom_id >= 0
    assert np.isclose(
        canonical_model.body_mass[canonical_cube_body_id], old_cube_mass,
        rtol=0.0, atol=1e-14,
    )
    assert np.allclose(
        canonical_model.body_inertia[canonical_cube_body_id], old_cube_inertia,
        rtol=0.0, atol=1e-14,
    )
    for field in ("size", "friction", "solref", "solimp"):
        assert np.allclose(
            getattr(canonical_model, f"geom_{field}")[canonical_core_geom_id],
            old_core_properties[field],
        )

    mujoco.mj_saveLastXML(str(source_xml), canonical_model)
    tree = ET.parse(source_xml)
    root = tree.getroot()
    compiler = root.find("compiler")
    if compiler is None:
        compiler = ET.SubElement(root, "compiler")
    compiler.set("meshdir", str((Path(PANDA_DIR) / "assets").resolve()))

    cube_body = next(
        (body for body in root.iter("body") if body.get("name") == "red_block"),
        None,
    )
    assert cube_body is not None
    removed_source_bump_count = 0
    for geom in list(cube_body.findall("geom")):
        if (geom.get("name") or "").startswith(CUBE_TEXTURE_GEOM_PREFIX):
            cube_body.remove(geom)
            removed_source_bump_count += 1
    assert removed_source_bump_count == 0, (
        "Cell-A canonical scene unexpectedly contains Cell-P texture geoms"
    )

    common_attributes = {
        "type": "sphere",
        "size": f"{CUBE_TEXTURE_BUMP_RADIUS_M:.17g}",
        "mass": "0",
        "friction": _texture_number_string(old_core_properties["friction"]),
        "solref": _texture_number_string(old_core_properties["solref"]),
        "solimp": _texture_number_string(old_core_properties["solimp"]),
        "solmix": f"{old_core_properties['solmix']:.17g}",
        "condim": str(old_core_properties["condim"]),
        "contype": str(old_core_properties["contype"]),
        "conaffinity": str(old_core_properties["conaffinity"]),
        "priority": str(old_core_properties["priority"]),
        "margin": f"{old_core_properties['margin']:.17g}",
        "gap": f"{old_core_properties['gap']:.17g}",
        "group": str(old_core_properties["group"]),
    }
    if material_name is not None:
        common_attributes["material"] = material_name
    else:
        common_attributes["rgba"] = _texture_number_string(
            old_model.geom_rgba[core_geom_id]
        )

    lattice = np.linspace(
        -CUBE_TEXTURE_GRID_HALF_SPAN_M,
        CUBE_TEXTURE_GRID_HALF_SPAN_M,
        CUBE_TEXTURE_GRID_SIZE,
        dtype=np.float64,
    )
    expected_positions = {}
    face_counts = {}
    for face_name, normal_axis, sign, row_axis, column_axis in _CUBE_TEXTURE_FACE_SPECS:
        face_counts[face_name] = 0
        for row, row_value in enumerate(lattice):
            for column, column_value in enumerate(lattice):
                local_position = np.zeros(3, dtype=np.float64)
                local_position[normal_axis] = sign * core_half_size[normal_axis]
                local_position[row_axis] = row_value
                local_position[column_axis] = column_value
                geom_name = (
                    f"{CUBE_TEXTURE_GEOM_PREFIX}{face_name}"
                    f"_r{row:02d}_c{column:02d}"
                )
                attributes = dict(common_attributes)
                attributes.update(
                    name=geom_name,
                    pos=_texture_number_string(local_position),
                )
                ET.SubElement(cube_body, "geom", **attributes)
                expected_positions[geom_name] = local_position
                face_counts[face_name] += 1

    assert len(expected_positions) == expected_bump_count
    assert set(face_counts.values()) == {CUBE_TEXTURE_GRID_SIZE ** 2}
    tree.write(textured_xml, encoding="utf-8", xml_declaration=True)

    new_model = mujoco.MjModel.from_xml_path(str(textured_xml))
    assert (
        new_model.nq, new_model.nv, new_model.nu, new_model.na,
        new_model.njnt, new_model.nbody, new_model.nsite, new_model.ncam,
    ) == old_dimensions
    assert new_model.ngeom == (
        canonical_model.ngeom - removed_source_bump_count + expected_bump_count
    )
    assert _texture_joint_layout(new_model) == old_joint_layout
    assert _texture_actuator_layout(new_model) == old_actuator_layout
    assert np.allclose(new_model.qpos0, old_qpos0, rtol=0.0, atol=0.0)
    assert np.allclose(new_model.jnt_range, old_joint_ranges, rtol=0.0, atol=0.0)
    assert np.allclose(
        new_model.actuator_ctrlrange, old_ctrl_ranges, rtol=0.0, atol=0.0
    )
    assert np.isclose(new_model.opt.timestep, old_timestep, rtol=0.0, atol=0.0)

    new_cube_body_id = mujoco.mj_name2id(
        new_model, mujoco.mjtObj.mjOBJ_BODY, "red_block"
    )
    new_core_geom_id = mujoco.mj_name2id(
        new_model, mujoco.mjtObj.mjOBJ_GEOM, "red_block_geom"
    )
    assert np.isclose(new_model.body_mass[new_cube_body_id], old_cube_mass, atol=1e-14)
    assert np.allclose(
        new_model.body_inertia[new_cube_body_id], old_cube_inertia,
        rtol=0.0, atol=1e-14,
    )
    assert np.allclose(
        new_model.geom_size[new_core_geom_id], old_core_properties["size"]
    )
    for field in ("friction", "solref", "solimp"):
        assert np.allclose(
            getattr(new_model, f"geom_{field}")[new_core_geom_id],
            old_core_properties[field],
        )
    for field in ("condim", "contype", "conaffinity", "priority"):
        assert int(getattr(new_model, f"geom_{field}")[new_core_geom_id]) == (
            old_core_properties[field]
        )

    for geom_name, expected_position in expected_positions.items():
        geom_id = mujoco.mj_name2id(
            new_model, mujoco.mjtObj.mjOBJ_GEOM, geom_name
        )
        assert geom_id >= 0
        assert int(new_model.geom_type[geom_id]) == int(
            mujoco.mjtGeom.mjGEOM_SPHERE
        )
        assert int(new_model.geom_bodyid[geom_id]) == new_cube_body_id
        assert np.isclose(
            new_model.geom_size[geom_id, 0], CUBE_TEXTURE_BUMP_RADIUS_M,
            rtol=0.0, atol=1e-12,
        )
        assert np.allclose(
            new_model.geom_pos[geom_id], expected_position,
            rtol=0.0, atol=1e-12,
        )
        assert int(new_model.geom_contype[geom_id]) != 0
        assert int(new_model.geom_conaffinity[geom_id]) != 0
        assert np.allclose(
            new_model.geom_friction[geom_id], old_core_properties["friction"]
        )
        assert int(new_model.geom_matid[geom_id]) == old_core_properties["matid"]

    new_data = mujoco.MjData(new_model)
    _copy_rollout_state(old_data, new_model, new_data)
    assert np.all(np.isfinite(new_data.qpos))
    assert np.all(np.isfinite(new_data.qvel))
    model, data = new_model, new_data
    _refresh_scene_bindings_after_texture()

    return {
        "source_xml": source_xml,
        "source_xml_sha256": _texture_file_sha256(source_xml),
        "textured_xml": textured_xml,
        "textured_xml_sha256": _texture_file_sha256(textured_xml),
        "geom_prefix": CUBE_TEXTURE_GEOM_PREFIX,
        "grid_size_per_face": int(CUBE_TEXTURE_GRID_SIZE),
        "grid_coordinates_m": lattice.tolist(),
        "face_names": [spec[0] for spec in _CUBE_TEXTURE_FACE_SPECS],
        "face_counts": face_counts,
        "bump_radius_m": float(CUBE_TEXTURE_BUMP_RADIUS_M),
        "hemisphere_protrusion_m": float(CUBE_TEXTURE_BUMP_RADIUS_M),
        "core_half_size_m": core_half_size.tolist(),
        "total_bump_geoms": int(expected_bump_count),
        "prior_live_bump_geoms_replaced": int(len(prior_live_bump_names)),
        "canonical_source_bump_geoms_removed": int(
            removed_source_bump_count
        ),
        "added_inertial_mass_kg": 0.0,
        "cube_body_mass_kg_before": old_cube_mass,
        "cube_body_mass_kg_after": float(new_model.body_mass[new_cube_body_id]),
        "joint_body_site_camera_actuator_layout_preserved": True,
        "core_geom_contact_properties_preserved": True,
        "state_copied_before_rollout_reset": True,
    }


CUBE_SURFACE_TEXTURE = _install_physical_cube_texture()
print("Physical cube verification texture ready:")
print(
    f"  {CUBE_TEXTURE_GRID_SIZE}x{CUBE_TEXTURE_GRID_SIZE} hemispheres/face x 6 "
    f"= {CUBE_SURFACE_TEXTURE['total_bump_geoms']} collidable bumps"
)
print(
    f"  radius/protrusion={1e3 * CUBE_TEXTURE_BUMP_RADIUS_M:.2f} mm; "
    "mass and inertia unchanged"
)
print("  textured rollout XML:", CUBE_SURFACE_TEXTURE["textured_xml"])


# -----------------------------------------------------------------------------
# Passive GelSight Mini acquisition model
# -----------------------------------------------------------------------------
# MuJoCo 3.3+ implements a dense tactile sensor whose first channel is maximum
# geometric penetration depth at every sampling-mesh vertex; channels 2 and 3
# are the absolute relative speeds along the two mesh tangents. The official
# example uses an invisible mesh geom welded to the collision body's frame. We
# reproduce that architecture in an auxiliary model. The only main-model
# mutation above is the requested physical cube microtexture; the passive
# sensor itself never applies forces to or advances the rollout model.

_GELSIGHT_SAMPLE_MESH_NAME = "__cell_p_gelsight_taxel_mesh"
_GELSIGHT_SAMPLE_GEOM_NAME = "__cell_p_gelsight_sampling_geom"
_GELSIGHT_NATIVE_SENSOR_NAME = "__cell_p_gelsight_native_tactile"


def _mj_name(model_, obj_type, obj_id):
    name = mujoco.mj_id2name(model_, obj_type, int(obj_id))
    return name if name is not None else f"unnamed_{int(obj_id)}"


def _joint_state_layout(model_):
    return [
        (
            _mj_name(model_, mujoco.mjtObj.mjOBJ_JOINT, joint_id),
            int(model_.jnt_type[joint_id]),
            int(model_.jnt_qposadr[joint_id]),
            int(model_.jnt_dofadr[joint_id]),
        )
        for joint_id in range(model_.njnt)
    ]


def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _build_passive_gelsight_model():
    pad_geom_id = mujoco.mj_name2id(
        model, mujoco.mjtObj.mjOBJ_GEOM, GELSIGHT_GEL_PAD_GEOM_NAME
    )
    body_id = mujoco.mj_name2id(
        model, mujoco.mjtObj.mjOBJ_BODY, GELSIGHT_BODY_NAME
    )
    assert pad_geom_id >= 0 and body_id >= 0
    assert int(model.geom_bodyid[pad_geom_id]) == body_id

    pad_half_size = np.asarray(model.geom_size[pad_geom_id, :3], dtype=np.float64)
    pad_local_pos = np.asarray(model.geom_pos[pad_geom_id], dtype=np.float64)
    pad_face_y = float(pad_local_pos[1] - pad_half_size[1])
    assert pad_face_y < 0.0, "Cell P expects the mounted Mini's sensing face at local -Y."

    flattened_xml = run_dir / "mujoco_rollout_model_flattened.xml"
    sensor_xml = run_dir / "mujoco_rollout_with_passive_gelsight.xml"
    mujoco.mj_saveLastXML(str(flattened_xml), model)

    tree = ET.parse(flattened_xml)
    root = tree.getroot()
    compiler = root.find("compiler")
    if compiler is None:
        compiler = ET.SubElement(root, "compiler")
    compiler.set("meshdir", str((Path(PANDA_DIR) / "assets").resolve()))

    asset = root.find("asset")
    assert asset is not None, "Flattened MuJoCo model has no asset element."
    for element in root.iter():
        assert element.get("name") not in {
            _GELSIGHT_SAMPLE_MESH_NAME,
            _GELSIGHT_SAMPLE_GEOM_NAME,
            _GELSIGHT_NATIVE_SENSOR_NAME,
        }

    # Built-in plate samples its cell centers. Slightly enlarge the mesh scale
    # so the outer taxel centers coincide with the existing 18 x 26 mm gel pad.
    scale_x = pad_half_size[0] * GELSIGHT_TAXELS_X / (GELSIGHT_TAXELS_X - 1)
    scale_z = pad_half_size[2] * GELSIGHT_TAXELS_Z / (GELSIGHT_TAXELS_Z - 1)
    ET.SubElement(
        asset,
        "mesh",
        name=_GELSIGHT_SAMPLE_MESH_NAME,
        builtin="plate",
        params=f"{GELSIGHT_TAXELS_X} {GELSIGHT_TAXELS_Z}",
        scale=f"{scale_x:.17g} {scale_z:.17g} {abs(pad_face_y):.17g}",
    )

    sensor_body = next(
        (body for body in root.iter("body") if body.get("name") == GELSIGHT_BODY_NAME),
        None,
    )
    assert sensor_body is not None
    ET.SubElement(
        sensor_body,
        "geom",
        name=_GELSIGHT_SAMPLE_GEOM_NAME,
        type="mesh",
        mesh=_GELSIGHT_SAMPLE_MESH_NAME,
        euler=f"{-np.pi / 2.0:.17g} 0 0",
        mass="0",
        contype="0",
        conaffinity="0",
        rgba="0 0 0 0",
        group="5",
    )

    sensors = root.find("sensor")
    if sensors is None:
        sensors = ET.SubElement(root, "sensor")
    # Important: geom is the invisible sampling mesh, exactly as in MuJoCo's
    # official tactile.xml. The existing box remains the physical contact geom.
    ET.SubElement(
        sensors,
        "tactile",
        name=_GELSIGHT_NATIVE_SENSOR_NAME,
        geom=_GELSIGHT_SAMPLE_GEOM_NAME,
        mesh=_GELSIGHT_SAMPLE_MESH_NAME,
    )
    tree.write(sensor_xml, encoding="utf-8", xml_declaration=True)

    sensor_model = mujoco.MjModel.from_xml_path(str(sensor_xml))
    sensor_data = mujoco.MjData(sensor_model)
    assert (sensor_model.nq, sensor_model.nv, sensor_model.nu, sensor_model.na) == (
        model.nq,
        model.nv,
        model.nu,
        model.na,
    )
    assert _joint_state_layout(sensor_model) == _joint_state_layout(model)
    assert np.isclose(sensor_model.opt.timestep, model.opt.timestep)

    sensor_id = mujoco.mj_name2id(
        sensor_model, mujoco.mjtObj.mjOBJ_SENSOR, _GELSIGHT_NATIVE_SENSOR_NAME
    )
    assert sensor_id >= 0
    sensor_dim = int(sensor_model.sensor_dim[sensor_id])
    expected_dim = 3 * GELSIGHT_TAXELS_X * GELSIGHT_TAXELS_Z
    assert sensor_dim == expected_dim, (sensor_dim, expected_dim)
    sensor_pad_geom_id = mujoco.mj_name2id(
        sensor_model, mujoco.mjtObj.mjOBJ_GEOM, GELSIGHT_GEL_PAD_GEOM_NAME
    )
    sensor_cube_geom_id = mujoco.mj_name2id(
        sensor_model, mujoco.mjtObj.mjOBJ_GEOM, "red_block_geom"
    )
    sensor_cube_body_id = mujoco.mj_name2id(
        sensor_model, mujoco.mjtObj.mjOBJ_BODY, "red_block"
    )
    assert (
        sensor_pad_geom_id >= 0
        and sensor_cube_geom_id >= 0
        and sensor_cube_body_id >= 0
    )
    sensor_cube_geom_ids = tuple(
        geom_id
        for geom_id in range(sensor_model.ngeom)
        if int(sensor_model.geom_bodyid[geom_id]) == sensor_cube_body_id
    )
    sensor_cube_geom_names = tuple(
        _mj_name(sensor_model, mujoco.mjtObj.mjOBJ_GEOM, geom_id)
        for geom_id in sensor_cube_geom_ids
    )
    assert sensor_cube_geom_id in sensor_cube_geom_ids
    assert len([
        name for name in sensor_cube_geom_names
        if name.startswith(CUBE_TEXTURE_GEOM_PREFIX)
    ]) == CUBE_SURFACE_TEXTURE["total_bump_geoms"]
    assert sensor_model.ngeom == model.ngeom + 1
    assert np.allclose(
        sensor_model.geom_size[sensor_pad_geom_id], model.geom_size[pad_geom_id]
    )
    assert np.allclose(
        sensor_model.geom_friction[sensor_pad_geom_id], model.geom_friction[pad_geom_id]
    )

    return {
        "model": sensor_model,
        "data": sensor_data,
        "sensor_id": int(sensor_id),
        "sensor_adr": int(sensor_model.sensor_adr[sensor_id]),
        "sensor_dim": sensor_dim,
        "main_pad_geom_id": int(pad_geom_id),
        "sensor_pad_geom_id": int(sensor_pad_geom_id),
        "sensor_cube_geom_id": int(sensor_cube_geom_id),
        "sensor_cube_body_id": int(sensor_cube_body_id),
        "sensor_cube_geom_ids": sensor_cube_geom_ids,
        "sensor_cube_geom_names": sensor_cube_geom_names,
        "pad_half_size_m": pad_half_size,
        "pad_face_local_y_m": pad_face_y,
        "flattened_xml": flattened_xml,
        "sensor_xml": sensor_xml,
        "flattened_xml_sha256": _sha256_file(flattened_xml),
        "sensor_xml_sha256": _sha256_file(sensor_xml),
    }


# The baseline image and polynomial table are calibration assets from Taxim,
# not rollout observations. They only convert MuJoCo's measured indentation into
# a familiar GelSight RGB view. Quantitative native depth and solver forces are
# always saved separately and are never inferred from this RGB rendering.
# Stored as sidecar files (not inlined) to keep this cell readable; both are
# still SHA256-verified on every load against the hashes recorded below.
_TAXIM_ASSETS_DIR = Path(PROJECT_DIR if "PROJECT_DIR" in globals() else Path.cwd()) / "assets"
_TAXIM_BACKGROUND_PNG_PATH = _TAXIM_ASSETS_DIR / "taxim_background.png"
_TAXIM_POLYCALIB_NPZ_PATH = _TAXIM_ASSETS_DIR / "taxim_polycalib.npz"
_TAXIM_BACKGROUND_SHA256 = "430ef8cbfc260c2f475f88dfa284327719a5bc43f886d4b2e172530b7b7f1ec7"
_TAXIM_POLYCALIB_SHA256 = "a7c45e66a764362996f66259d5a1e9b66569e94e032216ef1406727dd664c2b5"
_TAXIM_SOURCE_COMMIT = "4936657a42e00f09a94301111830250ef2c96a89"
_TAXIM_SOURCE_DATA_PACK_SHA256 = "090d348cb7f5ea5ad1d68c147580048780f507bc0e43aa656c3dbf4d667af5c6"
_TAXIM_SOURCE_POLYCALIB_SHA256 = "6ea61fd6a8dd1b32ddccdf0727b5abbfcdc128942ccc81760fea0196c191409a"
_TAXIM_MIT_LICENSE = """MIT License

Copyright (c) 2021 CMURoboTouch

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE."""


def _decode_taxim_calibration():
    background_bytes = _TAXIM_BACKGROUND_PNG_PATH.read_bytes()
    calibration_bytes = _TAXIM_POLYCALIB_NPZ_PATH.read_bytes()
    assert hashlib.sha256(background_bytes).hexdigest() == _TAXIM_BACKGROUND_SHA256
    assert hashlib.sha256(calibration_bytes).hexdigest() == _TAXIM_POLYCALIB_SHA256

    # The source pack was collected through OpenCV, so this array remains BGR
    # until the final renderer conversion to imageio/PIL RGB.
    background_bgr = np.asarray(
        Image.open(io.BytesIO(background_bytes)).convert("RGB"), dtype=np.float32
    )
    assert background_bgr.shape == (GELSIGHT_RGB_HEIGHT, GELSIGHT_RGB_WIDTH, 3)
    with np.load(io.BytesIO(calibration_bytes), allow_pickle=False) as calibration:
        bins = int(calibration["bins"])
        tables_bgr = np.stack([
            calibration["grad_r"],
            calibration["grad_g"],
            calibration["grad_b"],
        ]).astype(np.float32)
    assert bins == 125 and tables_bgr.shape == (3, bins, bins, 6)
    return background_bgr, tables_bgr, bins


class TaximGelSightOpticalRenderer:
    """Taxim-calibrated qualitative RGB view of native MuJoCo indentation."""

    def __init__(self, pad_half_size_m):
        self.height = GELSIGHT_RGB_HEIGHT
        self.width = GELSIGHT_RGB_WIDTH
        self.pad_width_m = 2.0 * float(pad_half_size_m[0])
        self.pad_height_m = 2.0 * float(pad_half_size_m[2])
        background_bgr, self.tables_bgr, self.bins = _decode_taxim_calibration()

        # Taxim's initial-frame processing, spatially scaled from 640x480 to
        # this 320x240 stream. It keeps real baseline texture while suppressing
        # isolated marker/high-frequency artifacts.
        filtered = np.stack([
            gaussian_filter(background_bgr[..., channel], sigma=25.0, mode="nearest")
            for channel in range(3)
        ], axis=-1)
        use_mixed = np.mean(filtered - background_bgr, axis=2) < 5.0
        self.background_bgr = filtered
        self.background_bgr[use_mixed] = (
            0.15 * filtered[use_mixed] + 0.85 * background_bgr[use_mixed]
        )

        x_calib = np.linspace(0.0, 639.0, self.width, dtype=np.float32)
        y_calib = np.linspace(0.0, 479.0, self.height, dtype=np.float32)
        xx, yy = np.meshgrid(x_calib, y_calib)
        self.polynomial_basis = np.stack([
            xx * xx,
            yy * yy,
            xx * yy,
            xx,
            yy,
            np.ones_like(xx),
        ], axis=-1)
        self.flat_direction_bin = int(np.floor((self.bins - 1) / 2.0))
        self.flat_delta_bgr = np.stack([
            np.sum(
                self.polynomial_basis
                * self.tables_bgr[channel, 0, self.flat_direction_bin],
                axis=-1,
            )
            for channel in range(3)
        ], axis=-1)
        self.baseline_rgb, _ = self._render_impl(
            np.zeros((GELSIGHT_TAXELS_Z, GELSIGHT_TAXELS_X), dtype=np.float32)
        )

    def _resize_depth(self, native_depth_m):
        image = Image.fromarray(np.asarray(native_depth_m, dtype=np.float32))
        return np.asarray(
            image.resize((self.width, self.height), Image.Resampling.BILINEAR),
            dtype=np.float32,
        )

    def _elastic_deformation(self, native_depth_m):
        raw = np.clip(
            self._resize_depth(native_depth_m), 0.0, GELSIGHT_GEL_THICKNESS_M
        )
        peak = float(np.max(raw))
        if peak <= 0.0:
            return raw, raw

        # Taxim-style multi-scale Gaussian relaxation. Native penetration is
        # retained at the deepest physical contact pixels; only the surrounding
        # elastomer response is interpolated. No object silhouette is painted.
        locked = (raw >= 0.40 * peak) & (raw > GELSIGHT_DEPTH_ACTIVE_EPS_M)
        deformed = raw.copy()
        for sigma in (12.0, 6.0, 3.0, 1.5):
            deformed = gaussian_filter(deformed, sigma=sigma, mode="nearest")
            deformed[locked] = raw[locked]
        deformed = gaussian_filter(deformed, sigma=0.8, mode="nearest")
        return raw, np.clip(deformed, 0.0, GELSIGHT_GEL_THICKNESS_M)

    def _render_impl(self, native_depth_m):
        raw, deformed = self._elastic_deformation(native_depth_m)
        spacing_z = self.pad_height_m / max(self.height - 1, 1)
        spacing_x = self.pad_width_m / max(self.width - 1, 1)
        grad_z, grad_x = np.gradient(deformed, spacing_z, spacing_x)
        tangent_magnitude = np.sqrt(grad_x * grad_x + grad_z * grad_z)
        gradient_magnitude = np.arctan(tangent_magnitude)
        gradient_direction = np.arctan2(grad_z, grad_x)

        magnitude_step = 0.5 * np.pi / (self.bins - 1)
        direction_step = 2.0 * np.pi / (self.bins - 1)
        magnitude_bin = np.clip(
            np.floor(gradient_magnitude / magnitude_step).astype(np.int32),
            0,
            self.bins - 1,
        )
        direction_bin = np.clip(
            np.floor((gradient_direction + np.pi) / direction_step).astype(np.int32),
            0,
            self.bins - 1,
        )

        rendered_bgr = self.background_bgr.copy()
        for channel in range(3):
            coefficients = self.tables_bgr[channel, magnitude_bin, direction_bin]
            calibrated_delta = np.sum(
                self.polynomial_basis * coefficients, axis=-1
            )
            rendered_bgr[..., channel] += (
                calibrated_delta - self.flat_delta_bgr[..., channel]
            )

        # A shallow physical contact changes the coating reflectance even when
        # a large flat cube fills the entire view and has no visible silhouette.
        # This fixed response depends only on native penetration (50 um scale),
        # never on proximity, policy phase, cube pose, or a fabricated mask.
        contact_response = 1.0 - np.exp(-raw / 50.0e-6)
        rendered_bgr += contact_response[..., None] * np.array(
            [-9.0, 10.0, 5.0], dtype=np.float32
        )
        rendered_bgr *= (1.0 - 0.08 * contact_response)[..., None]
        rendered_bgr = gaussian_filter(
            rendered_bgr, sigma=(0.5, 0.5, 0.0), mode="nearest"
        )
        # Taxim calibration assets are OpenCV BGR; videos are standard RGB.
        rendered_rgb = np.clip(rendered_bgr[..., ::-1], 0.0, 255.0).astype(np.uint8)
        return rendered_rgb, deformed.astype(np.float32)

    def render(self, native_depth_m):
        return self._render_impl(native_depth_m)


GELSIGHT_ACQUISITION = _build_passive_gelsight_model()
GELSIGHT_OPTICAL_RENDERER = TaximGelSightOpticalRenderer(
    GELSIGHT_ACQUISITION["pad_half_size_m"]
)


def _sync_passive_gelsight_state():
    sensor_data = GELSIGHT_ACQUISITION["data"]
    sensor_model = GELSIGHT_ACQUISITION["model"]
    sensor_data.time = float(data.time)
    sensor_data.qpos[:] = data.qpos
    sensor_data.qvel[:] = data.qvel
    if sensor_model.na:
        sensor_data.act[:] = data.act
    if sensor_model.nu:
        sensor_data.ctrl[:] = data.ctrl
    sensor_data.qacc_warmstart[:] = data.qacc_warmstart
    sensor_data.qfrc_applied[:] = data.qfrc_applied
    sensor_data.xfrc_applied[:] = data.xfrc_applied
    if sensor_model.nmocap:
        sensor_data.mocap_pos[:] = data.mocap_pos
        sensor_data.mocap_quat[:] = data.mocap_quat
    if sensor_model.nuserdata:
        sensor_data.userdata[:] = data.userdata
    if sensor_model.neq:
        sensor_data.eq_active[:] = data.eq_active
    mujoco.mj_forward(sensor_model, sensor_data)


def _synchronized_pad_contact_snapshot():
    # mj_step leaves the main data.contact array at the pre-integration state.
    # The passive model has just run mj_forward at the copied post-step qpos,
    # qvel, act, ctrl, warm-start, and applied forces, so its tactile field and
    # solver wrench are synchronous without modifying the rollout data object.
    sensor_model = GELSIGHT_ACQUISITION["model"]
    sensor_data = GELSIGHT_ACQUISITION["data"]
    pad_id = GELSIGHT_ACQUISITION["sensor_pad_geom_id"]
    cube_body_id = GELSIGHT_ACQUISITION["sensor_cube_body_id"]
    pad_half_size = GELSIGHT_ACQUISITION["pad_half_size_m"]
    pad_world_pos = np.asarray(sensor_data.geom_xpos[pad_id], dtype=np.float64)
    pad_world_rotation = np.asarray(
        sensor_data.geom_xmat[pad_id], dtype=np.float64
    ).reshape(3, 3)

    normal_grid = np.zeros(
        (GELSIGHT_TAXELS_Z, GELSIGHT_TAXELS_X), dtype=np.float32
    )
    tangent_grid = np.zeros_like(normal_grid)
    contacts = []
    normal_force_total = 0.0
    tangential_force_total = 0.0
    cube_contact_count = 0
    cube_texture_bump_contact_count = 0
    maximum_penetration_m = 0.0

    for contact_index in range(int(sensor_data.ncon)):
        contact = sensor_data.contact[contact_index]
        geom1 = int(contact.geom1)
        geom2 = int(contact.geom2)
        if pad_id not in (geom1, geom2):
            continue
        other_id = geom2 if geom1 == pad_id else geom1
        wrench = np.zeros(6, dtype=np.float64)
        mujoco.mj_contactForce(
            sensor_model, sensor_data, contact_index, wrench
        )
        assert np.all(np.isfinite(wrench))
        normal_force = abs(float(wrench[0]))
        tangential_force = float(np.linalg.norm(wrench[1:3]))
        world_position = np.asarray(contact.pos, dtype=np.float64).copy()
        local_position = pad_world_rotation.T @ (world_position - pad_world_pos)

        column = int(np.clip(np.rint(
            (local_position[0] + pad_half_size[0])
            / (2.0 * pad_half_size[0])
            * (GELSIGHT_TAXELS_X - 1)
        ), 0, GELSIGHT_TAXELS_X - 1))
        row = int(np.clip(np.rint(
            (pad_half_size[2] - local_position[2])
            / (2.0 * pad_half_size[2])
            * (GELSIGHT_TAXELS_Z - 1)
        ), 0, GELSIGHT_TAXELS_Z - 1))
        # Sparse histogram of actual MuJoCo solver contacts. No Gaussian
        # pressure blob, proximity trigger, or hand-drawn silhouette is used.
        normal_grid[row, column] += normal_force
        tangent_grid[row, column] += tangential_force

        # The textured cube is a compound body: most tactile contacts are with
        # a hemisphere rather than red_block_geom. Body ownership is the
        # physically correct classifier for every core/bump contact.
        other_name = _mj_name(
            sensor_model, mujoco.mjtObj.mjOBJ_GEOM, other_id
        )
        pair_is_cube = bool(
            0 <= other_id < sensor_model.ngeom
            and int(sensor_model.geom_bodyid[other_id]) == cube_body_id
        )
        pair_is_texture_bump = bool(
            pair_is_cube and other_name.startswith(CUBE_TEXTURE_GEOM_PREFIX)
        )
        cube_contact_count += int(pair_is_cube)
        cube_texture_bump_contact_count += int(pair_is_texture_bump)
        penetration_m = max(0.0, -float(contact.dist))
        maximum_penetration_m = max(maximum_penetration_m, penetration_m)
        normal_force_total += normal_force
        tangential_force_total += tangential_force
        contacts.append({
            "contact_index": int(contact_index),
            "geom1": _mj_name(sensor_model, mujoco.mjtObj.mjOBJ_GEOM, geom1),
            "geom2": _mj_name(sensor_model, mujoco.mjtObj.mjOBJ_GEOM, geom2),
            "other_geom": other_name,
            "is_cube_contact": bool(pair_is_cube),
            "is_cube_texture_bump_contact": bool(pair_is_texture_bump),
            "distance_m": float(contact.dist),
            "position_world_m": world_position.tolist(),
            "position_pad_frame_m": local_position.tolist(),
            "mj_contact_force_frame_N": wrench[:3].tolist(),
            "mj_contact_torque_frame_Nm": wrench[3:].tolist(),
            "normal_force_magnitude_N": normal_force,
            "tangential_force_magnitude_N": tangential_force,
            "force_grid_row": row,
            "force_grid_column": column,
        })

    return {
        "normal_force_grid_N": normal_grid,
        "tangential_force_grid_N": tangent_grid,
        "contacts": contacts,
        "pad_contact_count": len(contacts),
        "cube_contact_count": int(cube_contact_count),
        "cube_texture_bump_contact_count": int(
            cube_texture_bump_contact_count
        ),
        "normal_force_total_N": float(normal_force_total),
        "tangential_force_total_N": float(tangential_force_total),
        "maximum_solver_contact_penetration_m": float(maximum_penetration_m),
    }


def _fixed_scale_depth_rgb(depth_m):
    value = np.clip(
        np.asarray(depth_m, dtype=np.float32) / GELSIGHT_DEPTH_DISPLAY_MAX_M,
        0.0,
        1.0,
    )
    # Perceptual gamma (<1) boosts contrast for the shallow, tens-of-micron
    # native depths typical of this rigid pad; exact zero stays exact black.
    value = value ** GELSIGHT_DEPTH_DISPLAY_GAMMA
    # Fixed, documented scale. Exactly zero depth is exactly black.
    red = np.clip(2.0 * value - 0.5, 0.0, 1.0)
    green = np.clip(2.0 * value, 0.0, 1.0)
    blue = np.clip(4.0 * value, 0.0, 1.0) * (
        1.0 - np.clip(2.0 * value - 1.0, 0.0, 1.0)
    )
    rgb = np.stack([red, green, blue], axis=-1)
    return np.asarray(
        Image.fromarray((255.0 * rgb).astype(np.uint8)).resize(
            (GELSIGHT_RGB_WIDTH, GELSIGHT_RGB_HEIGHT), Image.Resampling.NEAREST
        )
    )


def _fixed_scale_force_rgb(force_N):
    value = np.clip(
        np.asarray(force_N, dtype=np.float32) / GELSIGHT_FORCE_DISPLAY_MAX_N,
        0.0,
        1.0,
    )
    rgb = np.stack([
        value,
        np.clip(2.0 * value - 0.5, 0.0, 1.0),
        np.zeros_like(value),
    ], axis=-1)
    return np.asarray(
        Image.fromarray((255.0 * rgb).astype(np.uint8)).resize(
            (GELSIGHT_RGB_WIDTH, GELSIGHT_RGB_HEIGHT), Image.Resampling.NEAREST
        )
    )


def _audit_panel(rgb_frame, depth_m, normal_force_grid_N, frame_meta):
    panel = Image.new("RGB", (3 * GELSIGHT_RGB_WIDTH, 288), (18, 18, 22))
    draw = ImageDraw.Draw(panel)
    panel.paste(Image.fromarray(rgb_frame), (0, 28))
    panel.paste(Image.fromarray(_fixed_scale_depth_rgb(depth_m)), (GELSIGHT_RGB_WIDTH, 28))
    panel.paste(Image.fromarray(_fixed_scale_force_rgb(normal_force_grid_N)), (2 * GELSIGHT_RGB_WIDTH, 28))
    draw.text((6, 7), "Taxim-calibrated RGB", fill=(240, 240, 240))
    draw.text(
        (GELSIGHT_RGB_WIDTH + 6, 7),
        f"Native depth (fixed 0-{GELSIGHT_DEPTH_DISPLAY_MAX_M * 1e3:.1f} mm)",
        fill=(240, 240, 240),
    )
    draw.text(
        (2 * GELSIGHT_RGB_WIDTH + 6, 7),
        f"Matched mj_contactForce (fixed 0-{GELSIGHT_FORCE_DISPLAY_MAX_N:.1f} N/taxel)",
        fill=(240, 240, 240),
    )
    draw.text(
        (6, 271),
        (
            f"t={frame_meta['sim_time_s']:.3f}s  "
            f"depth={frame_meta['max_native_depth_m'] * 1e3:.3f}mm  "
            f"normal={frame_meta['normal_force_total_N']:.3f}N  "
            f"pad/cube/bump contacts={frame_meta['pad_contact_count']}/"
            f"{frame_meta['cube_contact_count']}/"
            f"{frame_meta['cube_texture_bump_contact_count']}"
        ),
        fill=(235, 235, 235),
    )
    return np.asarray(panel)


class GelSightRolloutRecorder:
    """Acquire synchronized live MuJoCo tactile fields and solver contacts."""

    def __init__(self, rollout_dir):
        self.rollout_dir = Path(rollout_dir)
        self.sample_period_s = 1.0 / GELSIGHT_TACTILE_FPS
        self.next_sample_time = float(data.time)
        self.last_sample_time = None
        self.frames = []
        self.depth_frames = []
        self.tangent_x_frames = []
        self.tangent_z_frames = []
        self.normal_force_frames = []
        self.tangential_force_frames = []
        self.finalized_outputs = None
        self.rgb_video_path = self.rollout_dir / "gelsight_rgb.mp4"
        self.audit_video_path = self.rollout_dir / "gelsight_audit.mp4"
        writer_kwargs = {
            "fps": GELSIGHT_TACTILE_FPS,
            "codec": "libx264",
            "quality": 8,
            "pixelformat": "yuv420p",
            "macro_block_size": None,
        }
        self.rgb_writer = imageio.get_writer(str(self.rgb_video_path), **writer_kwargs)
        self.audit_writer = imageio.get_writer(str(self.audit_video_path), **writer_kwargs)

    @property
    def frame_count(self):
        return len(self.frames)

    def capture_initial(self):
        self._capture()
        self.next_sample_time = float(data.time) + self.sample_period_s

    def after_physics_step(self):
        if float(data.time) + 1.0e-12 < self.next_sample_time:
            return
        self._capture()
        while self.next_sample_time <= float(data.time) + 1.0e-12:
            self.next_sample_time += self.sample_period_s

    def capture_final_if_new(self):
        if self.last_sample_time is None or abs(float(data.time) - self.last_sample_time) > 1.0e-12:
            self._capture()

    def _capture(self):
        _sync_passive_gelsight_state()
        contact = _synchronized_pad_contact_snapshot()
        sensor_data = GELSIGHT_ACQUISITION["data"]
        adr = GELSIGHT_ACQUISITION["sensor_adr"]
        dim = GELSIGHT_ACQUISITION["sensor_dim"]
        raw = np.asarray(sensor_data.sensordata[adr:adr + dim], dtype=np.float64).copy()
        assert np.all(np.isfinite(raw))
        taxel_count = GELSIGHT_TAXELS_X * GELSIGHT_TAXELS_Z
        depth_m = np.maximum(raw[:taxel_count], 0.0).reshape(
            GELSIGHT_TAXELS_X, GELSIGHT_TAXELS_Z
        ).T.astype(np.float32)
        tangent_x_m_s = np.maximum(raw[taxel_count:2 * taxel_count], 0.0).reshape(
            GELSIGHT_TAXELS_X, GELSIGHT_TAXELS_Z
        ).T.astype(np.float32)
        tangent_z_m_s = np.maximum(raw[2 * taxel_count:], 0.0).reshape(
            GELSIGHT_TAXELS_X, GELSIGHT_TAXELS_Z
        ).T.astype(np.float32)

        rgb_frame, _ = GELSIGHT_OPTICAL_RENDERER.render(depth_m)
        optical_absolute_delta = np.abs(
            rgb_frame.astype(np.float32)
            - GELSIGHT_OPTICAL_RENDERER.baseline_rgb.astype(np.float32)
        )
        optical_delta = float(np.mean(optical_absolute_delta))
        optical_p95_delta = float(np.quantile(optical_absolute_delta, 0.95))
        optical_p99_delta = float(np.quantile(optical_absolute_delta, 0.99))
        depth_residual = depth_m - gaussian_filter(
            depth_m, sigma=1.0, mode="nearest"
        )
        native_depth_high_frequency_rms_m = float(np.sqrt(np.mean(
            depth_residual * depth_residual
        )))
        frame_index = len(self.frames)
        frame_meta = {
            "frame_index": int(frame_index),
            "sim_time_s": float(data.time),
            "controller_phase": str(globals().get("CONTROLLER_STATE", {}).get("mode", "initial")),
            "pad_contact_count": int(contact["pad_contact_count"]),
            "cube_contact_count": int(contact["cube_contact_count"]),
            "cube_texture_bump_contact_count": int(
                contact["cube_texture_bump_contact_count"]
            ),
            "normal_force_total_N": float(contact["normal_force_total_N"]),
            "tangential_force_total_N": float(contact["tangential_force_total_N"]),
            "maximum_solver_contact_penetration_m": float(
                contact["maximum_solver_contact_penetration_m"]
            ),
            "max_native_depth_m": float(np.max(depth_m)),
            "mean_native_depth_m": float(np.mean(depth_m)),
            "native_depth_high_frequency_rms_m": (
                native_depth_high_frequency_rms_m
            ),
            "active_native_taxels": int(np.count_nonzero(
                depth_m > GELSIGHT_DEPTH_ACTIVE_EPS_M
            )),
            "max_tangent_x_speed_m_s": float(np.max(tangent_x_m_s)),
            "max_tangent_z_speed_m_s": float(np.max(tangent_z_m_s)),
            "optical_mean_absolute_delta_rgb": optical_delta,
            "optical_p95_absolute_delta_rgb": optical_p95_delta,
            "optical_p99_absolute_delta_rgb": optical_p99_delta,
            "contacts": contact["contacts"],
        }
        self.rgb_writer.append_data(rgb_frame)
        self.audit_writer.append_data(_audit_panel(
            rgb_frame,
            depth_m,
            contact["normal_force_grid_N"],
            frame_meta,
        ))
        self.frames.append(frame_meta)
        self.depth_frames.append(depth_m)
        self.tangent_x_frames.append(tangent_x_m_s)
        self.tangent_z_frames.append(tangent_z_m_s)
        self.normal_force_frames.append(contact["normal_force_grid_N"])
        self.tangential_force_frames.append(contact["tangential_force_grid_N"])
        self.last_sample_time = float(data.time)

    def close_video_writers(self):
        if self.rgb_writer is not None:
            self.rgb_writer.close()
            self.rgb_writer = None
        if self.audit_writer is not None:
            self.audit_writer.close()
            self.audit_writer = None

    def _validate(self, incomplete):
        depth = np.stack(self.depth_frames)
        times = np.asarray([frame["sim_time_s"] for frame in self.frames])
        pad_contacts = np.asarray([frame["pad_contact_count"] for frame in self.frames]) > 0
        cube_contacts = np.asarray([frame["cube_contact_count"] for frame in self.frames]) > 0
        bump_contacts = np.asarray([
            frame["cube_texture_bump_contact_count"] for frame in self.frames
        ]) > 0
        depth_active = np.max(depth, axis=(1, 2)) > GELSIGHT_DEPTH_ACTIVE_EPS_M
        optical_delta = np.asarray([
            frame["optical_mean_absolute_delta_rgb"] for frame in self.frames
        ])
        optical_p95_delta = np.asarray([
            frame["optical_p95_absolute_delta_rgb"] for frame in self.frames
        ])
        optical_p99_delta = np.asarray([
            frame["optical_p99_absolute_delta_rgb"] for frame in self.frames
        ])
        normal_force = np.asarray([
            frame["normal_force_total_N"] for frame in self.frames
        ])

        no_depth_without_pad_contact = bool(np.all(~depth_active | pad_contacts))
        simultaneous_cube_depth = bool(np.any(cube_contacts & depth_active))
        cube_contact_observed = bool(np.any(cube_contacts))
        bump_contact_observed = bool(np.any(bump_contacts))
        simultaneous_bump_depth = bool(np.any(bump_contacts & depth_active))
        optical_contact_response = bool(
            np.any(optical_p99_delta[cube_contacts] >= 3.0)
        ) if cube_contact_observed else False
        maximum_depth_m = float(np.max(depth))
        physically_bounded_depth = bool(
            maximum_depth_m <= GELSIGHT_GEL_THICKNESS_M + 1.0e-9
        )

        release_recovery = "not_observed"
        if np.any(pad_contacts):
            last_contact = int(np.flatnonzero(pad_contacts)[-1])
            if last_contact + 1 < len(pad_contacts):
                release_recovery = bool(
                    np.max(depth[last_contact + 1:]) <= GELSIGHT_DEPTH_ACTIVE_EPS_M
                )

        force_depth_correlation = None
        variable = pad_contacts & depth_active
        if np.count_nonzero(variable) >= 3:
            selected_force = normal_force[variable]
            selected_depth = np.max(depth[variable], axis=(1, 2))
            if np.ptp(selected_force) > 1.0e-9 and np.ptp(selected_depth) > 1.0e-9:
                correlation = float(np.corrcoef(selected_force, selected_depth)[0, 1])
                if np.isfinite(correlation):
                    force_depth_correlation = correlation

        passed = bool(
            (not incomplete)
            and cube_contact_observed
            and simultaneous_cube_depth
            and bump_contact_observed
            and simultaneous_bump_depth
            and optical_contact_response
            and no_depth_without_pad_contact
            and physically_bounded_depth
            and (release_recovery is not False)
            and np.all(np.isfinite(times))
            and np.all(np.isfinite(normal_force))
        )
        status = (
            "passed"
            if passed
            else "not_assessable_no_cube_contact"
            if not cube_contact_observed
            else "failed"
        )
        return {
            "status": status,
            "passed": passed,
            "incomplete_rollout": bool(incomplete),
            "frame_count": int(len(self.frames)),
            "sample_rate_hz": int(GELSIGHT_TACTILE_FPS),
            "cube_contact_observed": cube_contact_observed,
            "simultaneous_cube_contact_and_native_depth": simultaneous_cube_depth,
            "physical_cube_texture_bump_contact_observed": bump_contact_observed,
            "simultaneous_texture_bump_contact_and_native_depth": simultaneous_bump_depth,
            "no_native_depth_without_live_pad_contact": no_depth_without_pad_contact,
            "optical_response_at_cube_contact": optical_contact_response,
            "release_recovered_to_baseline": release_recovery,
            "maximum_native_depth_m": maximum_depth_m,
            "maximum_allowed_gel_depth_m": float(GELSIGHT_GEL_THICKNESS_M),
            "physically_bounded_depth": physically_bounded_depth,
            "maximum_synchronized_solver_normal_force_N": float(np.max(normal_force)),
            "maximum_optical_p95_absolute_delta_rgb": float(np.max(optical_p95_delta)),
            "maximum_optical_p99_absolute_delta_rgb": float(np.max(optical_p99_delta)),
            "force_depth_correlation_when_variable": force_depth_correlation,
            "notes": (
                "A no-contact rollout is never labeled as a passed tactile test. "
                "Correlation is diagnostic only because force-depth response depends "
                "on contact area, orientation, and MuJoCo compliance."
            ),
        }

    def finalize(self, incomplete=False):
        if self.finalized_outputs is not None:
            return self.finalized_outputs
        self.capture_final_if_new()
        self.close_video_writers()
        assert self.frames

        raw_data_path = self.rollout_dir / "gelsight_quantitative_data.npz"
        np.savez_compressed(
            raw_data_path,
            sim_time_s=np.asarray([frame["sim_time_s"] for frame in self.frames], dtype=np.float64),
            native_depth_m=np.stack(self.depth_frames).astype(np.float32),
            native_tangent_x_speed_m_s=np.stack(self.tangent_x_frames).astype(np.float32),
            native_tangent_z_speed_m_s=np.stack(self.tangent_z_frames).astype(np.float32),
            solver_normal_force_histogram_N=np.stack(self.normal_force_frames).astype(np.float32),
            solver_tangential_force_histogram_N=np.stack(self.tangential_force_frames).astype(np.float32),
            pad_contact_count=np.asarray([frame["pad_contact_count"] for frame in self.frames], dtype=np.int32),
            cube_contact_count=np.asarray([frame["cube_contact_count"] for frame in self.frames], dtype=np.int32),
            cube_texture_bump_contact_count=np.asarray([frame["cube_texture_bump_contact_count"] for frame in self.frames], dtype=np.int32),
            synchronized_solver_normal_force_N=np.asarray([frame["normal_force_total_N"] for frame in self.frames], dtype=np.float32),
            synchronized_solver_tangential_force_N=np.asarray([frame["tangential_force_total_N"] for frame in self.frames], dtype=np.float32),
        )

        contacts_path = self.rollout_dir / "gelsight_live_contacts.jsonl"
        with contacts_path.open("w", encoding="utf-8") as stream:
            for frame in self.frames:
                stream.write(json.dumps(frame, allow_nan=False) + "\n")

        validation = self._validate(incomplete=incomplete)
        validation_path = self.rollout_dir / "gelsight_validation.json"
        validation_path.write_text(
            json.dumps(validation, indent=2, allow_nan=False), encoding="utf-8"
        )

        provenance = {
            "sensor": "GelSight Mini mounted body already present in Cell A",
            "main_rollout_model_mutated": True,
            "main_rollout_model_mutation_scope": (
                "only 294 massless collidable hemispherical verification bumps "
                "on red_block; policy, checkpoint, controller, robot, cameras, "
                "joints, actuators, cube core mass, and cube core inertia unchanged"
            ),
            "passive_auxiliary_model_state_layout_verified": True,
            "acquisition_timing": "live post-step qpos/qvel copied after each main-model mj_step; passive mj_forward sampled at 30 Hz",
            "native_quantitative_channels": {
                "depth": "MuJoCo <sensor><tactile> maximum geometric penetration depth at each mesh vertex",
                "tangent_x_speed": "MuJoCo native absolute relative speed along sampling-mesh tangent 1",
                "tangent_z_speed": "MuJoCo native absolute relative speed along sampling-mesh tangent 2",
                "force": "mujoco.mj_contactForce from the state-identical passive model at the same post-step qpos/qvel/act/ctrl",
                "force_histogram": "synchronized MuJoCo solver contacts binned to nearest taxel; no spatial smoothing",
            },
            "taxel_shape_rows_z_columns_x": [GELSIGHT_TAXELS_Z, GELSIGHT_TAXELS_X],
            "taxel_orientation": "rows run local +Z to -Z; columns run local -X to +X",
            "optical_rgb": {
                "quantitative": False,
                "method": "Taxim polynomial reflectance calibration plus Taxim-style multi-scale elastomer relaxation",
                "input": "only MuJoCo native depth; never proximity, policy state, or a drawn object mask",
                "storage": "derived deformation is not duplicated in NPZ; regenerate it from native_depth_m and the embedded calibration",
                "background_derivative_sha256": _TAXIM_BACKGROUND_SHA256,
                "polycalib_float32_derivative_sha256": _TAXIM_POLYCALIB_SHA256,
                "source_repository": "https://github.com/Robo-Touch/Taxim",
                "source_commit": _TAXIM_SOURCE_COMMIT,
                "source_dataPack_sha256": _TAXIM_SOURCE_DATA_PACK_SHA256,
                "source_polycalib_sha256": _TAXIM_SOURCE_POLYCALIB_SHA256,
                "license": _TAXIM_MIT_LICENSE,
            },
            "mujoco_references": [
                "https://mujoco.readthedocs.io/en/latest/XMLreference.html#sensor-tactile",
                "https://github.com/google-deepmind/mujoco/blob/main/model/tactile/tactile.xml",
            ],
            "fabricated_proximity_signal": False,
            "gaussian_force_blob": False,
            "random_tactile_noise": False,
            "cube_surface_pattern_added": True,
            "cube_surface_pattern": {
                "physical_not_image_overlay": True,
                "geom_prefix": CUBE_SURFACE_TEXTURE["geom_prefix"],
                "faces": CUBE_SURFACE_TEXTURE["face_names"],
                "same_pattern_and_count_on_every_face": True,
                "grid_size_per_face": CUBE_SURFACE_TEXTURE["grid_size_per_face"],
                "grid_coordinates_m": CUBE_SURFACE_TEXTURE["grid_coordinates_m"],
                "face_counts": CUBE_SURFACE_TEXTURE["face_counts"],
                "total_bump_geoms": CUBE_SURFACE_TEXTURE["total_bump_geoms"],
                "sphere_radius_m": CUBE_SURFACE_TEXTURE["bump_radius_m"],
                "hemisphere_protrusion_m": CUBE_SURFACE_TEXTURE["hemisphere_protrusion_m"],
                "sphere_centers_on_core_box_faces": True,
                "same_red_material_as_core": True,
                "collision_enabled": True,
                "added_inertial_mass_kg": CUBE_SURFACE_TEXTURE["added_inertial_mass_kg"],
                "cube_body_mass_kg_before": CUBE_SURFACE_TEXTURE["cube_body_mass_kg_before"],
                "cube_body_mass_kg_after": CUBE_SURFACE_TEXTURE["cube_body_mass_kg_after"],
                "layout_and_contact_invariants_verified": True,
            },
            "textured_main_xml": str(CUBE_SURFACE_TEXTURE["textured_xml"]),
            "textured_main_xml_sha256": CUBE_SURFACE_TEXTURE["textured_xml_sha256"],
            "pretexture_main_xml": str(CUBE_SURFACE_TEXTURE["source_xml"]),
            "pretexture_main_xml_sha256": CUBE_SURFACE_TEXTURE["source_xml_sha256"],
            "sensor_xml": str(GELSIGHT_ACQUISITION["sensor_xml"]),
            "sensor_xml_sha256": GELSIGHT_ACQUISITION["sensor_xml_sha256"],
            "flattened_main_xml_sha256": GELSIGHT_ACQUISITION["flattened_xml_sha256"],
        }
        provenance_path = self.rollout_dir / "gelsight_provenance.json"
        provenance_path.write_text(
            json.dumps(provenance, indent=2, allow_nan=False), encoding="utf-8"
        )

        self.finalized_outputs = {
            "rgb_video": str(self.rgb_video_path),
            "audit_video": str(self.audit_video_path),
            "quantitative_npz": str(raw_data_path),
            "live_contacts_jsonl": str(contacts_path),
            "validation_json": str(validation_path),
            "provenance_json": str(provenance_path),
            "validation": validation,
        }
        return self.finalized_outputs


print("Passive GelSight acquisition model ready:")
print(
    f"  native grid={GELSIGHT_TAXELS_Z}x{GELSIGHT_TAXELS_X}, "
    f"RGB={GELSIGHT_RGB_WIDTH}x{GELSIGHT_RGB_HEIGHT}@{GELSIGHT_TACTILE_FPS} Hz"
)
print("  main model change: requested massless physical cube microtexture only")
print("  synchronized passive solver contacts include core and bump geoms")
print("  sensor XML:", GELSIGHT_ACQUISITION["sensor_xml"])


worker_code = f"""
import json, sys, time
from pathlib import Path
from types import SimpleNamespace

sys.argv.append("panda_d3")
sys.path.insert(0, {json.dumps(str(OFT_REPO))})

import numpy as np
import torch
from PIL import Image
from experiments.robot.openvla_utils import (
    get_action_head,
    get_processor,
    get_proprio_projector,
    get_vla,
    get_vla_action,
)
from prismatic.vla.constants import ACTION_DIM, NUM_ACTIONS_CHUNK, PROPRIO_DIM

checkpoint_dir = Path({json.dumps(str(checkpoint_dir))})
dataset_name = {json.dumps(DATASET_NAME)}
assert (NUM_ACTIONS_CHUNK, ACTION_DIM, PROPRIO_DIM) == (8, 7, 8)
assert torch.cuda.is_available(), "CUDA is required for OpenVLA-OFT rollout inference."

cfg = SimpleNamespace(
    pretrained_checkpoint=str(checkpoint_dir),
    use_l1_regression=True,
    use_diffusion=False,
    use_film=False,
    num_images_in_input=1,
    use_proprio=True,
    load_in_8bit=False,
    load_in_4bit=False,
    center_crop=True,
    lora_rank=32,
    unnorm_key=dataset_name,
    num_diffusion_steps_train=50,
    num_diffusion_steps_inference=50,
)

print("OPENVLA_OFT_WORKER_LOADING", str(checkpoint_dir), flush=True)
vla = get_vla(cfg)
processor = get_processor(cfg)
action_head = get_action_head(cfg, vla.llm_dim)
proprio_projector = get_proprio_projector(cfg, vla.llm_dim, PROPRIO_DIM)
print("OPENVLA_OFT_WORKER_READY", flush=True)

for line in sys.stdin:
    try:
        req = json.loads(line)
        image = np.asarray(Image.open(req["image_path"]).convert("RGB"))           # model input
        proprio = np.asarray(req["proprio"], dtype=np.float32).reshape(-1)
        if proprio.shape != (PROPRIO_DIM,) or not np.all(np.isfinite(proprio)):
            raise ValueError(f"Invalid proprio: {{proprio}}")
        observation = {{"full_image": image, "state": proprio.copy()}}
        start = time.perf_counter()
        actions = np.asarray(
            get_vla_action(
                cfg,
                vla,
                processor,
                observation,
                req["task"],
                action_head,
                proprio_projector,
            ),
            dtype=np.float32,
        )
        if actions.shape != (NUM_ACTIONS_CHUNK, ACTION_DIM):
            raise ValueError(f"Expected 8x7 action chunk, got {{actions.shape}}")
        if not np.all(np.isfinite(actions)):
            raise ValueError("OFT action chunk contains NaN or infinity")
        print("OPENVLA_OFT_WORKER_RESULT " + json.dumps({{
            "id": req.get("id"),
            "actions": actions.tolist(),
            "inference_s": time.perf_counter() - start,
        }}), flush=True)
    except Exception as exc:
        print("OPENVLA_OFT_WORKER_ERROR " + json.dumps({{"error": repr(exc)}}), flush=True)
"""

def start_worker():
    env = oft_env({"TF_CPP_MIN_LOG_LEVEL": "3", "WANDB_MODE": "disabled"})
    proc = subprocess.Popen(
        [str(OFT_PYTHON), "-u", "-c", worker_code],
        cwd=str(OFT_REPO),
        env=env,
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    tail = []
    start = time.time()
    while time.time() - start < 900:
        ready, _, _ = select.select([proc.stdout], [], [], 0.5)
        if ready:
            line = proc.stdout.readline()
            if line:
                tail.append(line.rstrip())
                print(line, end="")
                if line.startswith("OPENVLA_OFT_WORKER_READY"):
                    return proc, tail
        if proc.poll() is not None:
            raise RuntimeError("OpenVLA worker exited early:\n" + "\n".join(tail[-40:]))
    raise TimeoutError("Timed out while loading OpenVLA worker.")

def predict_action_chunk(proc, image_path, proprio, request_id):
    proprio = np.asarray(proprio, dtype=np.float32).reshape(-1)
    assert proprio.shape == (8,) and np.all(np.isfinite(proprio))
    proc.stdin.write(json.dumps({
        "id": int(request_id),
        "image_path": str(image_path),
        "task": TASK_PROMPT,
        "proprio": proprio.tolist(),
    }) + "\n")
    proc.stdin.flush()

    tail = []
    start = time.time()
    while time.time() - start < 600:
        ready, _, _ = select.select([proc.stdout], [], [], 0.5)
        if ready:
            line = proc.stdout.readline()
            if not line:
                continue
            tail.append(line.rstrip())
            if line.startswith("OPENVLA_OFT_WORKER_RESULT "):
                payload = json.loads(line.split(" ", 1)[1])
                actions = np.asarray(payload["actions"], dtype=np.float32)
                assert actions.shape == (8, 7), actions.shape
                assert np.all(np.isfinite(actions))
                return actions, payload
            if line.startswith("OPENVLA_OFT_WORKER_ERROR "):
                raise RuntimeError(line)
        if proc.poll() is not None:
            raise RuntimeError("OpenVLA-OFT worker exited during prediction:\n" + "\n".join(tail[-40:]))
    raise TimeoutError("Timed out waiting for OpenVLA-OFT action chunk.")

def render_rgb(renderer, camera):
    renderer.update_scene(data, camera=camera)
    return renderer.render()

def set_arm_qpos(qpos_dict):
    for name, value in qpos_dict.items():
        if name in ARM_JOINT_NAMES:
            data.qpos[ARM_QPOS_ADR[ARM_JOINT_NAMES.index(name)]] = float(value)
    set_arm_position_targets(qpos_dict)
    mujoco.mj_forward(model, data)

def randomize_start(rng, rollout_idx):
    nominal_core_cube_pos = OOD_CUBE_STARTS[
        rollout_idx % len(OOD_CUBE_STARTS)
    ].copy()
    assert nominal_core_cube_pos[0] < 0.47 or nominal_core_cube_pos[0] > 0.53
    assert nominal_core_cube_pos[1] < -0.04 or nominal_core_cube_pos[1] > 0.04

    # Bottom-face hemisphere tips, rather than the hidden core box face, begin
    # exactly on the tabletop. This avoids an initialization overlap equal to
    # the bump radius.
    cube_pos = nominal_core_cube_pos.copy()
    cube_pos[2] += CUBE_TEXTURE_BUMP_RADIUS_M

    reset_task_state_randomized(cube_pos=cube_pos, robot_joint_noise=None, settle_steps=60)
    nominal_ee = get_ee_pose()["position"].copy()
    direction = rng.normal(size=3)
    direction /= max(np.linalg.norm(direction), 1e-12)
    ee_delta = direction * OOD_EE_JITTER_M
    ee_target = np.clip(nominal_ee + ee_delta, WORKSPACE_LOW, WORKSPACE_HIGH)

    qpos_target = solve_hand_position_ik(ee_target, seed_qpos=current_arm_qpos_dict())
    set_arm_qpos(qpos_target)
    set_gripper_opening(0.04)
    for _ in range(60):
        set_arm_position_targets(qpos_target)
        set_gripper_opening(0.04)
        mujoco.mj_step(model, data)
    mujoco.mj_forward(model, data)

    return {
        "cube_start": cube_pos.tolist(),
        "nominal_untextured_cube_start": nominal_core_cube_pos.tolist(),
        "texture_bottom_tip_initially_on_table": True,
        "robot_ee_start_delta_m": (get_ee_pose()["position"] - nominal_ee).tolist(),
        "outside_d3_cube_xy_envelope": True,
    }

def current_oft_proprio():
    joint_qpos = np.asarray(data.qpos[ARM_QPOS_ADR], dtype=np.float32)
    gripper_fraction = np.float32(np.clip(current_finger_opening_m() / 0.04, 0.0, 1.0))
    proprio = np.concatenate([joint_qpos, [gripper_fraction]]).astype(np.float32)
    assert proprio.shape == (8,) and np.all(np.isfinite(proprio))
    return proprio

CONTROLLER_STATE = {
    "mode": "open",
    "profile_step": 0,
    "applied_gripper_fraction": 1.0,
    "hold_position": None,
    "hold_quat": None,
    "reference_position": None,
    "reference_quat": None,
}


def reset_controller_state():
    initial_pose = get_ee_pose()
    CONTROLLER_STATE.update(
        mode="open",
        profile_step=0,
        applied_gripper_fraction=1.0,
        hold_position=None,
        hold_quat=None,
        reference_position=initial_pose["position"].copy(),
        reference_quat=initial_pose["quat_wxyz"].copy(),
        filtered_carry_translation=np.zeros(3, dtype=np.float64),
        filtered_carry_rotation=np.zeros(3, dtype=np.float64),
        last_wrist_alignment=None,
    )


def smoothstep01(value):
    value = float(np.clip(value, 0.0, 1.0))
    return value * value * (3.0 - 2.0 * value)


def controller_gripper_fraction(raw_fraction):
    """Update grasp/release phase bookkeeping from the policy gripper output."""
    state = CONTROLLER_STATE
    raw_fraction = float(np.clip(raw_fraction, 0.0, 1.0))

    if state["mode"] == "open" and raw_fraction < GRIPPER_CLOSE_TRIGGER:
        pose = get_ee_pose()
        state.update(
            mode="closing",
            profile_step=0,
            hold_position=pose["position"].copy(),
            hold_quat=pose["quat_wxyz"].copy(),
        )
    elif (
        state["mode"] in {"closed_hold", "lift_bootstrap", "carrying"}
        and raw_fraction > GRIPPER_OPEN_TRIGGER
    ):
        pose = get_ee_pose()
        release_position = pose["position"].copy()
        release_position[2] = min(
            release_position[2] + RELEASE_CLEARANCE_M,
            WORKSPACE_HIGH[2],
        )
        state.update(
            mode="opening",
            profile_step=0,
            hold_position=release_position,
            hold_quat=pose["quat_wxyz"].copy(),
        )

    if state["mode"] == "closing":
        state["profile_step"] += 1
        alpha = smoothstep01(
            state["profile_step"] / GRIPPER_CLOSE_PROFILE_STEPS
        )
        state["applied_gripper_fraction"] = 1.0 - alpha
        if state["profile_step"] >= GRIPPER_CLOSE_PROFILE_STEPS:
            state["mode"] = "closed_hold"
    elif state["mode"] == "opening":
        state["profile_step"] += 1
        alpha = smoothstep01(
            state["profile_step"] / GRIPPER_OPEN_PROFILE_STEPS
        )
        state["applied_gripper_fraction"] = alpha
        if state["profile_step"] >= GRIPPER_OPEN_PROFILE_STEPS:
            state["mode"] = "released_hold"
    elif state["mode"] in {"closed_hold", "lift_bootstrap", "carrying"}:
        state["applied_gripper_fraction"] = 0.0
    else:
        state["applied_gripper_fraction"] = 1.0

    return float(state["applied_gripper_fraction"])


def select_controller_action(action_chunk):
    """Execute only horizon 0; later horizons are predictions, not commands."""
    action_chunk = np.asarray(action_chunk, dtype=np.float64)
    assert action_chunk.shape == (8, 7)
    state = CONTROLLER_STATE
    if (
        state["mode"] == "lift_bootstrap"
        and state["hold_position"] is not None
        and get_ee_pose()["position"][2]
        >= state["hold_position"][2] + LIFT_BOOTSTRAP_HEIGHT_M - 0.005
    ):
        state["mode"] = "carrying"
    action = action_chunk[0].copy()
    return action, 0, float(action[6])


def wrist_visual_alignment_correction(current_pose):
    """Return an image-based XY correction without reading simulator object state."""
    inactive = {
        "active": False,
        "centroid_px": None,
        "error_px": None,
        "correction_xy_m": [0.0, 0.0],
    }
    if (
        not WRIST_VISUAL_SERVO_ENABLED
        or CONTROLLER_STATE["mode"] != "open"
        or current_pose["position"][2] > WRIST_VISUAL_SERVO_START_Z_M
        or current_pose["position"][2] < WRIST_VISUAL_SERVO_STOP_Z_M
    ):
        return np.zeros(2, dtype=np.float64), inactive

    rgb = render_rgb(wrist_alignment_renderer, AGENT_CAMERA_NAME)
    red = rgb[:, :, 0].astype(np.float64)
    green = rgb[:, :, 1].astype(np.float64)
    blue = rgb[:, :, 2].astype(np.float64)
    mask = (red > 140.0) & (red > 1.7 * green) & (red > 1.7 * blue)
    rows, cols = np.nonzero(mask)
    if len(cols) < 50:
        missing = dict(inactive)
        missing["reason"] = "red_target_not_found"
        return np.zeros(2, dtype=np.float64), missing

    centroid = np.array([cols.mean(), rows.mean()], dtype=np.float64)
    camera_rotation = data.cam_xmat[camera_id].reshape(3, 3).copy()
    camera_position = data.cam_xpos[camera_id].copy()
    image_height, image_width = rgb.shape[:2]
    focal_px = 0.5 * image_height / np.tan(
        np.deg2rad(model.cam_fovy[camera_id]) / 2.0
    )
    principal = np.array([0.5 * image_width, 0.5 * image_height])
    ray_camera = np.array([
        (centroid[0] - principal[0]) / focal_px,
        (principal[1] - centroid[1]) / focal_px,
        -1.0,
    ])
    ray_world = camera_rotation @ ray_camera
    if abs(ray_world[2]) < 1e-6:
        return np.zeros(2, dtype=np.float64), inactive
    distance = (WRIST_ALIGNMENT_PLANE_Z_M - camera_position[2]) / ray_world[2]
    estimated_point = camera_position + distance * ray_world

    desired_grasp_xy = estimated_point[:2] + WRIST_ESTIMATE_TO_GRASP_OFFSET_M
    world_error = desired_grasp_xy - current_pose["position"][:2]
    world_error_norm = float(np.linalg.norm(world_error))
    correction = WRIST_VISUAL_SERVO_GAIN * world_error
    if world_error_norm <= WRIST_ALIGNMENT_TOLERANCE_M:
        correction[:] = 0.0
    correction_norm = float(np.linalg.norm(correction))
    if correction_norm > WRIST_VISUAL_SERVO_MAX_STEP_M:
        correction *= WRIST_VISUAL_SERVO_MAX_STEP_M / correction_norm

    info = {
        "active": True,
        "centroid_px": centroid.tolist(),
        "correction_xy_m": correction.tolist(),
        "estimated_target_xy_m": estimated_point[:2].tolist(),
        "desired_grasp_xy_m": desired_grasp_xy.tolist(),
        "world_error_xy_m": world_error.tolist(),
        "world_error_norm_m": world_error_norm,
        "reason": (
            "within_tolerance"
            if world_error_norm <= WRIST_ALIGNMENT_TOLERANCE_M
            else "correcting"
        ),
    }
    return correction, info


def adapt_openvla_action(action):
    """Convert one policy delta into a smooth, contact-conscious 10 Hz pose target."""
    raw = np.asarray(action, dtype=np.float64).reshape(-1)
    assert raw.size == 7, f"Expected 7D OpenVLA action, got {raw}"

    clipped = raw.copy()
    clipped[:3] = np.clip(
        clipped[:3], -OPENVLA_MAX_TRANSLATION_M, OPENVLA_MAX_TRANSLATION_M
    )
    rotation_norm = float(np.linalg.norm(clipped[3:6]))
    if rotation_norm > OPENVLA_MAX_ROTATION_RAD:
        clipped[3:6] *= OPENVLA_MAX_ROTATION_RAD / rotation_norm
    clipped[6] = float(np.clip(clipped[6], 0.0, 1.0))

    state = CONTROLLER_STATE
    if state["mode"] == "closed_hold" and clipped[2] > LIFT_LOOKAHEAD_MIN_DZ_M:
        state["mode"] = "lift_bootstrap"

    current_pose = get_ee_pose()
    mode = state["mode"]
    translation_gain = (
        CARRY_TRANSLATION_GAIN if mode == "carrying" else APPROACH_TRANSLATION_GAIN
    )
    translation_delta = clipped[:3] * translation_gain
    rotation_delta = clipped[3:6] * ROTATION_GAIN

    visual_correction, visual_info = wrist_visual_alignment_correction(current_pose)
    state["last_wrist_alignment"] = visual_info
    if mode == "open" and visual_info["active"]:
        translation_delta[:2] += visual_correction
        if visual_info.get("world_error_norm_m", 0.0) > WRIST_ALIGNMENT_TOLERANCE_M:
            translation_delta[2] = max(
                translation_delta[2], -FINE_APPROACH_MAX_DESCENT_STEP_M
            )
    elif (
        WRIST_VISUAL_SERVO_ENABLED
        and mode == "open"
        and current_pose["position"][2] < WRIST_VISUAL_SERVO_STOP_Z_M
    ):
        # Keep the visually aligned XY target during the final straight descent.
        # The close-range mask is partially occluded by the fingers here.
        translation_delta[:2] = 0.0

    premature_close = (
        mode == "open"
        and clipped[6] < GRIPPER_CLOSE_TRIGGER
        and current_pose["position"][2] > MAX_VALID_GRASP_CLOSE_Z_M
    )
    if premature_close:
        clipped[6] = 1.0
        translation_delta[2] = min(
            translation_delta[2], -FINE_APPROACH_MAX_DESCENT_STEP_M
        )

    # Below the pregrasp clearance, retain the policy direction but limit lateral
    # sweeping and descent speed. This is a velocity limit, not a visual waypoint.
    fine_approach = mode == "open" and current_pose["position"][2] < FINE_APPROACH_START_Z_M
    if fine_approach:
        translation_delta[:2] = np.clip(
            translation_delta[:2],
            -FINE_APPROACH_MAX_XY_STEP_M,
            FINE_APPROACH_MAX_XY_STEP_M,
        )
        translation_delta[2] = max(
            translation_delta[2], -FINE_APPROACH_MAX_DESCENT_STEP_M
        )

    if mode == "carrying":
        previous_translation = state["filtered_carry_translation"]
        translation_delta = (
            CARRY_FILTER_ALPHA * translation_delta
            + (1.0 - CARRY_FILTER_ALPHA) * previous_translation
        )
        axis_limits = (
            PLACEMENT_AXIS_MAX_STEP_M
            if current_pose["position"][0] >= CARRY_DESCENT_START_X_M
            else CARRY_AXIS_MAX_STEP_M
        )
        translation_delta = np.clip(translation_delta, -axis_limits, axis_limits)
        state["filtered_carry_translation"] = translation_delta.copy()

        previous_rotation = state["filtered_carry_rotation"]
        rotation_delta = (
            CARRY_ROTATION_FILTER_ALPHA * rotation_delta
            + (1.0 - CARRY_ROTATION_FILTER_ALPHA) * previous_rotation
        )
        state["filtered_carry_rotation"] = rotation_delta.copy()
    else:
        state["filtered_carry_translation"][:] = 0.0
        state["filtered_carry_rotation"][:] = 0.0

    if mode == "closing" and state["hold_position"] is not None:
        # Let the physical fingers establish contact before lifting. The policy's
        # gripper output remains live, but pose deltas are deferred during closure.
        target_pos = state["hold_position"].copy()
        target_quat = state["hold_quat"].copy()
    else:
        target_pos = np.clip(
            state["reference_position"] + translation_delta,
            WORKSPACE_LOW,
            WORKSPACE_HIGH,
        )
        target_pos[2] = np.clip(
            current_pose["position"][2] + translation_delta[2],
            WORKSPACE_LOW[2],
            WORKSPACE_HIGH[2],
        )
        if mode == "carrying":
            target_pos[1] = np.clip(target_pos[1], *SAFE_CARRY_Y_BOUNDS_M)
        target_quat = quat_apply_world_delta(
            state["reference_quat"], rotation_delta
        )

    state["reference_position"] = target_pos.copy()
    state["reference_quat"] = target_quat.copy()
    qpos_target, ik_info = plan_pose_qpos(target_pos, target_quat)
    if not ik_info["converged"]:
        print("WARNING: 6D pose IK did not fully converge:", ik_info)

    mode_before_gripper_update = state["mode"]
    controller_gripper_fraction(clipped[6])
    if mode_before_gripper_update == "open" and state["mode"] == "closing":
        state["hold_position"] = target_pos.copy()
        state["hold_quat"] = target_quat.copy()

    applied_gripper_fraction = float(clipped[6])
    state["applied_gripper_fraction"] = applied_gripper_fraction
    gripper_opening = (
        OPENVLA_GRIPPER_CLOSED_M
        + applied_gripper_fraction
        * (OPENVLA_GRIPPER_OPENING_M - OPENVLA_GRIPPER_CLOSED_M)
    )
    return (
        raw,
        clipped,
        qpos_target,
        target_pos,
        target_quat,
        gripper_opening,
        applied_gripper_fraction,
        ik_info,
    )


def current_gripper_command_m():
    tendon_actuator_ids = [
        actuator_id
        for actuator_id in range(model.nu)
        if model.actuator_trntype[actuator_id] == mujoco.mjtTrn.mjTRN_TENDON
    ]
    assert tendon_actuator_ids, "No Panda gripper tendon actuator found."
    actuator_id = tendon_actuator_ids[0]
    return float(np.clip(data.ctrl[actuator_id] / 255.0 * 0.04, 0.0, 0.04))


def step_controller(
    qpos_target, target_pos, target_quat, gripper_opening, qpos_frames, tactile_recorder=None
):
    """Execute one 10 Hz target through MuJoCo's physical position actuators."""
    target_pos = np.asarray(target_pos, dtype=np.float64)
    target_quat = quat_normalize(target_quat)
    desired_qpos = np.asarray(
        [qpos_target[name] for name in ARM_JOINT_NAMES], dtype=np.float64
    )
    current_qpos = data.qpos[ARM_QPOS_ADR].copy()
    start_arm_qpos = current_qpos.copy()
    target_values = current_qpos + ACTUATOR_LEAD * (desired_qpos - current_qpos)
    target_values = np.asarray([
        np.clip(value, *model.jnt_range[joint_id])
        for value, joint_id in zip(target_values, ARM_JOINT_IDS)
    ], dtype=np.float64)
    steps = max(1, int(round(CONTROL_HORIZON_S / model.opt.timestep)))
    frame_every = max(1, int(round((1.0 / VIDEO_FPS) / model.opt.timestep)))
    start_gripper_command = current_gripper_command_m()
    max_position_error_m = 0.0
    max_orientation_error_rad = 0.0

    for i in range(steps):
        alpha = smoothstep01((i + 1) / steps)
        gripper_command = (
            (1.0 - alpha) * start_gripper_command
            + alpha * gripper_opening
        )
        interpolated_arm_command = {
            name: float(
                (1.0 - alpha) * start_arm_qpos[index]
                + alpha * target_values[index]
            )
            for index, name in enumerate(ARM_JOINT_NAMES)
        }
        set_arm_position_targets(interpolated_arm_command)
        set_gripper_opening(gripper_command)
        mujoco.mj_step(model, data)
        if tactile_recorder is not None:
            tactile_recorder.after_physics_step()

        pose = get_ee_pose()
        max_position_error_m = max(
            max_position_error_m,
            float(np.linalg.norm(target_pos - pose["position"])),
        )
        max_orientation_error_rad = max(
            max_orientation_error_rad,
            float(np.linalg.norm(quat_error_world(
                target_quat, pose["quat_wxyz"]
            ))),
        )
        if (i + 1) % frame_every == 0 or i == steps - 1:
            qpos_frames.append(data.qpos.copy())

    final_pose = get_ee_pose()
    endpoint_position_error_m = float(np.linalg.norm(
        target_pos - final_pose["position"]
    ))
    endpoint_orientation_error_rad = float(np.linalg.norm(
        quat_error_world(target_quat, final_pose["quat_wxyz"])
    ))
    return {
        "mode": "mujoco_smoothed_position_actuators_no_state_projection",
        "physics_steps": int(steps),
        "feedback_projection_steps": 0,
        "reference_ik_failures": 0,
        "max_pre_correction_position_error_m": max_position_error_m,
        "max_pre_correction_orientation_error_rad": max_orientation_error_rad,
        "endpoint_position_error_m": endpoint_position_error_m,
        "endpoint_orientation_error_rad": endpoint_orientation_error_rad,
        "endpoint_within_tolerance": bool(
            endpoint_position_error_m < 0.003
            and endpoint_orientation_error_rad < np.deg2rad(1.0)
        ),
    }


def settle_after_release(qpos_target, qpos_frames, tactile_recorder=None):
    """Keep the released pose stable long enough for the cube to settle."""
    steps = max(1, int(round(RELEASE_SETTLE_S / model.opt.timestep)))
    frame_every = max(1, int(round((1.0 / VIDEO_FPS) / model.opt.timestep)))
    for i in range(steps):
        set_arm_position_targets(qpos_target)
        set_gripper_opening(OPENVLA_GRIPPER_OPENING_M)
        mujoco.mj_step(model, data)
        if tactile_recorder is not None:
            tactile_recorder.after_physics_step()
        if (i + 1) % frame_every == 0 or i == steps - 1:
            qpos_frames.append(data.qpos.copy())


def make_grasp_diagnostic_cameras(cube_start):
    cube_start = np.asarray(cube_start, dtype=np.float64)
    grasp_lookat = cube_start + np.array([0.0, 0.0, 0.055], dtype=np.float64)

    # Head-on view of the finger/cube contact at the initial grasp location.
    grasp_front_cam = mujoco.MjvCamera()
    grasp_front_cam.distance = 0.58
    grasp_front_cam.azimuth = 180
    grasp_front_cam.elevation = -12
    grasp_front_cam.lookat[:] = grasp_lookat

    # Lower view from the opposite table corner, also fixed on the initial cube.
    grasp_corner_cam = mujoco.MjvCamera()
    grasp_corner_cam.distance = 0.72
    grasp_corner_cam.azimuth = 315
    grasp_corner_cam.elevation = -16
    grasp_corner_cam.lookat[:] = grasp_lookat

    return grasp_front_cam, grasp_corner_cam


def render_rollout_videos(
    qpos_frames,
    cube_start,
    world_path,
    wrist_path,
    grasp_front_path,
    grasp_corner_path,
):
    saved_qpos = data.qpos.copy()
    saved_qvel = data.qvel.copy()

    world_cam = make_third_person_camera() if "make_third_person_camera" in globals() else mujoco.MjvCamera()
    grasp_front_cam, grasp_corner_cam = make_grasp_diagnostic_cameras(cube_start)
    world_renderer = mujoco.Renderer(model, height=480, width=640)
    wrist_renderer = mujoco.Renderer(model, height=WRIST_IMAGE_SIZE, width=WRIST_IMAGE_SIZE)
    grasp_front_renderer = mujoco.Renderer(model, height=480, width=640)
    grasp_corner_renderer = mujoco.Renderer(model, height=480, width=640)

    world_frames, wrist_frames = [], []
    grasp_front_frames, grasp_corner_frames = [], []
    try:
        for qpos in qpos_frames:
            data.qpos[:] = qpos
            data.qvel[:] = 0.0
            mujoco.mj_forward(model, data)
            world_frames.append(render_rgb(world_renderer, world_cam))
            wrist_frames.append(render_rgb(wrist_renderer, AGENT_CAMERA_NAME))
            grasp_front_frames.append(render_rgb(grasp_front_renderer, grasp_front_cam))
            grasp_corner_frames.append(render_rgb(grasp_corner_renderer, grasp_corner_cam))
    finally:
        world_renderer.close()
        wrist_renderer.close()
        grasp_front_renderer.close()
        grasp_corner_renderer.close()
        data.qpos[:] = saved_qpos
        data.qvel[:] = saved_qvel
        mujoco.mj_forward(model, data)

    imageio.mimsave(world_path, world_frames, fps=VIDEO_FPS)
    imageio.mimsave(wrist_path, wrist_frames, fps=VIDEO_FPS)
    imageio.mimsave(grasp_front_path, grasp_front_frames, fps=VIDEO_FPS)
    imageio.mimsave(grasp_corner_path, grasp_corner_frames, fps=VIDEO_FPS)

    return {
        "grasp_front": {
            "distance": float(grasp_front_cam.distance),
            "azimuth": float(grasp_front_cam.azimuth),
            "elevation": float(grasp_front_cam.elevation),
            "lookat": grasp_front_cam.lookat.astype(float).tolist(),
        },
        "grasp_corner_low": {
            "distance": float(grasp_corner_cam.distance),
            "azimuth": float(grasp_corner_cam.azimuth),
            "elevation": float(grasp_corner_cam.elevation),
            "lookat": grasp_corner_cam.lookat.astype(float).tolist(),
        },
    }

worker = None
rng = np.random.default_rng(ROLLOUT_SEED)
all_logs = []

## Start the model worker and each rollout
try:
    worker, worker_tail = start_worker()

    rollout_corner_indices = (
        SELECTED_OOD_START_INDICES if SELECTED_OOD_START_INDICES is not None
        else list(range(N_ROLLOUTS))
    )
    for rollout_idx, corner_idx in enumerate(rollout_corner_indices):
        rollout_dir = run_dir / f"rollout_{rollout_idx:03d}"
        rollout_dir.mkdir(parents=True, exist_ok=True)

        start_info = randomize_start(rng, corner_idx)
        reset_controller_state()
        qpos_frames = [data.qpos.copy()]
        tactile_recorder = GelSightRolloutRecorder(rollout_dir)
        tactile_recorder.capture_initial()
        tactile_outputs = None
        records = []
        rollout_termination_reason = "maximum_action_budget"

        policy_camera = make_third_person_camera()
        policy_renderer = mujoco.Renderer(
            model,
            height=POLICY_IMAGE_HEIGHT,
            width=POLICY_IMAGE_WIDTH,
        )
        wrist_alignment_renderer = mujoco.Renderer(
            model, height=WRIST_IMAGE_SIZE, width=WRIST_IMAGE_SIZE
        )

        try:
            executed_action_count = 0
            query_idx = 0
            policy_queries = []
            while executed_action_count < MAX_EXECUTED_ACTIONS:
                policy_rgb = render_rgb(policy_renderer, policy_camera)
                image_path = rollout_dir / f"query_{query_idx:03d}_third_person.png"
                Image.fromarray(policy_rgb).save(image_path)

                query_proprio = current_oft_proprio()
                action_chunk, pred_info = predict_action_chunk(
                    worker,
                    image_path,
                    query_proprio,
                    request_id=query_idx,
                )
                model_h0_action = action_chunk[0].copy()
                (
                    controller_action,
                    selected_first_horizon,
                    gripper_lookahead_fraction,
                ) = select_controller_action(action_chunk)
                controller_phase_before = CONTROLLER_STATE["mode"]
                steps_this_query = min(
                    ACTIONS_PER_CHUNK,
                    MAX_EXECUTED_ACTIONS - executed_action_count,
                )
                query_record = {
                    "query": int(query_idx),
                    "image": str(image_path),
                    "proprio_input": query_proprio.tolist(),
                    "predicted_action_chunk": action_chunk.tolist(),
                    "model_h0_action": model_h0_action.tolist(),
                    "controller_source_action": controller_action.tolist(),
                    "selected_first_horizon": int(selected_first_horizon),
                    "gripper_lookahead_fraction": float(gripper_lookahead_fraction),
                    "executed_chunk_actions": int(steps_this_query),
                    "inference_s": float(pred_info["inference_s"]),
                }

                for chunk_step in range(steps_this_query):
                    measured_proprio_before = current_oft_proprio()
                    (
                        controller_source_raw,
                        clipped,
                        qpos_target,
                        ee_target,
                        ee_target_quat,
                        gripper_opening,
                        applied_gripper_fraction,
                        ik_info,
                    ) = adapt_openvla_action(controller_action)
                    controller_phase_after = CONTROLLER_STATE["mode"]
                    gripper_before_m = current_finger_opening_m()
                    tactile_frame_start = tactile_recorder.frame_count
                    controller_info = step_controller(
                        qpos_target,
                        ee_target,
                        ee_target_quat,
                        gripper_opening,
                        qpos_frames,
                        tactile_recorder=tactile_recorder,
                    )
                    gripper_after_m = current_finger_opening_m()
                    finger_qpos_after_m = np.asarray(
                        data.qpos[FINGER_QPOS_ADR], dtype=np.float64
                    ).copy()
                    measured_proprio_after = current_oft_proprio()

                    ee_pose_after = get_ee_pose()
                    ee_pos = ee_pose_after["position"].copy()
                    orientation_error_deg = float(np.rad2deg(np.linalg.norm(
                        quat_error_world(ee_target_quat, ee_pose_after["quat_wxyz"])
                    )))
                    cube_pos = get_cube_pose()["position"].copy()
                    rec = {
                        "action_step": int(executed_action_count),
                        "query": int(query_idx),
                        "chunk_step": int(chunk_step),
                        "selected_chunk_horizon": int(selected_first_horizon),
                        "image": str(image_path),
                        "camera_view": POLICY_CAMERA_VIEW,
                        "source": "fine_tuned_openvla_oft_horizon_0_replanning",
                        "query_proprio_input": query_proprio.tolist(),
                        "measured_proprio_before_action": measured_proprio_before.tolist(),
                        "measured_proprio_after_action": measured_proprio_after.tolist(),
                        "raw_openvla_action": model_h0_action.tolist(),
                        "controller_source_action": controller_source_raw.tolist(),
                        "clipped_controller_action": clipped.tolist(),
                        "wrist_alignment": CONTROLLER_STATE.get("last_wrist_alignment"),
                        "controller_phase_before_action": controller_phase_before,
                        "controller_phase_after_planning": controller_phase_after,
                        "gripper_lookahead_fraction": float(gripper_lookahead_fraction),
                        "ee_target": ee_target.tolist(),
                        "ee_target_quat_wxyz": ee_target_quat.tolist(),
                        "ee_after": ee_pos.tolist(),
                        "ee_after_quat_wxyz": ee_pose_after["quat_wxyz"].tolist(),
                        "rotation_command_deg": np.rad2deg(clipped[3:6]).tolist(),
                        "orientation_tracking_error_deg": orientation_error_deg,
                        "pose_ik": ik_info,
                        "cartesian_controller": controller_info,
                        "cube_after_eval_only": cube_pos.tolist(),
                        "gripper_opening_m": float(gripper_opening),
                        "gripper_target_open_fraction": float(applied_gripper_fraction),
                        "gripper_controller_source_open_fraction": float(clipped[6]),
                        "gripper_target_per_finger_m": float(gripper_opening),
                        "gripper_before_per_finger_m": float(gripper_before_m),
                        "gripper_after_per_finger_m": float(gripper_after_m),
                        "gripper_after_total_width_m": float(2.0 * gripper_after_m),
                        "finger_qpos_after_m": finger_qpos_after_m.tolist(),
                        "tactile_frame_start": int(tactile_frame_start),
                        "tactile_frame_end_exclusive": int(tactile_recorder.frame_count),
                    }
                    records.append(rec)
                    print(
                        f"[rollout {rollout_idx} q{query_idx:02d} h0] "
                        f"step={executed_action_count:03d} "
                        f"phase={controller_phase_before}->{controller_phase_after} "
                        f"selected_h={selected_first_horizon} "
                        f"model_h0={np.round(model_h0_action, 4).tolist()} "
                        f"control={np.round(clipped, 4).tolist()} "
                        f"position_error_mm={controller_info['endpoint_position_error_m'] * 1000.0:.3f} "
                        f"ee={np.round(ee_pos, 3).tolist()} "
                        f"cube={np.round(cube_pos, 3).tolist()}"
                    )
                    executed_action_count += 1

                policy_queries.append(query_record)
                query_idx += 1
                if CONTROLLER_STATE["mode"] == "released_hold":
                    settle_after_release(
                        qpos_target, qpos_frames, tactile_recorder=tactile_recorder
                    )
                    rollout_termination_reason = "release_profile_complete"
                    break
        finally:
            policy_renderer.close()
            wrist_alignment_renderer.close()
            tactile_outputs = tactile_recorder.finalize(
                incomplete=sys.exc_info()[0] is not None
            )

        final_cube = get_cube_pose()["position"].copy()
        success = bool(
            final_cube[0] >= PLACE_BLOCK_POS[0] - 0.06
            and abs(final_cube[1] - PLACE_BLOCK_POS[1]) <= 0.08
            and 0.06 <= final_cube[2] <= 0.14
        )

        world_video = rollout_dir / "world.mp4"
        wrist_video = rollout_dir / "wrist.mp4"
        grasp_front_video = rollout_dir / "grasp_front.mp4"
        grasp_corner_video = rollout_dir / "grasp_corner_low.mp4"
        diagnostic_camera_config = render_rollout_videos(
            qpos_frames,
            start_info["cube_start"],
            world_video,
            wrist_video,
            grasp_front_video,
            grasp_corner_video,
        )

        payload = {
            "checkpoint": str(checkpoint_dir),
            "prompt": OPENVLA_ROLLOUT_PROMPT,
            "closed_loop_note": (
                "No TFDS/expert actions are used. OpenVLA-OFT receives live image, "
                "language, and 8D proprio, then replans after one horizon-0 action. "
                "Horizons 1-7 are diagnostics only. Arm and gripper targets are "
                "smoothly applied through MuJoCo actuators without state projection."
            ),
            "action_chunk_length": 8,
            "actions_executed_per_query": int(ACTIONS_PER_CHUNK),
            "max_executed_actions": int(MAX_EXECUTED_ACTIONS),
            "rollout_termination_reason": rollout_termination_reason,
            "proprio_convention": "[q1, q2, q3, q4, q5, q6, q7, measured_gripper_open_fraction]",
            "controller_mode": "hybrid_delta_reference_physical_actuators",
            "rotation_action_convention": "action[3:6] is a world-frame rotation vector in radians",
            "gripper_action_convention": (
                "continuous policy open fraction applied directly and interpolated within "
                "each 100 ms interval; 0=closed, 1=open; per-finger target is 0.00-0.04 m"
            ),
            "controller_parameters": {
                "approach_translation_gain": APPROACH_TRANSLATION_GAIN.tolist(),
                "carry_translation_gain": CARRY_TRANSLATION_GAIN.tolist(),
                "carry_max_translation_m": float(CARRY_MAX_TRANSLATION_M),
                "rotation_gain": float(ROTATION_GAIN),
                "actuator_lead": float(ACTUATOR_LEAD),
                "vertical_reference": "live measured end-effector z",
                "horizontal_reference": "persistent commanded x/y",
                "gripper_close_trigger": float(GRIPPER_CLOSE_TRIGGER),
                "gripper_open_trigger": float(GRIPPER_OPEN_TRIGGER),
                "lift_bootstrap_height_m": float(LIFT_BOOTSTRAP_HEIGHT_M),
                "release_clearance_m": float(RELEASE_CLEARANCE_M),
            },
            "max_rotation_per_action_deg": float(np.rad2deg(OPENVLA_MAX_ROTATION_RAD)),
            "policy_camera": {
                "view": POLICY_CAMERA_VIEW,
                "width": int(POLICY_IMAGE_WIDTH),
                "height": int(POLICY_IMAGE_HEIGHT),
                "distance": float(policy_camera.distance),
                "azimuth": float(policy_camera.azimuth),
                "elevation": float(policy_camera.elevation),
                "lookat": policy_camera.lookat.astype(float).tolist(),
            },
            "diagnostic_cameras": diagnostic_camera_config,
            "gelsight_tactile": tactile_outputs,
            "start_info": start_info,
            "success_rough_eval_only": success,
            "final_cube_pos": final_cube.tolist(),
            "policy_queries": policy_queries,
            "records": records,
            "videos": {
                "world": str(world_video),
                "wrist": str(wrist_video),
                "grasp_front": str(grasp_front_video),
                "grasp_corner_low": str(grasp_corner_video),
                "gelsight_rgb": tactile_outputs["rgb_video"],
                "gelsight_audit": tactile_outputs["audit_video"],
            },
        }
        log_path = rollout_dir / "closed_loop_log.json"
        log_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
        all_logs.append(payload)

        print("Saved log:", log_path)
        print("World video:", world_video)
        print("Wrist video:", wrist_video)
        print("Grasp-front video:", grasp_front_video)
        print("Low corner video:", grasp_corner_video)
        print("GelSight RGB video:", tactile_outputs["rgb_video"])
        print("GelSight audit video:", tactile_outputs["audit_video"])
        print("GelSight quantitative data:", tactile_outputs["quantitative_npz"])
        print("GelSight validation:", tactile_outputs["validation"]["status"])
        print("Termination:", rollout_termination_reason)
        print("Rough success:", success)
        if DISPLAY_ROLLOUT_VIDEOS:
            display(Video(str(world_video), embed=False, width=640))
            display(Video(str(wrist_video), embed=False, width=320))
            display(Video(str(grasp_front_video), embed=False, width=480))
            display(Video(str(grasp_corner_video), embed=False, width=480))
        if DISPLAY_GELSIGHT_VIDEOS:
            display(Video(tactile_outputs["rgb_video"], embed=False, width=480))
            display(Video(tactile_outputs["audit_video"], embed=False, width=960))

finally:
    if worker is not None:
        try:
            worker.stdin.close()
        except Exception:
            pass
        worker.terminate()
        try:
            worker.wait(timeout=30)
        except subprocess.TimeoutExpired:
            worker.kill()
            worker.wait(timeout=30)

print("Closed-loop output directory:", run_dir)


In [ ]:
# Cell P2: OpenVLA-OFT chunk/controller/robot/proprio diagnostics.

print("Closed-loop output directory:", run_dir)

from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image as NotebookImage, display


DIAGNOSTIC_RUN_DIR = Path(run_dir)

SIGN_THRESHOLD_MM = 0.10
ROTATION_SIGN_THRESHOLD_DEG = 0.02

required_rotation_helpers = [
    "quat_apply_world_delta",
    "quat_error_world",
]
missing_helpers = [
    name for name in required_rotation_helpers if name not in globals()
]
assert not missing_helpers, (
    "Run Cell P0 first. Missing: " + ", ".join(missing_helpers)
)

axis_colors = {
    "X": "#376fd0",
    "Y": "#a0459b",
    "Z": "#e08b2c",
}
translation_names = ("X", "Y", "Z")
rotation_names = ("Roll", "Pitch", "Yaw")

log_paths = sorted(
    DIAGNOSTIC_RUN_DIR.glob("rollout_*/closed_loop_log.json")
)
assert log_paths, f"No rollout logs found under {DIAGNOSTIC_RUN_DIR}"


for log_path in log_paths:
    payload = json.loads(log_path.read_text(encoding="utf-8"))
    records = payload["records"]
    assert records, f"No records in {log_path}"

    raw = np.asarray(
        [record["raw_openvla_action"] for record in records],
        dtype=np.float64,
    )
    clipped = np.asarray(
        [record["clipped_controller_action"] for record in records],
        dtype=np.float64,
    )

    # ------------------------------------------------------------------
    # Translation calculations
    # ------------------------------------------------------------------
    ee_target = np.asarray(
        [record["ee_target"] for record in records],
        dtype=np.float64,
    )
    ee_after = np.asarray(
        [record["ee_after"] for record in records],
        dtype=np.float64,
    )

    # Before action step s is the measured pose after action step s-1.
    ee_before = np.empty_like(ee_after)
    ee_before[1:] = ee_after[:-1]

    # Reconstruct the initial pose before the first command.
    ee_before[0] = ee_target[0] - clipped[0, :3]

    model_delta_mm = raw[:, :3] * 1000.0
    controller_delta_mm = (ee_target - ee_before) * 1000.0
    actual_delta_mm = (ee_after - ee_before) * 1000.0

    translation_mae = []
    translation_opposite_masks = []

    for axis_index, axis_name in enumerate(translation_names):
        model_values = model_delta_mm[:, axis_index]
        controller_values = controller_delta_mm[:, axis_index]
        actual_values = actual_delta_mm[:, axis_index]

        opposite_sign = (
            (np.abs(controller_values) >= SIGN_THRESHOLD_MM)
            & (np.abs(actual_values) >= SIGN_THRESHOLD_MM)
            & (np.sign(controller_values) != np.sign(actual_values))
        )
        mae_mm = float(
            np.mean(np.abs(controller_values - actual_values))
        )

        translation_mae.append(mae_mm)
        translation_opposite_masks.append(opposite_sign)

        print(
            f"{log_path.parent.name} {axis_name}: "
            f"OpenVLA-OFT requested total={model_values.sum():+.1f} mm | "
            f"controller target total={controller_values.sum():+.1f} mm | "
            f"robot moved total={actual_values.sum():+.1f} mm | "
            f"MAE={mae_mm:.3f} mm | "
            f"opposite-sign action steps={opposite_sign.sum()}"
        )

    # ------------------------------------------------------------------
    # Rotation calculations
    # ------------------------------------------------------------------
    target_quat = np.asarray(
        [record["ee_target_quat_wxyz"] for record in records],
        dtype=np.float64,
    )
    after_quat = np.asarray(
        [record["ee_after_quat_wxyz"] for record in records],
        dtype=np.float64,
    )

    # Before action step s is the measured orientation after action step s-1.
    before_quat = np.empty_like(after_quat)
    before_quat[1:] = after_quat[:-1]

    # Reconstruct the initial orientation before the first command.
    before_quat[0] = quat_apply_world_delta(
        target_quat[0],
        -clipped[0, 3:6],
    )

    model_rotation_deg = np.rad2deg(raw[:, 3:6])

    controller_rotation_deg = np.rad2deg(np.asarray([
        quat_error_world(target_quat[index], before_quat[index])
        for index in range(len(records))
    ]))

    actual_rotation_deg = np.rad2deg(np.asarray([
        quat_error_world(after_quat[index], before_quat[index])
        for index in range(len(records))
    ]))

    rotation_mae = []
    rotation_opposite_masks = []

    for axis_index, axis_name in enumerate(rotation_names):
        model_values = model_rotation_deg[:, axis_index]
        controller_values = controller_rotation_deg[:, axis_index]
        actual_values = actual_rotation_deg[:, axis_index]

        opposite_sign = (
            (np.abs(controller_values) >= ROTATION_SIGN_THRESHOLD_DEG)
            & (np.abs(actual_values) >= ROTATION_SIGN_THRESHOLD_DEG)
            & (np.sign(controller_values) != np.sign(actual_values))
        )
        mae_deg = float(
            np.mean(np.abs(controller_values - actual_values))
        )

        rotation_mae.append(mae_deg)
        rotation_opposite_masks.append(opposite_sign)

        print(
            f"{log_path.parent.name} {axis_name}: "
            f"model max={np.max(np.abs(model_values)):.3f} degrees | "
            f"controller max={np.max(np.abs(controller_values)):.3f} degrees | "
            f"robot max={np.max(np.abs(actual_values)):.3f} degrees | "
            f"MAE={mae_deg:.3f} degrees | "
            f"opposite-sign action steps={opposite_sign.sum()}"
        )

    # ------------------------------------------------------------------
    # Gripper calculations
    # ------------------------------------------------------------------
    missing_gripper_measurements = [
        record["query"]
        for record in records
        if "gripper_after_per_finger_m" not in record
    ]
    assert not missing_gripper_measurements, (
        "This rollout predates measured gripper logging. "
        "Run Cell P again, then run Cell P2 on its new run_dir."
    )

    gripper_open_m = float(
        globals().get("OPENVLA_GRIPPER_OPENING_M", 0.04)
    )
    gripper_closed_m = float(
        globals().get("OPENVLA_GRIPPER_CLOSED_M", 0.0)
    )
    opening_range = gripper_open_m - gripper_closed_m
    assert opening_range > 0.0, "Invalid gripper opening range."

    model_gripper = raw[:, 6]
    gripper_target_m = np.asarray(
        [record["gripper_target_per_finger_m"] for record in records],
        dtype=np.float64,
    )
    gripper_observed_m = np.asarray(
        [record["gripper_after_per_finger_m"] for record in records],
        dtype=np.float64,
    )
    controller_gripper = (
        gripper_target_m - gripper_closed_m
    ) / opening_range
    mujoco_gripper = (
        gripper_observed_m - gripper_closed_m
    ) / opening_range
    gripper_mae_fraction = float(
        np.mean(np.abs(controller_gripper - mujoco_gripper))
    )
    gripper_mae_per_finger_mm = (
        gripper_mae_fraction * opening_range * 1000.0
    )
    print(
        f"{log_path.parent.name} gripper: "
        f"controller-vs-MuJoCo MAE="
        f"{gripper_mae_per_finger_mm:.2f} mm per finger"
    )

    # ------------------------------------------------------------------
    # Combined figure: translation, rotation, and gripper
    # ------------------------------------------------------------------
    queries = np.arange(len(records))

    figure, axes = plt.subplots(
        3,
        1,
        figsize=(15, 13),
        sharex=True,
        constrained_layout=True,
    )

    figure.suptitle(
        f"{log_path.parent.name}: closed-loop command execution\n"
        "OpenVLA-OFT action vs controller target vs observed robot motion",
        fontsize=15,
    )

    # Translation panel
    axis = axes[0]

    translation_any_opposite = np.logical_or.reduce(
        translation_opposite_masks
    )
    axis.fill_between(
        queries,
        0,
        1,
        where=translation_any_opposite,
        transform=axis.get_xaxis_transform(),
        step="mid",
        color="#dc2626",
        alpha=0.10,
        label="Any controller/robot opposite sign",
    )

    for axis_index, axis_name in enumerate(translation_names):
        color = axis_colors[axis_name]

        axis.plot(
            queries,
            model_delta_mm[:, axis_index],
            color=color,
            linewidth=2.3,
            alpha=0.80,
            label=f"OpenVLA-OFT delta {axis_name}",
        )
        axis.plot(
            queries,
            controller_delta_mm[:, axis_index],
            color=color,
            linewidth=1.7,
            linestyle="--",
            label=f"Controller delta {axis_name}",
        )
        axis.plot(
            queries,
            actual_delta_mm[:, axis_index],
            color=color,
            linewidth=2.0,
            linestyle=":",
            label=f"Robot delta {axis_name}",
        )

    axis.axhline(0.0, color="#333333", linewidth=0.9)
    axis.set_title(
        "Translation commands | controller-vs-robot MAE: "
        f"X={translation_mae[0]:.3f}, "
        f"Y={translation_mae[1]:.3f}, "
        f"Z={translation_mae[2]:.3f} mm"
    )
    axis.set_ylabel("Millimetres / executed action")
    axis.grid(alpha=0.25)
    axis.legend(loc="upper right", fontsize=8, ncol=3)

    # Rotation panel
    axis = axes[1]

    rotation_any_opposite = np.logical_or.reduce(
        rotation_opposite_masks
    )
    axis.fill_between(
        queries,
        0,
        1,
        where=rotation_any_opposite,
        transform=axis.get_xaxis_transform(),
        step="mid",
        color="#dc2626",
        alpha=0.10,
        label="Any controller/robot opposite sign",
    )

    for axis_index, axis_name in enumerate(rotation_names):
        color = tuple(axis_colors.values())[axis_index]

        axis.plot(
            queries,
            model_rotation_deg[:, axis_index],
            color=color,
            linewidth=2.3,
            alpha=0.80,
            label=f"OpenVLA-OFT delta {axis_name}",
        )
        axis.plot(
            queries,
            controller_rotation_deg[:, axis_index],
            color=color,
            linewidth=1.7,
            linestyle="--",
            label=f"Controller delta {axis_name}",
        )
        axis.plot(
            queries,
            actual_rotation_deg[:, axis_index],
            color=color,
            linewidth=2.0,
            linestyle=":",
            label=f"Robot delta {axis_name}",
        )

    axis.axhline(0.0, color="#333333", linewidth=0.9)
    axis.set_title(
        "Rotational commands | controller-vs-robot MAE: "
        f"Roll={rotation_mae[0]:.3f}, "
        f"Pitch={rotation_mae[1]:.3f}, "
        f"Yaw={rotation_mae[2]:.3f} degrees"
    )
    axis.set_ylabel("Degrees / executed action")
    axis.grid(alpha=0.25)
    axis.legend(loc="upper right", fontsize=8, ncol=3)

    # Gripper panel
    axis = axes[2]

    axis.plot(
        queries,
        model_gripper,
        color="#2563eb",
        linewidth=2.3,
        label="OpenVLA-OFT gripper output",
    )
    axis.plot(
        queries,
        controller_gripper,
        color="#f59e0b",
        linewidth=1.9,
        linestyle="--",
        label="Controller target opening",
    )
    axis.plot(
        queries,
        mujoco_gripper,
        color="#16a34a",
        linewidth=2.1,
        label="MuJoCo observed opening",
    )
    axis.axhline(
        0.5,
        color="#20242a",
        linewidth=1.0,
        linestyle=":",
        label="Half-open reference",
    )

    axis.set_title(
        "Continuous gripper behavior | "
        f"controller-vs-MuJoCo MAE = "
        f"{gripper_mae_per_finger_mm:.2f} mm per finger"
    )
    axis.set_xlabel(
        f"Executed action step ({CONTROL_HORIZON_S:.3f} s control horizon)"
    )
    axis.set_ylabel("Open fraction")
    axis.set_ylim(-0.05, 1.05)
    axis.grid(alpha=0.25)
    axis.legend(loc="upper right", fontsize=8)

    output_path = (
        log_path.parent / "closed_loop_command_comparison.png"
    )
    figure.savefig(output_path, dpi=150)
    plt.close(figure)

    print("Saved:", output_path)
    display(NotebookImage(filename=str(output_path)))

    # ------------------------------------------------------------------
    # Measured proprio and the state supplied at each chunk boundary
    # ------------------------------------------------------------------
    measured_proprio = np.asarray(
        [record["measured_proprio_before_action"] for record in records],
        dtype=np.float64,
    )
    query_proprio = np.asarray(
        [record["query_proprio_input"] for record in records],
        dtype=np.float64,
    )
    chunk_steps = np.asarray(
        [record["chunk_step"] for record in records], dtype=np.int64
    )
    chunk_start_steps = np.flatnonzero(chunk_steps == 0)
    assert measured_proprio.shape == query_proprio.shape == (len(records), 8)

    proprio_figure, proprio_axes = plt.subplots(
        2, 1, figsize=(15, 9), sharex=True, constrained_layout=True
    )
    proprio_figure.suptitle(
        f"{log_path.parent.name}: live OFT proprioception",
        fontsize=15,
    )
    for joint_index in range(7):
        line = proprio_axes[0].plot(
            queries,
            measured_proprio[:, joint_index],
            linewidth=1.7,
            label=f"measured q{joint_index + 1}",
        )[0]
        proprio_axes[0].scatter(
            chunk_start_steps,
            query_proprio[chunk_start_steps, joint_index],
            color=line.get_color(),
            marker="o",
            s=22,
        )
    proprio_axes[0].set(
        title="Joint positions; circles mark the state supplied at each policy query",
        ylabel="Radians",
    )
    proprio_axes[0].grid(alpha=0.25)
    proprio_axes[0].legend(ncol=4, fontsize=8)

    proprio_axes[1].plot(
        queries,
        measured_proprio[:, 7],
        color="#18a058",
        linewidth=2.2,
        label="Measured before each action",
    )
    proprio_axes[1].scatter(
        chunk_start_steps,
        query_proprio[chunk_start_steps, 7],
        color="#dc3a3e",
        s=30,
        label="OFT query input",
    )
    proprio_axes[1].set(
        title="Measured gripper proprioception",
        xlabel=f"Executed action step ({CONTROL_HORIZON_S:.3f} s each)",
        ylabel="Open fraction",
        ylim=(-0.05, 1.05),
    )
    proprio_axes[1].grid(alpha=0.25)
    proprio_axes[1].legend()

    proprio_output_path = log_path.parent / "closed_loop_proprio.png"
    proprio_figure.savefig(proprio_output_path, dpi=150)
    plt.close(proprio_figure)
    print("Saved:", proprio_output_path)
    display(NotebookImage(filename=str(proprio_output_path)))


In [ ]:
# Cell P3: World video with live OFT translation, rotation, and gripper diagrams.
# Run Cell P2 immediately before this cell.
from pathlib import Path
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from IPython.display import Video, display

required_m2_values = [
    "log_path", "payload", "records", "queries", "axis_colors",
    "translation_names", "rotation_names", "model_delta_mm",
    "controller_delta_mm", "actual_delta_mm", "model_rotation_deg",
    "controller_rotation_deg", "actual_rotation_deg", "model_gripper",
    "controller_gripper", "mujoco_gripper",
]
missing = [name for name in required_m2_values if name not in globals()]
assert not missing, "Run Cell P2 first. Missing: " + ", ".join(missing)

source_video = Path(payload["videos"]["world"])
if not source_video.is_absolute():
    source_video = log_path.parent / source_video
assert source_video.exists(), f"Missing world video: {source_video}"

motion_panels = [
    (
        "Translation commands",
        "mm / executed action",
        translation_names,
        (model_delta_mm, controller_delta_mm, actual_delta_mm),
        1.0,
    ),
    (
        "Rotational commands",
        "degrees / executed action",
        rotation_names,
        (
            model_rotation_deg,
            controller_rotation_deg,
            actual_rotation_deg,
        ),
        0.1,
    ),
]

source_labels = ("OpenVLA-OFT", "Controller", "MuJoCo")
source_styles = (("-", 1.8), ("--", 1.5), (":", 1.7))

output_path = log_path.parent / "world_with_live_diagnostics.mp4"
reader = imageio.get_reader(str(source_video))
output_fps = float(
    reader.get_meta_data().get("fps", globals().get("VIDEO_FPS", 10))
)
writer = imageio.get_writer(
    str(output_path),
    fps=output_fps,
    codec="libx264",
    pixelformat="yuv420p",
    macro_block_size=None,
)

figure, axes = plt.subplots(
    3,
    1,
    figsize=(9.6, 7.2),
    dpi=100,
    sharex=True,
    constrained_layout=True,
)

# Translation and rotation panels.
motion_lines = []
for panel_index, (title, ylabel, names, arrays, minimum) in enumerate(
    motion_panels
):
    panel_lines = []

    for dimension, name in enumerate(names):
        color = tuple(axis_colors.values())[dimension]
        dimension_lines = []

        for source_index, source_label in enumerate(source_labels):
            style, width = source_styles[source_index]
            line, = axes[panel_index].plot(
                [],
                [],
                color=color,
                linestyle=style,
                linewidth=width,
                alpha=0.85,
                label=f"{source_label} {name}",
            )
            dimension_lines.append(line)

        panel_lines.append(dimension_lines)

    all_values = np.concatenate([
        np.asarray(values).reshape(-1) for values in arrays
    ])
    limit = max(
        minimum,
        1.08 * float(np.nanmax(np.abs(all_values))),
    )

    axes[panel_index].set(
        title=title,
        ylabel=ylabel,
        ylim=(-limit, limit),
    )
    axes[panel_index].legend(
        loc="upper right",
        fontsize=6.5,
        ncol=3,
    )
    motion_lines.append(panel_lines)

# Gripper panel.
gripper_lines = [
    axes[2].plot(
        [],
        [],
        color=color,
        linestyle=style,
        linewidth=width,
        label=label,
    )[0]
    for color, style, width, label in (
        ("#2563eb", "-", 2.0, "OpenVLA-OFT gripper output"),
        ("#f59e0b", "--", 1.7, "Controller target opening"),
        ("#16a34a", "-", 1.9, "MuJoCo observed opening"),
    )
]

axes[2].axhline(
    0.5,
    color="#20242a",
    linewidth=1.0,
    linestyle=":",
    label="Half-open reference",
)
axes[2].set(
    title="Continuous gripper behavior",
    ylabel="Open fraction",
    ylim=(-0.05, 1.05),
)
axes[2].set_xlabel(
    f"Executed action step ({float(CONTROL_HORIZON_S):.3f} s control horizon)"
)
axes[2].legend(loc="upper right", fontsize=7, ncol=2)

# Fixed axes and current-action cursors.
query_cursors = []
for axis in axes:
    axis.axhline(
        0.0,
        color="#333333",
        linewidth=0.7,
        alpha=0.55,
    )
    axis.set_xlim(0, max(1, len(records) - 1))
    axis.grid(alpha=0.25)

    cursor = axis.axvline(
        0,
        color="#dc2626",
        linewidth=1.1,
        alpha=0.75,
    )
    cursor.set_visible(False)
    query_cursors.append(cursor)

gripper_arrays = (
    model_gripper,
    controller_gripper,
    mujoco_gripper,
)

frame_count = 0

try:
    for frame_index, source_frame in enumerate(reader):
        # Frame 0 is before action step 0.
        # Frame s+1 is the simulated state after executed action step s.
        completed = min(frame_index, len(records))
        shown_queries = queries[:completed]

        # Update translation and rotation lines.
        for panel_index, (_, _, names, arrays, _) in enumerate(
            motion_panels
        ):
            for dimension in range(len(names)):
                for source_index in range(3):
                    motion_lines[
                        panel_index
                    ][dimension][source_index].set_data(
                        shown_queries,
                        arrays[source_index][:completed, dimension],
                    )

        # Update gripper lines.
        for line, values in zip(gripper_lines, gripper_arrays):
            line.set_data(shown_queries, values[:completed])

        if completed:
            current_step = completed - 1

            for cursor in query_cursors:
                cursor.set_xdata([current_step, current_step])
                cursor.set_visible(True)

            current_record = records[current_step]
            status = (
                f"action {current_step:03d} | "
                f"query {current_record['query']:03d} "
                f"h{current_record['chunk_step']}"
            )
        else:
            status = "before action 000"

        figure.suptitle(
            f"{log_path.parent.name}: live command execution | {status}",
            fontsize=13,
        )

        figure.canvas.draw()
        plot_frame = np.asarray(
            figure.canvas.buffer_rgba(),
            dtype=np.uint8,
        )[:, :, :3].copy()

        # Resize the world video without changing its aspect ratio.
        source_frame = np.asarray(
            source_frame,
            dtype=np.uint8,
        )[:, :, :3]

        panel_height = plot_frame.shape[0]
        source_width = int(round(
            panel_height
            * source_frame.shape[1]
            / source_frame.shape[0]
        ))
        source_width += source_width % 2

        resampling = getattr(Image, "Resampling", Image).LANCZOS
        source_frame = np.asarray(
            Image.fromarray(source_frame).resize(
                (source_width, panel_height),
                resampling,
            )
        )

        combined_frame = np.concatenate(
            [source_frame, plot_frame],
            axis=1,
        )
        writer.append_data(combined_frame)
        frame_count += 1

finally:
    reader.close()
    writer.close()
    plt.close(figure)

assert frame_count > 0, f"No frames read from {source_video}"

action_aligned_frames = len(records) + 1
assert frame_count >= action_aligned_frames, (
    f"Video ended at {frame_count} frames before all "
    f"{action_aligned_frames} action-aligned frames were rendered."
)
settle_frames = frame_count - action_aligned_frames
if settle_frames:
    print(
        f"Included {settle_frames} post-release settle frames; "
        "diagnostic traces hold their final action value."
    )

print("Saved synchronized live-diagnostics video:", output_path)
display(Video(str(output_path), embed=False, width=1200))

In [ ]:
# Cell P4: Judgment video -- zoomed grasp view, Taxim tactile RGB, and native
# depth side by side, time-synchronized, for visually judging whether the
# tactile sensor tracks real contact geometry (e.g. the cube edge, the bump
# texture) correctly. Reads only files already written by Cell P for a given
# rollout; it does not re-run the policy or touch the live rollout model.

# Which rollouts to build. None (default) processes every rollout_* folder
# found under run_dir. To look at specific ones instead, set this to a list
# of rollout indices, e.g. JUDGMENT_ROLLOUT_INDICES = [1, 3] for only
# rollout_001 and rollout_003.
JUDGMENT_ROLLOUT_INDICES = [1]

def _fixed_scale_depth_rgb_for_judgment(depth_m):
    # Mirrors Cell P's _fixed_scale_depth_rgb exactly, so this panel always
    # matches gelsight_audit.mp4's current display scale and gamma.
    return _fixed_scale_depth_rgb(depth_m)


def render_tactile_judgment_video(rollout_dir, out_name="tactile_judgment.mp4"):
    import imageio.v3 as imageio_v3

    rollout_dir = Path(rollout_dir)
    grasp_front_path = rollout_dir / "grasp_front.mp4"
    tactile_rgb_path = rollout_dir / "gelsight_rgb.mp4"
    quant_path = rollout_dir / "gelsight_quantitative_data.npz"
    for required in (grasp_front_path, tactile_rgb_path, quant_path):
        assert required.is_file(), f"Missing {required}; run Cell P for this rollout first."

    quant = np.load(quant_path)
    native_depth_m = quant["native_depth_m"]
    sim_time_s = quant["sim_time_s"]
    n_frames = native_depth_m.shape[0]

    # imageio.v2's ffmpeg reader can mis-report frame count as inf and over-read
    # on repeated iteration; v3's imiter reads exactly the encoded frame stream.
    grasp_front_frames = list(imageio_v3.imiter(str(grasp_front_path)))
    tactile_rgb_frames = list(imageio_v3.imiter(str(tactile_rgb_path)))
    assert len(tactile_rgb_frames) == n_frames, (len(tactile_rgb_frames), n_frames)

    panel_size = (640, 480)
    label_h = 30
    foot_h = 22

    writer = imageio.get_writer(
        str(rollout_dir / out_name),
        fps=GELSIGHT_TACTILE_FPS,
        codec="libx264",
        quality=8,
        pixelformat="yuv420p",
        macro_block_size=None,
    )
    try:
        for i in range(n_frames):
            grasp_idx = int(np.clip(round(sim_time_s[i] * VIDEO_FPS), 0, len(grasp_front_frames) - 1))
            front = np.asarray(
                Image.fromarray(grasp_front_frames[grasp_idx]).resize(panel_size, Image.Resampling.LANCZOS)
            )
            tactile = np.asarray(
                Image.fromarray(tactile_rgb_frames[i]).resize(panel_size, Image.Resampling.NEAREST)
            )
            depth = np.asarray(
                Image.fromarray(_fixed_scale_depth_rgb_for_judgment(native_depth_m[i])).resize(
                    panel_size, Image.Resampling.NEAREST
                )
            )

            panel = Image.new("RGB", (3 * panel_size[0], panel_size[1] + label_h + foot_h), (18, 18, 22))
            draw = ImageDraw.Draw(panel)
            panel.paste(Image.fromarray(front), (0, label_h))
            panel.paste(Image.fromarray(tactile), (panel_size[0], label_h))
            panel.paste(Image.fromarray(depth), (2 * panel_size[0], label_h))
            draw.text((6, 7), "Zoomed grasp view (front camera)", fill=(240, 240, 240))
            draw.text((panel_size[0] + 6, 7), "Taxim-calibrated tactile RGB", fill=(240, 240, 240))
            draw.text(
                (2 * panel_size[0] + 6, 7),
                f"Native depth (fixed 0-{GELSIGHT_DEPTH_DISPLAY_MAX_M * 1e3:.1f} mm, gamma {GELSIGHT_DEPTH_DISPLAY_GAMMA:.2f})",
                fill=(240, 240, 240),
            )
            draw.text(
                (6, panel.height - foot_h + 4),
                f"t={sim_time_s[i]:.3f}s   grasp-view frame {grasp_idx} (10 Hz)   tactile frame {i} (30 Hz)",
                fill=(200, 200, 200),
            )
            writer.append_data(np.asarray(panel))
    finally:
        writer.close()

    return rollout_dir / out_name


if "run_dir" in globals():
    _judgment_video_paths = []
    if JUDGMENT_ROLLOUT_INDICES is None:
        _rollout_dirs = sorted(run_dir.glob("rollout_*"))
    else:
        _rollout_dirs = [run_dir / f"rollout_{i:03d}" for i in JUDGMENT_ROLLOUT_INDICES]
        _missing = [d for d in _rollout_dirs if not d.is_dir()]
        assert not _missing, f"No such rollout folder(s): {_missing}"
    for _rollout_dir in _rollout_dirs:
        _judgment_video_paths.append(render_tactile_judgment_video(_rollout_dir))
    print(f"Wrote {len(_judgment_video_paths)} tactile judgment video(s):")
    for _p in _judgment_video_paths:
        print(" ", _p)
    if DISPLAY_GELSIGHT_VIDEOS and _judgment_video_paths:
        display(Video(str(_judgment_video_paths[-1]), embed=False, width=3 * 480))
else:
    print("run_dir not in globals(); call render_tactile_judgment_video(<rollout_dir>) manually.")
